# Cross-system comparison: aggregate DVP of HEK293 cells, iPSC-derived neurons and human brain

Compares the aggregate-versus-host contrasts of the three aggregate DVP datasets: HEK293 cells seeded with α-synuclein preformed fibrils (PFF) (1_HEK notebook), iPSC-derived dopaminergic neurons treated with PFF (2_iDA notebook), and post-mortem brain tissue of donors with Parkinson's disease or Lewy body disease and of controls (3_BrainTissue notebook). Each dataset is loaded and filtered here as in its own notebook, so this notebook runs without the other three. It makes the panels of Figure 5, Figure S7 and Figure S8.

| Short | Meaning |
|---|---|
| EA | excised aggregates; HEK obs value "Pure Aggregates" |
| RC | rest cells: HEK cells with aggregates after the aggregate was excised |
| SAgg, NAgg | iPSC somatic, neuritic aggregates |
| Round, Elong | brain round, elongated inclusions |
| DAB+, DAB− | brain neurons with, without a pS129 α-synuclein (DAB-stained) inclusion |
| PFF, PBS | α-synuclein preformed fibrils; PBS control |
| BafA1 | bafilomycin A1 |
| log2FC | log2 fold change |
| FDR, BH | false discovery rate, Benjamini-Hochberg |
| eBayes | limma linear model with empirical Bayes moderation of the variances |
| GO, GAF | Gene Ontology; GO annotation file |
| GSEA, NES | gene set enrichment analysis, normalised enrichment score |
| DCV | dense-core vesicle |
| FTD, ALS, RBP | frontotemporal dementia, amyotrophic lateral sclerosis, RNA-binding protein |
| PDB | Protein Data Bank |
| TM | transmembrane |

**Inputs:** the DIA-NN `report.pg_matrix.tsv` and `metadata.csv` of the three datasets; the UniProt-to-gene table `src/Uniprot_to_GeneSymbol.csv` (from the alphapepttools documentation); the MSigDB gene-set cache `src/gsea_genesets_cache.parquet` and the GSEA decision files `review.csv` in `gsea_dir/<contrast>/` (section 10); `LITERATURE_MASTER.xlsx`, the published gene lists (section 11); `Organelles/` (section 4), `Complexes/` (section 12) and `Physicochemical_Properties/` (section 12.3). All paths are set in the first cell of section 2.

**Outputs:** result tables in `output_dir` (`results/CrossSystem/`); among them the organelle module member table `SupplementaryTable_organelle_modules_members.csv`; figure panels in `figures/`, written by `save_panel` and assembled in section 14.

# 1. Imports

In [ ]:
# --- standard library ---
import gzip
import itertools
import logging
import os
import re
import shutil
import sys
import time
from collections import Counter, defaultdict
from dataclasses import replace
from datetime import date
from itertools import combinations
from math import comb

# --- scientific stack ---
import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
from inmoose.limma import eBayes, lmFit, topTable
from scipy.stats import (false_discovery_control, mannwhitneyu, pearsonr, rankdata,
                         spearmanr, ttest_1samp)

# --- plotting ---
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from matplotlib.patches import Patch

# --- alphapepttools ---
import alphapepttools as at
from alphapepttools.pl.figure import create_figure, label_axes
from alphapepttools.tl.stats import nan_safe_bh_correction

# --- project modules in src/ ---
# The path is relative to this notebook's folder, which VS Code uses as the
# working directory of a notebook. Importing figure_style sets the matplotlib and
# alphapepttools style of every panel.
sys.path.insert(0, "src")
from figure_style import (ANNOT_FS, AXES_LW, BG_COLOR, DOWN_COLOR, DRAW_SCALE, LABEL_FS,
                          MARKER_LW, REF_COLOR, THIN_LW, TICK_FS, TITLE_FS, UP_COLOR,
                          compose_figure, figsize, native_pt, save_panel, shade,
                          style_summary)
from gsea_pipeline import (GSEAConfig, build_review, load_gene_sets, plot_combined,
                           run_contrast, run_contrast_from_rank)
from crosssystem_plots import (concept_pair_heatmap, organelle_bars, organelle_ylim,
                               property_levels, split_dumbbell, split_property_heatmap)
from dotplot import cross_system_dotplot
from preprocessing import flag_contaminants

logging.getLogger("fontTools.subset").setLevel(logging.WARNING)
today = date.today().strftime("%Y%m%d")      # date stamp of the output file names

# 2. Data loading

In [ ]:
# Every path of the notebook, relative to the repository root.

# --- inputs: DIA-NN protein groups (report.pg_matrix.tsv) and sample metadata
#     (metadata.csv) of the three datasets ---
BRAIN_DIR = 'data/BrainTissue/ADVP04_011_Combined_withBannerLibrary/'
IPSC_DIR  = 'data/iDA/DIANN_OptimizedSettings'
HEK_DIR   = 'data/HEK/DIANN_OptimizedSettings'
var_metadata_path = 'src/Uniprot_to_GeneSymbol.csv'   # UniProt to gene symbol, from the alphapepttools documentation

# --- inputs: annotation resources ---
ORG_DIR = "data/CrossSystem/Organelles"                   # Gene Ontology, GO annotations, organelle module list (section 4)
CX_DIR  = "data/CrossSystem/Complexes"                    # Complex Portal and UniProt tables (section 12)
PP_DIR  = "data/CrossSystem/Physicochemical_Properties"   # UniProt sequences and CamSol scores (section 12.3)
literature_list = 'data/CrossSystem/LITERATURE_MASTER.xlsx'   # published gene lists: sheet INDEX (tier and
                                             # notes per sheet), then one sheet per study

# --- outputs ---
output_dir = "results/CrossSystem/"                   # result tables
gsea_dir   = "gsea/CrossSystem"   # GSEA results; <contrast>/review.csv holds the
                                             # decisions of section 10 and is an input
FIG5_DIR = "figures/Figure5_Comparison"   # Figure 5 panels
SI7_DIR  = "figures/SIs/SI7_CrossSystem"  # Figure S7 panels
SI8_DIR  = "figures/SIs/SI8_CrossSystem"  # Figure S8 panels
os.makedirs(output_dir, exist_ok=True)

In [ ]:
# Completeness rule of the three bootstraps below, as in the 3_BrainTissue notebook.
def complete_in_any_group(adata, group_column, max_missing):
    """Keep the proteins missing in at most max_missing of the samples of at least
    one group, so a protein specific to one fraction stays. With group_column,
    filter_data_completeness drops a protein that fails in any group, so each group
    is flagged on its own and the flags are combined with OR."""
    passes = np.zeros(adata.n_vars, dtype=bool)
    for group in adata.obs[group_column].dropna().unique():
        flagged = at.pp.filter_data_completeness(
            adata.copy(), max_missing=max_missing,
            group_column=group_column, groups=[group], action="flag",
        )
        passes |= flagged.var["passed_threshold_missing_values"].to_numpy()
    return adata[:, passes].copy()

## 2.1 Brain tissue

In [ ]:
# Brain tissue: load the DIA-NN protein groups and filter them as in the BrainTissue
# notebook. Later cells read adata_sf_brain (detection per gene) and adata_log_brain
# (the eBayes tests of section 3.3).
input_dir = BRAIN_DIR

pg_table_path = f'{input_dir}/report.pg_matrix.tsv'
obs_metadata_path = f'{input_dir}/metadata.csv'

adata = at.io.read_pg_table(pg_table_path, search_engine="diann")
obs_metadata = pd.read_csv(obs_metadata_path, sep=",", index_col=0)
var_metadata = pd.read_csv(var_metadata_path, sep=",", index_col=0)   # UniProt to gene symbol; the iPSC and HEK cells read it too

# Sample metadata onto obs; samples without a metadata row are dropped.
adata = at.pp.add_metadata(
    adata=adata,
    incoming_metadata=obs_metadata,
    axis=0,
    keep_data_shape=False,
)
adata.var = adata.var.merge(var_metadata, left_index=True, right_index=True, how="left")


# --- Samples excluded after inspection of the cutting ---
# One empty well and three cuts that were not clean. Each entry is a substring of
# exactly one raw file name.
DROP_SUBSTRINGS = [
    # empty sample
    "Plate3C2_D05_260212_CTRL2_Glia_WhiteMatter",
    # cuts that were not clean
    "Plate4C3_D06_260212_LBD2_RoundAggs1_Cortex",
    "Plate3C2_A10_260212_LBD2_ElonAggs1_Cortex",
    "Plate3C2_G06_260212_LBD2_RoundAggs2_Cortex"
]

def _mask_keep(obs_index):
    bad = pd.Series(False, index=obs_index)
    for s in DROP_SUBSTRINGS:
        bad |= obs_index.str.contains(s, regex=False)
    return ~bad

keep = _mask_keep(adata.obs.index)
print(f"Dropping {(~keep).sum()} problematic samples (of {keep.size}): "
      f"{list(adata.obs.index[~keep])}")
adata = adata[keep, :].copy()

# The random controls (RandomControl_Random) have no brain region in the metadata
# and belong to no fraction.
BRAIN_REGION_COL = "Brain Region"
if BRAIN_REGION_COL in adata.obs.columns:
    random_mask = adata.obs[BRAIN_REGION_COL].isna()
    print(f"Dropping {random_mask.sum()} RandomControl_Random samples (missing {BRAIN_REGION_COL!r})")
    adata = adata[~random_mask, :].copy()
else:
    print(f"WARNING: column {BRAIN_REGION_COL!r} not found in adata.obs — "
          f"skipping RandomControl_Random drop. "
          f"If the column exists under a different name, update BRAIN_REGION_COL.")

print(f"Final: adata = {adata.n_obs} samples x {adata.n_vars} proteins")

# --- Parameters ---
biological   = "Disease_CellType_Aggregate"   # condition column of the sample filters and the group counts
forfiltering = "CellType_Aggregate"           # groups of the 80 % completeness filter, disease status collapsed
MIN_PROTEINS = 2000          # samples with fewer protein groups are excluded
MAX_MISSING_COND = 0.20      # a protein is kept if it is missing in at most 20 % of one group

# ------------------------------------------------------------
# 1. Protein groups per sample
# ------------------------------------------------------------
adata.obs["num_proteins"] = np.sum(~np.isnan(adata.X), axis=1)
n_start_samples, n_start_proteins = adata.shape

# ------------------------------------------------------------
# 2. Sample filters: blanks, fewer than MIN_PROTEINS protein groups, missing group label
# ------------------------------------------------------------
adata = adata[~adata.obs[biological].isin(["Blank", "EvoTip Blank"]), :].copy()
n_after_blanks = adata.n_obs

adata.obs["is_outlier"] = adata.obs["num_proteins"] < MIN_PROTEINS
adata_unfiltered = adata.copy()      # all samples before the protein-count filter
adata = adata_unfiltered[~adata_unfiltered.obs["is_outlier"], :].copy()
n_after_outliers = adata.n_obs

nan_mask = adata.obs[biological].isna() | adata.obs[forfiltering].isna()
if nan_mask.any():
    print(f"WARNING: dropping {nan_mask.sum()} samples with NaN in {biological} or {forfiltering}")
    adata = adata[~nan_mask, :].copy()
n_after_nan = adata.n_obs

# Samples per group: the completeness filter works on the forfiltering groups,
# the tests on the biological groups.
filter_counts = adata.obs[forfiltering].value_counts()
print(f"\nGroups feeding completeness filter (column={forfiltering!r}, n={len(filter_counts)}):")
for g, n in filter_counts.items():
    print(f"  {g!r:30s} n={n}")
bio_counts = adata.obs[biological].value_counts()
print(f"\nDownstream biological groups (column={biological!r}, n={len(bio_counts)}):")
for g, n in bio_counts.items():
    print(f"  {g!r:30s} n={n}")
print()

# ------------------------------------------------------------
# 3. Contaminants
# ------------------------------------------------------------
# flag_contaminants (src/preprocessing.py) applies the keratin, cRAP and
# immunoglobulin rules and this list, the one of the 3_BrainTissue notebook.
# UniProt entry names.
adata.var["single_gene_name"] = adata.var["genes"].str.split(";").str[0].copy()

BRAIN_EXTRA_CONTAMINANTS = [
    # Keratins
    "K1C13_HUMAN","K1C18_HUMAN","K1C27_HUMAN",
    "K2C1B_HUMAN","K2C3_HUMAN","K2C4_HUMAN",
    "K2C72_HUMAN","K2C78_HUMAN","K2C79_HUMAN","K2C8_HUMAN",
    # Skin barrier / epidermal
    "HORN_HUMAN","FILA_HUMAN","FILA2_HUMAN",
    "DCD_HUMAN","CDSN_HUMAN","DMKN_HUMAN","SBSN_HUMAN","PROL4_HUMAN",
    "DSG1_HUMAN","DSC1_HUMAN","DSC3_HUMAN","DESP_HUMAN","PLAK_HUMAN",
    "TGM1_HUMAN","TGM3_HUMAN",
    # Skin/secretory + S100 family
    "SPB12_HUMAN","SG1D2_HUMAN","LCN1_HUMAN","PIP_HUMAN",
    "ZA2G_HUMAN","CYTA_HUMAN","CYTM_HUMAN","BPIA1_HUMAN",
    "S10A7_HUMAN","S10A8_HUMAN","S10A9_HUMAN","S10AB_HUMAN",
    # Serum / blood
    "ALBU_HUMAN","HBA_HUMAN","HBB_HUMAN","HBD_HUMAN",
    "APOA1_HUMAN","APOB_HUMAN",
    "TRFE_HUMAN","TRFL_HUMAN","LYSC_HUMAN","ANT3_HUMAN",
    # Immunoglobulins (the gene-symbol rule catches the IGK, IGL and IGH families)
    "IGHA1_HUMAN","IGHG1_HUMAN",
    # Lab / non-human
    "cRAP-TRYP_PIG","cRAP-K2M3_SHEEP","INS_HUMAN","TRY3_HUMAN",
    # Matrigel coating
    "CILP2_HUMAN",
]
adata.var = flag_contaminants(adata.var, extra=BRAIN_EXTRA_CONTAMINANTS)

# A protein group is a cRAP contaminant only if every one of its IDs is
# cRAP-prefixed. The cRAP database lists tau, which this tissue contains.
assert not adata.var.loc[adata.var["single_gene_name"].isin(["MAPT", "TAU"]),
                         "is_contaminant"].any(), "MAPT (tau) is flagged as a contaminant"

n_contaminants = int(adata.var["is_contaminant"].sum())
adata = adata[:, ~adata.var["is_contaminant"]].copy()
n_after_contaminants = adata.n_vars

adata_sf = adata.copy()      # samples filtered, contaminants removed, all proteins

# ------------------------------------------------------------
# 4. Completeness filter and log2 copy
# ------------------------------------------------------------
adata = complete_in_any_group(adata, forfiltering, MAX_MISSING_COND)

adata_log = adata.copy()
sc.pp.log1p(adata_log, base=2)

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------
print("=" * 64)
print("Bootstrap summary")
print("=" * 64)
print(f"Starting:                    {n_start_samples} samples × {n_start_proteins} proteins")
print(f"After blank filter:          {n_after_blanks} samples  (dropped {n_start_samples - n_after_blanks})")
print(f"After <{MIN_PROTEINS}-protein filter: {n_after_outliers} samples  (dropped {n_after_blanks - n_after_outliers})")
print(f"After NaN-{biological} filter: {n_after_nan} samples  (dropped {n_after_outliers - n_after_nan})")
print(f"After contaminant filter:    {n_after_contaminants} proteins  (dropped {n_contaminants})")
print("-" * 64)
print(f"adata_sf       (all non-contaminant):    {adata_sf.n_obs} × {adata_sf.n_vars}")
print(f"adata          (80% per condition):      {adata.n_obs} × {adata.n_vars}")
print(f"adata_log      (80% → log2):             {adata_log.n_obs} × {adata_log.n_vars}")
print("=" * 64)

# The iPSC and HEK cells reuse the generic names, so the brain objects are kept
# under the suffix _brain.
adata_sf_brain               = adata_sf
adata_unfiltered_brain       = adata_unfiltered
adata_log_brain              = adata_log
print("Brain-tissue variables published with suffix '_brain'.")

## 2.2 iPSC-derived neurons

In [ ]:
# iPSC-derived dopaminergic neurons: load the DIA-NN protein groups and filter them
# as in the 2_iDA notebook. Later cells read adata_sf_ipsc (detection per gene),
# adata_log_ipsc and adata_strict_no_contaminants_ipsc (the t-tests of section 3.2).
input_dir = IPSC_DIR

pg_table_path = f'{input_dir}/report.pg_matrix.tsv'
obs_metadata_path = f'{input_dir}/metadata.csv'

adata = at.io.read_pg_table(pg_table_path, search_engine="diann")
obs_metadata = pd.read_csv(obs_metadata_path, sep=",", index_col=0)

# Sample metadata onto obs; samples without a metadata row are dropped.
adata = at.pp.add_metadata(
    adata=adata,
    incoming_metadata=obs_metadata,
    axis=0,
    keep_data_shape=False,
)
adata.var = adata.var.merge(var_metadata, left_index=True, right_index=True, how="left")

# --- Parameters ---
biological = "class_clean"   # condition column
MIN_PROTEINS = 2000          # samples with fewer protein groups are excluded

adata.obs["num_proteins"] = np.sum(~np.isnan(adata.X), axis=1)

# --- Sample filtering ---
adata.obs["is_outlier"] = adata.obs["num_proteins"] < MIN_PROTEINS
adata_unfiltered = adata.copy()      # all samples before the protein-count filter
adata = adata_unfiltered[~adata_unfiltered.obs["is_outlier"], :].copy()

# --- Gene names and contaminant flags (src/preprocessing.py) ---
# Keratins, cRAP-only protein groups and immunoglobulins are flagged in every
# notebook; extra_contaminants is the list of the 2_iDA notebook: the serum, skin,
# secretion and coating proteins of this culture (UniProt entry names or gene
# symbols). The flags are set before the copies below, so every object carries
# them. Only the t-test input removes the flagged proteins; adata_sf and adata
# keep them, as in the 2_iDA notebook.
adata.var["single_gene_name"] = adata.var["genes"].str.split(";").str[0].copy()
extra_contaminants = [
    'K1C13_HUMAN', 'K1C18_HUMAN', 'K2C1B_HUMAN', 'K2C3_HUMAN',
    'K2C4_HUMAN', 'K2C72_HUMAN', 'K2C78_HUMAN', 'K2C79_HUMAN', 'K2C8_HUMAN',
    'HORN_HUMAN', 'FILA_HUMAN', 'FILA2_HUMAN', 'DCD_HUMAN', 'CDSN_HUMAN',
    'DSG1_HUMAN', 'DSC1_HUMAN', 'DSC3_HUMAN', 'DESP_HUMAN', 'PLAK_HUMAN',
    'TGM1_HUMAN', 'TGM3_HUMAN',
    'DMKN_HUMAN', 'CYTM_HUMAN', 'CYTA_HUMAN', 'S10AB_HUMAN',
    'PROL4_HUMAN',
    'ALBU_HUMAN', 'HBA_HUMAN', 'HBB_HUMAN', 'HBD_HUMAN',
    'TRFE_HUMAN', 'TRFL_HUMAN', 'LYSC_HUMAN', 'APOA1_HUMAN', 'ANT3_HUMAN',
    'IGHA1_HUMAN', 'IGHG1_HUMAN',
    'cRAP-TRYP_PIG', 'cRAP-K2M3_SHEEP',
    'INS_HUMAN', 'TRY3_HUMAN',
    'AZGP1', 'S100A7', 'BPIFA1',   # gene symbols of the next three entries
    'ZA2G_HUMAN',     # AZGP1 — zinc-α-2-glycoprotein (serum/saliva)
    'S10A7_HUMAN',    # S100A7 / psoriasin — skin
    'BPIA1_HUMAN',    # BPIFA1 / SPLUNC1 — secretory
    'CILP2_HUMAN',    # CILP2 — cartilage, probably from the matrigel coating
]
adata.var = flag_contaminants(adata.var, extra=extra_contaminants)

adata_sf = adata.copy()      # samples filtered, all proteins

# --- Completeness filter ---
# adata: no missing value in at least one condition. adata_strict: no missing
# value in any sample.
adata = complete_in_any_group(adata, biological, 0)

adata_strict = adata[:, at.pp.filter_data_completeness(
    adata, max_missing=0, action="flag"
).var["passed_threshold_missing_values"]].copy()

# --- Log2 copy, log2(x + 1), missing values kept ---
adata_log = adata.copy()
sc.pp.log1p(adata_log, base=2)

# --- t-test input: complete in every sample, without contaminants, log2 ---
adata_strict_no_contaminants = adata_strict[:, ~adata_strict.var["is_contaminant"]].copy()
sc.pp.log1p(adata_strict_no_contaminants, base=2)

print(f"Bootstrap complete: {adata.n_obs} samples, {adata.n_vars} proteins")
print(f"  adata_sf:                     {adata_sf.n_vars} proteins (all, sample-filtered)")
print(f"  contaminants flagged:         {int(adata_sf.var['is_contaminant'].sum())} of {adata_sf.n_vars}")
print(f"  adata_log:                    {adata_log.n_vars} proteins (log2)")
print(f"  adata_strict_no_contaminants: {adata_strict_no_contaminants.n_vars} proteins (t-test input)")

# The HEK cell reuses the generic names, so the iPSC objects are kept under the
# suffix _ipsc.
adata_sf_ipsc                     = adata_sf
adata_unfiltered_ipsc             = adata_unfiltered
adata_strict_no_contaminants_ipsc = adata_strict_no_contaminants
adata_log_ipsc                    = adata_log
biological_ipsc                   = biological
print("iPSC variables published with suffix '_ipsc'.")

## 2.3 HEK293 cells

In [ ]:
# HEK293 cells (Autophagy experiment): load the DIA-NN protein groups and filter them
# as in the 1_HEK notebook. Later cells read adata_sf_autoph (detection per gene)
# and adata_log_autoph (the t-test of section 3.1).
input_dir = HEK_DIR

pg_table_path = f'{input_dir}/report.pg_matrix.tsv'
obs_metadata_path = f'{input_dir}/metadata.csv'

adata = at.io.read_pg_table(pg_table_path, search_engine="diann")
obs_metadata = pd.read_csv(obs_metadata_path, sep=",", index_col=0)

# Sample metadata onto obs; samples without a metadata row are dropped.
adata = at.pp.add_metadata(
    adata=adata,
    incoming_metadata=obs_metadata,
    axis=0,
    keep_data_shape=False,
)
adata.var = adata.var.merge(var_metadata, left_index=True, right_index=True, how="left")

# --- Parameters ---
biological   = "class_renamed"   # condition column
MIN_PROTEINS = 2000              # samples with fewer protein groups are excluded
MAX_MISSING_COND = 0.0           # no missing value allowed within a condition (step 4)

# ------------------------------------------------------------
# 1. Protein groups per sample
# ------------------------------------------------------------
adata.obs["num_proteins"] = np.sum(~np.isnan(adata.X), axis=1)
n_start_samples, n_start_proteins = adata.shape

# ------------------------------------------------------------
# 2. Sample filters: blanks, fewer than MIN_PROTEINS protein groups, missing label
# ------------------------------------------------------------
adata = adata[~adata.obs[biological].isin(["Blank", "EvoTip Blank"]), :].copy()
n_after_blanks = adata.n_obs

adata.obs["is_outlier"] = adata.obs["num_proteins"] < MIN_PROTEINS
adata_unfiltered = adata.copy()      # all samples before the protein-count filter
adata = adata_unfiltered[~adata_unfiltered.obs["is_outlier"], :].copy()
n_after_outliers = adata.n_obs

# A missing label would become a group of its own in the completeness filter
nan_mask = adata.obs[biological].isna()
if nan_mask.any():
    print(f"WARNING: dropping {nan_mask.sum()} samples with NaN in {biological}")
    adata = adata[~nan_mask, :].copy()
n_after_nan = adata.n_obs

bio_counts = adata.obs[biological].value_counts()
print(f"\nBiological groups (column={biological!r}, n={len(bio_counts)}):")
for g, n in bio_counts.items():
    print(f"  {g!r:45s} n={n}")
print()

# ------------------------------------------------------------
# 3. Contaminants: keratins, cRAP-only protein groups, immunoglobulins and the
#    default list of skin, serum and laboratory proteins (src/preprocessing.py),
#    as in the 1_HEK notebook
# ------------------------------------------------------------
adata.var["single_gene_name"] = adata.var["genes"].str.split(";").str[0].copy()
adata.var = flag_contaminants(adata.var)

n_contaminants = int(adata.var["is_contaminant"].sum())
adata = adata[:, ~adata.var["is_contaminant"]].copy()
n_after_contaminants = adata.n_vars

adata_sf = adata.copy()      # samples filtered, contaminants removed, all proteins

# ------------------------------------------------------------
# 4. Completeness filter and log2 copy
#    adata: no missing value in at least one condition.
# ------------------------------------------------------------
adata = complete_in_any_group(adata, biological, MAX_MISSING_COND)

adata_log = adata.copy()
sc.pp.log1p(adata_log, base=2)

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------
print("=" * 64)
print("Bootstrap summary")
print("=" * 64)
print(f"Starting:                    {n_start_samples} samples × {n_start_proteins} proteins")
print(f"After blank filter:          {n_after_blanks} samples  (dropped {n_start_samples - n_after_blanks})")
print(f"After <{MIN_PROTEINS}-protein filter: {n_after_outliers} samples  (dropped {n_after_blanks - n_after_outliers})")
print(f"After NaN-{biological} filter: {n_after_nan} samples  (dropped {n_after_outliers - n_after_nan})")
print(f"After contaminant filter:    {n_after_contaminants} proteins  (dropped {n_contaminants})")
print("-" * 64)
print(f"adata_sf          (all non-contaminant):       {adata_sf.n_obs} × {adata_sf.n_vars}")
print(f"adata             (100% per condition):        {adata.n_obs} × {adata.n_vars}")
print(f"adata_log         (per-cond → log2):           {adata_log.n_obs} × {adata_log.n_vars}")
print("=" * 64)

adata_sf_autoph               = adata_sf
adata_unfiltered_autoph       = adata_unfiltered
adata_log_autoph              = adata_log
biological_autoph             = biological
print("Autophagy variables published with suffix '_autoph'.")

## 2.4 Style and system colours

In [ ]:
# ============================================================
# Style and system colours
# Page grid, fonts, line widths, colour roles, save_panel and compose_figure come
# from src/figure_style.py (imported in section 1), shared by all notebooks of the
# project.
# ============================================================
print(style_summary())

# ---------------------------------------------------------------- SYSTEMS
# The system is an axis of its own, separate from fraction, category and
# direction, so it takes hues that none of those use. Paul Tol "bright" entries,
# distinguishable under the common colour-vision deficiencies: indigo is dark
# enough to separate from the CWO blue (#7fb3d8) and from the direction blue
# (#2166ac), which no panel shows together with it.
SYSTEM_COLORS = {
    "HEK":   "#4E79A7",   # steel blue
    "iPSC":  "#7B2D8E",   # plum
    "Brain": "#2F3E46",   # charcoal
}

# 3. Differential abundance per system

## 3.1 HEK293: Welch t-test

In [ ]:
# HEK: Welch t-test of excised aggregates against rest cells, PFF without
# bafilomycin A1, on adata_log_autoph (log2, missing values kept). This is the HEK
# contrast of every cross-system panel. Result: ttest_results_autoph.
print(f"Autophagy t-test input (adata_log): "
      f"{adata_log_autoph.n_obs} × {adata_log_autoph.n_vars}")

comparisons_autoph = {
    "PureAggs_vs_RestCells_minusBafA1": ("-BafA1 PFF Pure Aggregates",
                                          "-BafA1 PFF Rest Cells"),
}

ttest_results_autoph = {}
for name, (group1, group2) in comparisons_autoph.items():
    result = at.tl.diff_exp_ttest(
        adata=adata_log_autoph,
        between_column=biological_autoph,
        comparison=(group1, group2),
    )
    if result is None:
        print(f"{name}: FAILED")
        continue

    result = result.merge(
        adata_log_autoph.var[["single_gene_name"]],
        left_on="protein", right_index=True, how="left",
    )
    cols = ["protein", "single_gene_name"] + [c for c in result.columns
                                              if c not in ("protein", "single_gene_name")]
    result = result[cols]
    if "-log10(fdr)" not in result.columns:
        result["-log10(fdr)"] = -np.log10(result["fdr"].clip(lower=1e-300))
    ttest_results_autoph[name] = result

    n_sig  = int((result["fdr"] < 0.05).sum())
    n_up   = int(((result["fdr"] < 0.05) & (result["log2fc"] >  1)).sum())
    n_down = int(((result["fdr"] < 0.05) & (result["log2fc"] < -1)).sum())
    print(f"{name}: {len(result)} tested, {n_sig} sig (FDR<0.05), "
          f"{n_up} up, {n_down} down")
    result.to_csv(f"{output_dir}/{today}_ttest_autoph_{name}.csv", index=False)

print(f"\nAutophagy: {len(ttest_results_autoph)}/{len(comparisons_autoph)} contrasts saved.")

## 3.2 iPSC: Welch t-tests

In [ ]:
# iPSC: Welch t-tests on adata_strict_no_contaminants_ipsc (proteins quantified in
# every sample, contaminants removed, log2), as in the 2_iDA notebook.
# Result: ttest_results_ipsc.
adata_to_use   = adata_strict_no_contaminants_ipsc
filter_label   = "strict_no_contaminants"
print(f"iPSC t-test input ({filter_label}): "
      f"{adata_to_use.n_obs} × {adata_to_use.n_vars}")

comparisons_ipsc = {
    "PBS_Neurites_vs_Soma": (
        "iDA D56 PBS N1 Neurites Without Aggregates",
        "iDA D56 PBS N1 Soma Without Aggregates",
    ),
    "PFF_Neurites_vs_Soma": (
        "iDA D56 PFF N2 Neurites Without Aggregates",
        "iDA D56 PFF N2 Soma Without Aggregates",
    ),
    "Neurites_PFF_vs_PBS": (
        "iDA D56 PFF N2 Neurites Without Aggregates",
        "iDA D56 PBS N1 Neurites Without Aggregates",
    ),
    "Soma_PFF_vs_PBS": (
        "iDA D56 PFF N2 Soma Without Aggregates",
        "iDA D56 PBS N1 Soma Without Aggregates",
    ),
    "Neuritic_Agg_vs_Neurites": (
        "iDA D56 PFF N2 Aggregates Neuritic",
        "iDA D56 PFF N2 Neurites Without Aggregates",
    ),
    "Somatic_Agg_vs_Soma": (
        "iDA D56 PFF N2 Aggregates Somatic",
        "iDA D56 PFF N2 Soma Without Aggregates",
    ),
}

ttest_results_ipsc = {}
for name, (group1, group2) in comparisons_ipsc.items():
    result = at.tl.diff_exp_ttest(
        adata=adata_to_use,
        between_column=biological_ipsc,
        comparison=(group1, group2),
    )
    if result is None:
        print(f"{name}: FAILED")
        continue

    result = result.merge(
        adata_to_use.var[["single_gene_name", "proteins", "uniprot_ids"]],
        left_on="protein", right_index=True, how="left",
    )
    front = ["single_gene_name", "proteins", "uniprot_ids", "protein"]
    cols  = front + [c for c in result.columns if c not in front]
    result = result[cols].sort_values("log2fc", ascending=False)
    if "-log10(fdr)" not in result.columns:
        result["-log10(fdr)"] = -np.log10(result["fdr"].clip(lower=1e-300))
    ttest_results_ipsc[name] = result

    n_sig  = int((result["fdr"] < 0.05).sum())
    n_up   = int(((result["fdr"] < 0.05) & (result["log2fc"] >  1)).sum())
    n_down = int(((result["fdr"] < 0.05) & (result["log2fc"] < -1)).sum())
    print(f"{name}: {len(result)} tested, {n_sig} sig (FDR<0.05), "
          f"{n_up} up, {n_down} down")
    result.to_csv(f"{output_dir}/{today}_ttest_ipsc_{name}.csv", index=False)

print(f"\niPSC: {len(ttest_results_ipsc)}/{len(comparisons_ipsc)} contrasts saved.")

## 3.3 Brain: eBayes on donor pseudobulk

In [ ]:
# Brain: limma eBayes on donor-level pseudobulk, all brain regions together, for the
# four contrasts the cross-system panels read. The samples of one donor and one
# group are averaged first, so the donor is the unit of replication.
# Result: ebayes_results_brain.
DONOR_COL_BRAIN  = "Brain"
MIN_DONORS_BRAIN = 3         # a contrast with fewer donors in either group is skipped

adata_eb_brain = adata_log_brain   # 80 % per group, log2, missing values kept
print(f"BRAIN eBayes input: "
      f"{adata_eb_brain.n_obs} samples × {adata_eb_brain.n_vars} proteins")

# 'fraction' column of the group definitions below
fraction_map = {
    "Neuron Without Aggregate":      "Neuron | Without Aggregate",
    "Neuron DAB Positive":           "Neuron | DAB Positive",
    "Glia Without Aggregate":        "Glia | Without Aggregate",
    "Glia DAB Positive":             "Glia | DAB Positive",
    "Neuropil Without Aggregate":    "Neuropil",
    "Round Aggregate Aggregate":     "Round Aggregate",
    "Elongated Aggregate Aggregate": "Elongated Aggregate",
}
adata_eb_brain.obs["fraction"] = (
    adata_eb_brain.obs["CellType_Aggregate"].map(fraction_map)
)
unmapped = adata_eb_brain.obs["fraction"].isna().sum()
if unmapped:
    print(f"WARNING: {unmapped} samples have no fraction mapping. "
          f"Unique unmapped CellType_Aggregate: "
          f"{set(adata_eb_brain.obs.loc[adata_eb_brain.obs['fraction'].isna(),'CellType_Aggregate']) }")

_gene_map_brain = adata_eb_brain.var["single_gene_name"].to_dict()

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def _mask_for(adata, group_dict):
    """Boolean row mask: ALL key:value pairs must hold.
    Values may be scalar or list/tuple/set (any-of semantics)."""
    mask = np.ones(adata.n_obs, dtype=bool)
    for k, v in group_dict.items():
        col = adata.obs[k].astype(str).values
        if isinstance(v, (list, tuple, set)):
            mask &= np.isin(col, [str(x) for x in v])
        else:
            mask &= (col == str(v))
    return mask

def _pseudobulk_pair(adata, mask_a, mask_b,
                     donor_col=DONOR_COL_BRAIN,
                     min_donors=MIN_DONORS_BRAIN):
    """Collapse to one row per (donor, group): the mean of the donor's samples,
    missing values ignored. Returns (pseudobulk_adata, n_donors_a, n_donors_b), or
    (None, n_donors_a, n_donors_b) if either group has fewer than min_donors."""
    obs = adata.obs
    donors_a = sorted(obs.loc[mask_a, donor_col].astype(str).unique())
    donors_b = sorted(obs.loc[mask_b, donor_col].astype(str).unique())
    if len(donors_a) < min_donors or len(donors_b) < min_donors:
        return None, len(donors_a), len(donors_b)

    rows_X, idx, contrast = [], [], []
    for d in donors_a:
        sel = mask_a & (obs[donor_col].astype(str).values == d)
        if sel.sum() == 0:
            continue
        rows_X.append(np.nanmean(adata.X[sel], axis=0))
        idx.append(f"{d}__A");  contrast.append("A")
    for d in donors_b:
        sel = mask_b & (obs[donor_col].astype(str).values == d)
        if sel.sum() == 0:
            continue
        rows_X.append(np.nanmean(adata.X[sel], axis=0))
        idx.append(f"{d}__B");  contrast.append("B")

    pb = ad.AnnData(
        X=np.vstack(rows_X),
        obs=pd.DataFrame({"__contrast": contrast}, index=idx),
        var=adata.var.copy(),
    )
    return pb, len(donors_a), len(donors_b)

# ------------------------------------------------------------
# Contrasts: group A against group B. Names and group definitions as in the
# 3_BrainTissue notebook.
# ------------------------------------------------------------
contrasts_brain = [
    # Disease vs Control within one fraction (DCV heatmap, Figure S7 b)
    dict(name="Neuron_DABneg_Disease_vs_Control",
         group_a={"fraction": "Neuron | Without Aggregate", "disease_binary": "Disease"},
         group_b={"fraction": "Neuron | Without Aggregate", "disease_binary": "Control"}),
    dict(name="Neuropil_Disease_vs_Control",
         group_a={"fraction": "Neuropil", "disease_binary": "Disease"},
         group_b={"fraction": "Neuropil", "disease_binary": "Control"}),
    # Excised inclusions against their host fraction
    dict(name="Round_vs_NeuronDABpos_Disease",
         group_a={"fraction": "Round Aggregate", "disease_binary": "Disease"},
         group_b={"fraction": "Neuron | DAB Positive", "disease_binary": "Disease"}),
    dict(name="Elong_vs_Neuropil_Disease",
         group_a={"fraction": "Elongated Aggregate", "disease_binary": "Disease"},
         group_b={"fraction": "Neuropil", "disease_binary": "Disease"}),
]

# ------------------------------------------------------------
# Run
# ------------------------------------------------------------
ebayes_results_brain = {}
for c in contrasts_brain:
    name = c["name"]
    ad_sub = adata_eb_brain.copy()

    pb, n_a, n_b = _pseudobulk_pair(
        ad_sub,
        _mask_for(ad_sub, c["group_a"]),
        _mask_for(ad_sub, c["group_b"]),
    )
    if pb is None:
        print(f"{name}: SKIPPED (donors A={n_a}, B={n_b})")
        continue

    _, res = at.tl.diff_exp_ebayes(pb, "__contrast", ("A", "B"))
    res = res.rename(columns={
        "logFC": "log2fc", "P.Value": "p_value", "adj.P.Val": "fdr",
        "pvalue": "p_value", "padj": "fdr", "adj_pvalue": "fdr",
    })
    if "protein" not in res.columns:
        res = res.reset_index().rename(columns={"index": "protein"})
    if "single_gene_name" not in res.columns:
        res["single_gene_name"] = res["protein"].map(_gene_map_brain)
    if "-log10(fdr)" not in res.columns:
        res["-log10(fdr)"] = -np.log10(res["fdr"].clip(lower=1e-300))

    ebayes_results_brain[name] = res
    n_sig = int((res["fdr"] < 0.05).sum())
    print(f"{name}: donors A={n_a} B={n_b}, tested={len(res)}, sig={n_sig}")

print(f"\nBrain eBayes: {len(ebayes_results_brain)}/{len(contrasts_brain)} contrasts run.")

# 4. Organelle modules from the Gene Ontology

Inputs in `data/CrossSystem/Organelles/`: the Gene Ontology `go-basic.obo` (https://current.geneontology.org/ontology/go-basic.obo), the human GO annotations `goa_human.gaf.gz` (https://current.geneontology.org/annotations/goa_human.gaf.gz), and `SupplementaryTable_organelle_modules.csv`, the list of organelle modules (one GO cellular-component term each), curated by hand with the help of Claude (Anthropic). The cell writes the member table of the modules (Figure 5d, Figure S8 a–m). It comes before the cross-system cells because section 8 and section 12.2 also read its parse of the Gene Ontology.

In [ ]:
# ============================================================
# Organelle modules from the Gene Ontology (Figure 5 d, Figure S8 a-m, the
# supplementary module table)
#
#   Builds the member list of every module from go-basic.obo and
#   goa_human.gaf.gz. SupplementaryTable_organelle_modules.csv names the modules
#   (one GO cellular-component term each); the members are derived here, so the
#   table is reproducible from the two GO files.
#
#   Membership is the part_of descendant closure of the module term. The is_a
#   closure is not used: it adds subtypes, so the lysosomal membrane would gain
#   the azurophil granule, autolysosome, phagolysosome and cytolytic granule
#   membranes. n_genes_isa records the size under the is_a closure for comparison.
#
#   The parsed ontology (_terms, _part_kids) and the annotations (_term2genes)
#   are also read by the FTD/ALS control and the complex cells (GO route of the
#   split candidates).
# ============================================================
ORG_MIN_MEMBERS = 4        # below this a module is not scored and no bar is drawn


def _parse_obo(path):
    """Terms with namespace, obsolete flag, and is_a / part_of edges kept apart."""
    terms, version, cur = {}, None, None
    with open(path) as fh:
        for line in fh:
            line = line.rstrip("\n")
            if version is None and line.startswith("data-version:"):
                version = line.split(":", 1)[1].strip()
            if line == "[Term]":
                if cur and cur["id"]:
                    terms[cur["id"]] = cur
                cur = dict(id=None, name=None, ns=None, obsolete=False,
                           is_a=[], part_of=[], alt=[])
                continue
            if line.startswith("["):                 # [Typedef] and anything else
                if cur and cur["id"]:
                    terms[cur["id"]] = cur
                cur = None
                continue
            if cur is None or not line:
                continue
            k, _, v = line.partition(": ")
            v = v.strip()
            if   k == "id":          cur["id"] = v
            elif k == "name":        cur["name"] = v
            elif k == "namespace":   cur["ns"] = v
            elif k == "is_obsolete": cur["obsolete"] = (v == "true")
            elif k == "alt_id":      cur["alt"].append(v)
            elif k == "is_a":        cur["is_a"].append(v.split()[0])
            elif k == "relationship" and v.startswith("part_of "):
                cur["part_of"].append(v.split()[1])
    if cur and cur["id"]:
        terms[cur["id"]] = cur
    return terms, version


_terms, GO_VERSION = _parse_obo(f"{ORG_DIR}/go-basic.obo")
_cc = {t: d for t, d in _terms.items()
       if d["ns"] == "cellular_component" and not d["obsolete"]}
_alt2main = {a: t for t, d in _terms.items() for a in d["alt"]}
print(f"go-basic.obo {GO_VERSION}: {len(_terms)} terms, "
      f"{len(_cc)} current cellular-component terms")

# --- the module table ---------------------------------------------------
org_table = pd.read_csv(f"{ORG_DIR}/SupplementaryTable_organelle_modules.csv")

# CHECK 1: every module identifier is a current, non-obsolete CC term.
_bad = []
for _g, _n in zip(org_table.go_id, org_table.go_term_name):
    _d = _terms.get(_g)
    if _d is None:
        _bad.append((_g, _n, "not in the ontology"))
    elif _d["obsolete"]:
        _bad.append((_g, _n, "obsolete"))
    elif _d["ns"] != "cellular_component":
        _bad.append((_g, _n, f"namespace is {_d['ns']}"))
if _bad:
    raise AssertionError("module terms that are not current CC terms:\n  "
                         + "\n  ".join(f"{g} {n}: {why}" for g, n, why in _bad))
print(f"CHECK all {len(org_table)} module terms are current cellular-component terms")

# The module names are taken from the ontology; a name that differs from the
# table is printed.
_drift = [(g, n, _cc[g]["name"]) for g, n in zip(org_table.go_id, org_table.go_term_name)
          if _cc[g]["name"] != n]
if _drift:
    print("term names have changed in the ontology since the table was written:")
    for g, old, new in _drift:
        print(f"  {g}: '{old}' -> '{new}'")
org_table["go_term_name"] = org_table.go_id.map(lambda g: _cc[g]["name"])

# --- closures -----------------------------------------------------------
_part_kids, _isa_kids = defaultdict(list), defaultdict(list)
for _t, _d in _cc.items():
    for _p in _d["part_of"]:
        _part_kids[_p].append(_t)
    for _p in _d["is_a"]:
        _isa_kids[_p].append(_t)


def _closure(root, *kid_maps):
    seen, stack = {root}, [root]
    while stack:
        n = stack.pop()
        for km in kid_maps:
            for c in km.get(n, []):
                if c not in seen:
                    seen.add(c)
                    stack.append(c)
    return seen


# --- the annotation file ------------------------------------------------
# Rows kept: aspect C, qualifier not starting with NOT, object type protein. The
# GAF also annotates complexes and RNAs, which the protein tables never contain as
# rows of their own. Alternative GO IDs are mapped to the main ID.
_rows, GAF_VERSION = [], None
with gzip.open(f"{ORG_DIR}/goa_human.gaf.gz", "rt") as fh:
    for line in fh:
        if line.startswith("!"):
            if GAF_VERSION is None and line.startswith("!date-generated:"):
                GAF_VERSION = line.split(":", 1)[1].strip()
            continue
        f = line.rstrip("\n").split("\t")
        if f[8] != "C" or f[11] != "protein" or f[3].startswith("NOT"):
            continue
        _rows.append((f[2].upper().strip(), f[4]))

_gaf = pd.DataFrame(_rows, columns=["gene", "go_id"]).drop_duplicates()
_gaf["go_id"] = _gaf.go_id.map(lambda g: _alt2main.get(g, g))
_term2genes = _gaf.groupby("go_id").gene.apply(set).to_dict()
print(f"goa_human.gaf.gz generated {GAF_VERSION}: {len(_gaf)} protein CC annotations, "
      f"{_gaf.gene.nunique()} symbols")

_recs = []
for _, r in org_table.iterrows():
    cl_p = _closure(r.go_id, _part_kids)
    cl_i = _closure(r.go_id, _part_kids, _isa_kids)
    gp = set().union(*[_term2genes.get(t, set()) for t in cl_p], set())
    gi = set().union(*[_term2genes.get(t, set()) for t in cl_i], set())
    _recs.append(dict(organelle=r.organelle, go_id=r.go_id, module=r.go_term_name,
                      n_terms=len(cl_p), genes=gp, n_annotated=len(gp),
                      n_genes_isa=len(gi)))
org_modules = pd.DataFrame(_recs)

# --- within one organelle, a gene may belong to only one module ----------
# A gene annotated to two modules of one organelle stays in the module with the
# smaller annotated set (ties: the lower GO ID). Removing it from both would empty
# every complex that is part of a compartment: the translocon, the TOM and TIM23
# translocases and the two mitoribosomal subunits would lose every member to
# their membrane or matrix module. With this rule a complex module is compared
# with the rest of its compartment, e.g. the translocon with the rest of the ER
# membrane.
org_modules["n_before_overlap"] = org_modules.n_annotated
_reassigned = []
for _org, _idx in org_modules.groupby("organelle").groups.items():
    _owner = {}                      # gene -> index of the smallest module holding it
    for i in _idx:
        for _gene in org_modules.at[i, "genes"]:
            _cur = _owner.get(_gene)
            if _cur is None or (org_modules.at[i, "n_before_overlap"],
                                org_modules.at[i, "go_id"]) < \
                               (org_modules.at[_cur, "n_before_overlap"],
                                org_modules.at[_cur, "go_id"]):
                _owner[_gene] = i
    for i in _idx:
        _lost = {g for g in org_modules.at[i, "genes"] if _owner[g] != i}
        if _lost:
            _reassigned.append((_org, org_modules.at[i, "module"], len(_lost)))
            org_modules.at[i, "genes"] = org_modules.at[i, "genes"] - _lost
org_modules["n_annotated"] = org_modules.genes.map(len)
if _reassigned:
    print(f"shared genes given to the smaller module, taken from "
          f"{len(_reassigned)} larger ones:")
    for _o, _m, _n in sorted(_reassigned, key=lambda r: -r[2])[:8]:
        print(f"  {_o} / {_m}: -{_n}")

# CHECK 2: no gene appears in two modules of the same organelle.
for _org, _idx in org_modules.groupby("organelle").groups.items():
    _seen = set()
    for i in _idx:
        _g = org_modules.at[i, "genes"]
        assert not (_seen & _g), f"{_org}: {sorted(_seen & _g)[:5]} in two modules"
        _seen |= _g
print(f"CHECK no gene is in two modules of the same organelle "
      f"({sum(n for _, _, n in _reassigned)} memberships moved to the more "
      f"specific module)")

ORG_VERSION_NOTE = f"GO {GO_VERSION}, GOA generated {GAF_VERSION}"
_small = org_modules[org_modules.n_annotated < ORG_MIN_MEMBERS]
if len(_small):
    print(f"\n{len(_small)} modules are annotated with fewer than {ORG_MIN_MEMBERS} "
          f"proteins under the part_of rule and can never be scored:")
    print(_small[["organelle", "go_id", "module", "n_before_overlap",
                  "n_annotated", "n_genes_isa"]].to_string(index=False))

org_modules.assign(genes=org_modules.genes.map(lambda s: ";".join(sorted(s)))).to_csv(
    f"{output_dir}/SupplementaryTable_organelle_modules_members.csv", index=False)
print(f"\n{len(org_modules)} modules in {org_modules.organelle.nunique()} organelles | "
      f"{ORG_VERSION_NOTE}")
print(org_modules.groupby("organelle", sort=False)
      .agg(modules=("go_id", "size"), proteins=("n_annotated", "sum")).to_string())

# 5. Shared signature across systems (Figure 5a)

In [ ]:
# ============================================================
# Three-system gene table of Figure 5 a: one log2FC and one FDR per gene and
# system, and the shared-up / shared-down classification.
#   HEK    excised aggregates vs rest cells, PFF, without bafilomycin A1
#   iPSC   somatic aggregates vs soma, PFF
#   brain  round inclusions vs DAB+ neurons, disease donors
#
# Rules set before the data were inspected:
#   detection:    a gene is kept per system if its best-detected protein has a
#                 value in >= MIN_DETECTION_FRAC of the samples of adata_sf_<system>
#   shared-UP   = log2FC >= +UNIV_LFC in all three contrasts
#   shared-DOWN = log2FC <= -UNIV_LFC in all three contrasts
#   none        = every other gene
# The classification below also requires FDR < UNIV_FDR (see there).
# The detection tables det_* and the lookup _gene_values are used by every later
# cross-system cell.
# ============================================================
MIN_DETECTION_FRAC = 0.50
UNIV_LFC = 0.3

HEK_KEY   = "PureAggs_vs_RestCells_minusBafA1"
IPSC_KEY  = "Somatic_Agg_vs_Soma"
BRAIN_KEY = "Round_vs_NeuronDABpos_Disease"

print("Sign convention (positive log2FC = enriched in the FIRST group):")
print(f"  HEK  : {HEK_KEY}   →   '−BafA1 PFF Pure Aggregates' vs Rest Cells")
print(f"  iPSC : {IPSC_KEY}  →   'PFF N2 Aggregates Somatic'  vs PFF N2 Soma")
print(f"  brain: {BRAIN_KEY} →   'Round Aggregate (Disease)'  vs 'Neuron DAB+ (Disease)'")
print(f"All three are aggregate-side vs host-side; shared-UP = enriched in aggregates.\n")

# --- Detection per gene and system ---
def _gene_detection_table(adata, min_frac=MIN_DETECTION_FRAC):
    """Per gene symbol, the protein with the highest fraction of samples with a
    value, and that fraction."""
    det = pd.DataFrame({
        "gene":    adata.var["single_gene_name"].astype(str).values,
        "protein": adata.var.index.values,
        "frac":    (~np.isnan(adata.X)).mean(axis=0),
    })
    det = det.dropna(subset=["gene"])
    det["gene"] = det["gene"].str.upper().str.strip()
    det = (det.sort_values("frac", ascending=False)
              .drop_duplicates("gene", keep="first"))
    return det.set_index("gene")

det_brain  = _gene_detection_table(adata_sf_brain)
det_ipsc   = _gene_detection_table(adata_sf_ipsc)
det_autoph = _gene_detection_table(adata_sf_autoph)

genes_brain  = set(det_brain [det_brain ["frac"] >= MIN_DETECTION_FRAC].index)
genes_ipsc   = set(det_ipsc  [det_ipsc  ["frac"] >= MIN_DETECTION_FRAC].index)
genes_autoph = set(det_autoph[det_autoph["frac"] >= MIN_DETECTION_FRAC].index)
# sorted: the iteration order of a set of strings changes with every Python
# process, and the row order of df_cs sets the permutation draws of the
# correlation null and the drawing order of Figure 5a
intersection = sorted(genes_brain & genes_ipsc & genes_autoph)

print(f"Detected genes (≥{MIN_DETECTION_FRAC:.0%} non-NaN per system):")
print(f"  HEK   : {len(genes_autoph):>5}")
print(f"  iPSC  : {len(genes_ipsc  ):>5}")
print(f"  brain : {len(genes_brain ):>5}")
print(f"  intersection: {len(intersection):>5}")

# --- One value per gene from a result table ---
def _gene_values(result_df, det_df, column, genes=None):
    """One value of `column` per gene symbol: the row of the gene's best-detected
    protein (det_df). If that protein has no row in result_df, the mean over the
    gene's rows; for "fdr" the maximum, so the gene is only as well supported as
    its least significant protein. genes=None returns every gene of result_df; a
    gene in `genes` without a row gets NaN."""
    r = result_df.copy()
    r["gene_u"] = r["single_gene_name"].astype(str).str.upper().str.strip()
    on_best = r["protein"] == r["gene_u"].map(det_df["protein"])
    has_best = on_best.groupby(r["gene_u"]).any()
    exact = r[on_best].set_index("gene_u")[column]
    assert exact.index.is_unique, "two rows claim the same best protein"
    fallback = r.groupby("gene_u")[column].agg("max" if column == "fdr" else "mean")
    out = exact.reindex(has_best.index).where(has_best, fallback).astype(float)
    if genes is None:
        return out
    genes = list(genes)
    n_fallback = int((~has_best.reindex(genes, fill_value=True)).sum())
    if n_fallback:
        print(f"    ({n_fallback} genes of {len(genes)}: {column} from the multi-protein fallback)")
    return out.reindex(genes)

lfc_brain  = _gene_values(ebayes_results_brain[BRAIN_KEY], det_brain,  "log2fc", intersection)
fdr_brain  = _gene_values(ebayes_results_brain[BRAIN_KEY], det_brain,  "fdr",    intersection)
lfc_ipsc   = _gene_values(ttest_results_ipsc[IPSC_KEY],    det_ipsc,   "log2fc", intersection)
fdr_ipsc   = _gene_values(ttest_results_ipsc[IPSC_KEY],    det_ipsc,   "fdr",    intersection)
lfc_autoph = _gene_values(ttest_results_autoph[HEK_KEY],   det_autoph, "log2fc", intersection)
fdr_autoph = _gene_values(ttest_results_autoph[HEK_KEY],   det_autoph, "fdr",    intersection)

df_cs = pd.DataFrame({
    "HEK": lfc_autoph, "iPSC": lfc_ipsc, "brain": lfc_brain,
    "fdr_HEK": fdr_autoph, "fdr_iPSC": fdr_ipsc, "fdr_brain": fdr_brain,
})
n_before = len(df_cs)
df_cs = df_cs.dropna(subset=["HEK", "iPSC", "brain"])
print(f"\nAligned log2FC matrix: {len(df_cs)} / {n_before} intersection genes "
      f"have non-NaN log2FC in all three contrasts.")

# --- Classification ---
# The rules above threshold log2FC only, so three non-significant fold changes
# would make a gene shared. The classification therefore also requires
# FDR < UNIV_FDR in UNIV_MIN_SIG systems. category_lfc keeps the log2FC-only
# call; the table printed below gives both, as the Methods report them.
UNIV_FDR     = 0.05
UNIV_MIN_SIG = 3          # systems that must clear UNIV_FDR; 3 = all of them

_fdr_cols = ["fdr_HEK", "fdr_iPSC", "fdr_brain"]
_lfc_cols = ["HEK", "iPSC", "brain"]

def _classify_lfc(row, thresh=UNIV_LFC):
    if (row[_lfc_cols] >=  thresh).all(): return "shared-UP"
    if (row[_lfc_cols] <= -thresh).all(): return "shared-DOWN"
    return "none"

df_cs["category_lfc"] = df_cs.apply(_classify_lfc, axis=1)
df_cs["n_sig"] = (df_cs[_fdr_cols] < UNIV_FDR).sum(axis=1)
df_cs["category"] = np.where(df_cs["n_sig"] >= UNIV_MIN_SIG,
                             df_cs["category_lfc"], "none")

# log2FC-only calls split by the number of systems with FDR < UNIV_FDR
print(f"\nClassification (|log2FC| >= {UNIV_LFC} in ALL three, "
      f"FDR < {UNIV_FDR} in >= {UNIV_MIN_SIG} of 3):")
_lfc_only = df_cs["category_lfc"]
for lab in ("shared-UP", "shared-DOWN"):
    row = f"  {lab:<16}"
    for k in (0, 1, 2, 3):
        row += f"  >={k}sig: {int(((_lfc_only == lab) & (df_cs['n_sig'] >= k)).sum()):>4}"
    print(row)
counts = df_cs["category"].value_counts().reindex(
    ["shared-UP", "shared-DOWN", "none"]).fillna(0).astype(int)
print(counts.to_string())


_lfc_tag = f"lfc{UNIV_LFC:g}".replace(".", "p")   # 0.1 → 'lfc0p1', 1.0 → 'lfc1'
_csv_path = f"{output_dir}/{today}_crosssystem_aligned_{_lfc_tag}.csv"
df_cs.to_csv(_csv_path)
print(f"\nSaved → {_csv_path}")

up_genes   = df_cs.index[df_cs["category"] == "shared-UP"].tolist()
down_genes = df_cs.index[df_cs["category"] == "shared-DOWN"].tolist()
print(f"\nshared-UP   ({len(up_genes)}): {up_genes}")
print(f"shared-DOWN ({len(down_genes)}): {down_genes}")

In [ ]:
# ============================================================
# Two-system gene table, iPSC and brain only (HEK left out): the same rules as
# the three-system cell above, classification on these two systems.
# Result: df_cs_ib, read by the two-system dot plot (Figure 5 b) and the
# literature comparison (Figure S7 e, f).
# ============================================================
intersection_ib = sorted(genes_ipsc & genes_brain)   # sorted, as in the cell above
print(f"Detected genes (≥{MIN_DETECTION_FRAC:.0%} non-NaN per system):")
print(f"  iPSC  : {len(genes_ipsc ):>5}")
print(f"  brain : {len(genes_brain):>5}")
print(f"  iPSC ∩ brain: {len(intersection_ib):>5}")

lfc_ipsc_ib  = _gene_values(ttest_results_ipsc[IPSC_KEY],    det_ipsc,  "log2fc", intersection_ib)
fdr_ipsc_ib  = _gene_values(ttest_results_ipsc[IPSC_KEY],    det_ipsc,  "fdr",    intersection_ib)
lfc_brain_ib = _gene_values(ebayes_results_brain[BRAIN_KEY], det_brain, "log2fc", intersection_ib)
fdr_brain_ib = _gene_values(ebayes_results_brain[BRAIN_KEY], det_brain, "fdr",    intersection_ib)

df_cs_ib = pd.DataFrame({
    "iPSC": lfc_ipsc_ib, "brain": lfc_brain_ib,
    "fdr_iPSC": fdr_ipsc_ib, "fdr_brain": fdr_brain_ib,
})
n_before = len(df_cs_ib)
df_cs_ib = df_cs_ib.dropna(subset=["iPSC", "brain"])

print(f"\nAligned log2FC matrix: {len(df_cs_ib)} / {n_before} intersection genes "
      f"have non-NaN log2FC in both contrasts.")

# --- Classification ---
# As in the three-system cell: log2FC in both systems and FDR < UNIV_FDR in
# UNIV_MIN_SIG_IB systems; category_lfc keeps the log2FC-only call.
UNIV_MIN_SIG_IB = 2       # systems that must clear UNIV_FDR; 2 = both

_fdr_cols_ib = ["fdr_iPSC", "fdr_brain"]
_lfc_cols_ib = ["iPSC", "brain"]

def _classify_ib_lfc(row, thresh=UNIV_LFC):
    if (row[_lfc_cols_ib] >=  thresh).all(): return "shared-UP"
    if (row[_lfc_cols_ib] <= -thresh).all(): return "shared-DOWN"
    return "none"

df_cs_ib["category_lfc"] = df_cs_ib.apply(_classify_ib_lfc, axis=1)
df_cs_ib["n_sig"] = (df_cs_ib[_fdr_cols_ib] < UNIV_FDR).sum(axis=1)
df_cs_ib["category"] = np.where(df_cs_ib["n_sig"] >= UNIV_MIN_SIG_IB,
                                df_cs_ib["category_lfc"], "none")

print(f"\nClassification iPSC ∩ brain (|log2FC| >= {UNIV_LFC} in BOTH, "
      f"FDR < {UNIV_FDR} in >= {UNIV_MIN_SIG_IB} of 2):")
_lfc_only_ib = df_cs_ib["category_lfc"]
for lab in ("shared-UP", "shared-DOWN"):
    row = f"  {lab:<16}"
    for k in (0, 1, 2):
        row += f"  >={k}sig: {int(((_lfc_only_ib == lab) & (df_cs_ib['n_sig'] >= k)).sum()):>4}"
    print(row)
counts_ib = (df_cs_ib["category"]
             .value_counts()
             .reindex(["shared-UP", "shared-DOWN", "none"])
             .fillna(0).astype(int))
print(counts_ib.to_string())

_lfc_tag = f"lfc{UNIV_LFC:g}".replace(".", "p")
_csv_path_ib = f"{output_dir}/{today}_crosssystem_aligned_iPSC_brain_{_lfc_tag}.csv"
df_cs_ib.to_csv(_csv_path_ib)
print(f"\nSaved → {_csv_path_ib}")

up_genes_ib   = df_cs_ib.index[df_cs_ib["category"] == "shared-UP"].tolist()
down_genes_ib = df_cs_ib.index[df_cs_ib["category"] == "shared-DOWN"].tolist()
print(f"\nshared-UP   ({len(up_genes_ib)}): {up_genes_ib}")
print(f"shared-DOWN ({len(down_genes_ib)}): {down_genes_ib}")

In [ ]:
# ============================================================
# Correlation of the log2FC between each pair of systems (Figure 5 a, text in
# each scatter): Pearson r with a permutation p value (N_PERM shuffles of one
# system's values), and Spearman rho.
# Empirical p = (#|null_r| ≥ |real_r| + 1) / (N_PERM + 1)
# ============================================================
N_PERM = 10_000
rng    = np.random.default_rng(0)
pairs  = [("HEK", "iPSC"), ("HEK", "brain"), ("iPSC", "brain")]

stat_rows = []
for a, b in pairs:
    x, y = df_cs[a].values, df_cs[b].values
    r_p, p_p = pearsonr(x, y)
    r_s, p_s = spearmanr(x, y)

    # Permutation null of Pearson r: y shuffled against x
    xc      = x - x.mean()
    denom_x = np.sqrt((xc ** 2).sum())
    null_r  = np.empty(N_PERM)
    for k in range(N_PERM):
        y_perm = rng.permutation(y)
        yc     = y_perm - y_perm.mean()
        null_r[k] = (xc * yc).sum() / (denom_x * np.sqrt((yc ** 2).sum()))
    emp_p = ((np.abs(null_r) >= abs(r_p)).sum() + 1) / (N_PERM + 1)

    stat_rows.append(dict(
        pair=f"{a} × {b}", n_genes=len(x),
        pearson_r=r_p,    pearson_p_param=p_p, pearson_p_perm=emp_p,
        spearman_rho=r_s, spearman_p_param=p_s,
    ))
    print(f"{a:>5} × {b:<5}  n={len(x):>4}  "
          f"r = {r_p:+.3f}  (param p={p_p:.2e}, perm p={emp_p:.4f})  |  "
          f"ρ = {r_s:+.3f}  (p={p_s:.2e})")

stats_df = pd.DataFrame(stat_rows)
stats_df.to_csv(f"{output_dir}/{today}_crosssystem_correlations.csv", index=False)
print(f"\nSaved → {output_dir}/{today}_crosssystem_correlations.csv")

In [ ]:
# ============================================================
# Figure 5 a: log2FC of one system against another for the three pairs, one
# scatter each (3 x 1 grid units). Statistics in the top-left corner of each
# scatter; the axis labels name the pair. One legend, on the iPSC x brain scatter.
# ============================================================
S_BG, S_HL  = native_pt(1.0) ** 2, native_pt(2.4) ** 2   # dot diameters 1.0 / 2.4 pt
ALPHA_BG    = 0.5          # opacity of the unclassified genes
GUIDE_COLOR = REF_COLOR    # one grey for every reference line; dash style and
                           # width tell them apart

group_colors = {"shared-UP": UP_COLOR,       "shared-DOWN": DOWN_COLOR}
group_labels = {"shared-UP": "Shared up", "shared-DOWN": "Shared down"}

LEGEND_PAIR = ("iPSC", "brain")   # the scatter that carries the legend

pair_order  = [("HEK", "iPSC"), ("HEK", "brain"), ("iPSC", "brain")]

fig, axm = create_figure(1, 3, figsize=figsize(3, 1))
for (a, b) in pair_order:
    sub_df = df_cs.dropna(subset=[a, b])
    x, y   = sub_df[a].values, sub_df[b].values

    ax = axm.next()
    ax.set_axisbelow(True)   # dots and reference lines above the grid

    bg = sub_df[sub_df["category"] == "none"]
    ax.scatter(bg[a], bg[b], s=S_BG, alpha=ALPHA_BG, color=BG_COLOR,
               linewidths=0, zorder=1)

    lim_lo = float(min(x.min(), y.min())) - 0.3
    lim_hi = float(max(x.max(), y.max())) + 0.3
    ax.plot([lim_lo, lim_hi], [lim_lo, lim_hi], ls=":", color=GUIDE_COLOR,
            lw=AXES_LW, zorder=2, label="y = x")
    # zorder 0.8: above the grid (0.5), which has a line at 0, and below the dots
    ax.axhline(0, color=GUIDE_COLOR, lw=THIN_LW, zorder=0.8)
    ax.axvline(0, color=GUIDE_COLOR, lw=THIN_LW, zorder=0.8)

    for grp, color in group_colors.items():
        sub = sub_df[sub_df["category"] == grp]
        ax.scatter(sub[a], sub[b], s=S_HL, color=color,
                   edgecolor="black", linewidth=MARKER_LW, zorder=5,
                   label=f"{group_labels[grp]}  (n = {len(sub)})")

    ax.set_xlim(lim_lo, lim_hi)
    ax.set_ylim(lim_lo, lim_hi)
    # No equal aspect: the y = x line marks equal values at any aspect ratio.

    # Statistics as grey text, top left: the quadrant of opposite signs, which
    # holds the fewest genes. The white box covers the x = 0 line behind the text.
    stat = next(s for s in stat_rows if s["pair"] == f"{a} × {b}")
    ax.text(
        0.02, 0.98,
        f"n = {stat['n_genes']}\n"
        f"r = {stat['pearson_r']:+.2f}  (perm p = {stat['pearson_p_perm']:.3g})\n"
        f"ρ = {stat['spearman_rho']:+.2f}",
        transform=ax.transAxes, ha="left", va="top",
        fontsize=ANNOT_FS, color=GUIDE_COLOR, linespacing=1.2,
        bbox=dict(boxstyle="square,pad=0.1", facecolor="white", edgecolor="none"),
    )

    label_axes(
        ax,
        xlabel=f"{a} log2 fold change",
        ylabel=f"{b} log2 fold change",
    )

    if (a, b) == LEGEND_PAIR:
        ax.legend(loc="lower right", frameon=False)

save_panel(fig, "a", "crosssystem_scatter", FIG5_DIR)
plt.show()
plt.close(fig)

# 6. Dot plots of the shared signature (Figure 5b, Figure S7c)

In [ ]:
# Gene x system matrices of the cross-system dot plots (Figure 5 b, Figure S7 a, c,
# d). cross_system_dotplot (src/dotplot.py) draws them: colour = log2FC, dot area
# = -log10 FDR, black outline = FDR < 0.05, grey cross = no row for the gene.
def build_panel_data(panel_dict, system_results):
    """
    panel_dict: {section: {group label: [genes]}}; a solid line separates the
                sections, a dashed line the groups inside a section
    system_results: {column name: (result_df, det_df)}
    Returns gene_order, groups, section_breaks, df_lfc, df_fdr; one value per gene
    from _gene_values (cell of Figure 5 a).
    """
    gene_order, groups, section_breaks = [], [], []
    for di, (_dir, subgroups) in enumerate(panel_dict.items()):
        if di > 0:
            section_breaks.append(len(gene_order))
        for lbl, genes in subgroups.items():
            groups.append((lbl, list(genes)))
            gene_order.extend(genes)
    _genes = list(dict.fromkeys(gene_order))
    df_lfc = pd.DataFrame({s: _gene_values(rdf, ddf, "log2fc", _genes)
                           for s, (rdf, ddf) in system_results.items()})
    df_fdr = pd.DataFrame({s: _gene_values(rdf, ddf, "fdr", _genes)
                           for s, (rdf, ddf) in system_results.items()})
    return gene_order, groups, section_breaks, df_lfc, df_fdr

## 6.1 Three systems (Figure S7c)

In [ ]:
# ============================================================
# Figure S7 c: dot plot of the three-system signature (HEK, iPSC, brain).
# UP: every three-system shared-UP gene (up_genes). DOWN: 20 genes chosen from
# the shared-DOWN list and grouped by function; most shared-DOWN genes are nuclear.
# ============================================================
panel_A_genes = {
    "UP": {
        "V-ATPase V0":   ["ATP6V0A1", "ATP6V0C", "ATP6V0D1", "WDR7"],
        "Syn. vesicle":  ["VAMP2", "SV2A"],
        "Novel":         ["GGT7"],          # not in earlier α-synuclein proteomes (literature section)
    },
    "DOWN": {
        "FTD / ALS RBPs":  ["TARDBP", "FUS", "MATR3", "TAF15"],
        "Spliceosome":     ["SF3B1", "SNRPE", "U2AF2", "PRPF8"],
        "hnRNP":           ["HNRNPA1", "HNRNPC", "HNRNPK"],
        "mRNA processing": ["SRSF1", "RBM39", "NUDT21"],
        "Chromatin":       ["H2AC11", "MACROH2A2", "TRIM28"],
        "Nucleolus":       ["NCL", "NPM1", "FBL"],
    },
}

system_results_3way = {
    "HEK":   (ttest_results_autoph[HEK_KEY],   det_autoph),
    "iPSC":  (ttest_results_ipsc[IPSC_KEY],    det_ipsc),
    "brain": (ebayes_results_brain[BRAIN_KEY], det_brain),
}
g_order, grps, s_breaks, df_lfc_a, df_fdr_a = build_panel_data(panel_A_genes, system_results_3way)

_up_drawn = {g for genes in panel_A_genes["UP"].values() for g in genes}
print(f"CHECK UP section = every three-system shared-UP gene: {_up_drawn == set(up_genes)} "
      f"({len(_up_drawn)} drawn, {len(up_genes)} shared-UP)")

cross_system_dotplot(
    df_lfc_a, df_fdr_a, ["HEK", "iPSC", "brain"], grps,
    "c", f"crosssystem_dotplot_3way_lfc{UNIV_LFC:g}".replace(".", "p"), SI7_DIR,
    size=(1, 2), section_breaks=s_breaks,
    group_layers=3,   # at 1 x 2 units a row is 2.5 mm high; group names in two columns overlap
)

## 6.2 iPSC and brain (Figure 5b)

In [ ]:
# ============================================================
# Figure 5 b: dot plot of the two-system signature (iPSC, brain), genes chosen by
# hand from the shared-UP and shared-DOWN lists of df_cs_ib and grouped by
# function. The DOWN section holds cytoplasmic genes only.
# ============================================================
panel_B_genes = {
    "UP": {
        "SNARE / release":    ["SNAP25", "STX1A", "STX1B", "STXBP1", "VAMP2"],
        "Synaptic vesicle":   ["SYT1", "SYT7", "SYP", "SV2A", "SYN1", "DNM1"],
        "V-ATPase V0":        ["ATP6V0A1", "ATP6V0C", "ATP6V0D1", "WDR7"],
        "PM transport":       ["ATP1A3", "ATP2B1", "SLC8A2", "SLC2A3"],
        "Adhesion":           ["CADM3", "NCAM1", "NCAM2"],
        "Novel": ["GGT7", "DAAM1", "FAM171A2"],
    },
    "DOWN_cytoplasmic_only": {
        "ER / UPR":           ["HSPA5", "HSP90B1", "CALR", "PDIA6", "ERP44"],
        "COPI / COPII":       ["COPA", "COPB1", "COPG1", "SEC23A"],
        "Proteasome 19S":     ["PSMC1", "PSMC5", "PSMD2"],
        "Translation / aaRS": ["AIMP1", "AIMP2", "EEF1B2", "IARS1"],
        "Cytosolic \nribosome": ["RPL10", "RPL24", "RPS19"],
    },
}

system_results_2way = {
    "iPSC":  (ttest_results_ipsc[IPSC_KEY],    det_ipsc),
    "brain": (ebayes_results_brain[BRAIN_KEY], det_brain),
}
g_order, grps, s_breaks, df_lfc_b, df_fdr_b = build_panel_data(panel_B_genes, system_results_2way)

cross_system_dotplot(
    df_lfc_b, df_fdr_b, ["iPSC", "brain"], grps,
    "b", f"crosssystem_dotplot_iPSCbrain_lfc{UNIV_LFC:g}".replace(".", "p"), FIG5_DIR,
    section_breaks=s_breaks,
)

# 7. Dense-core vesicle proteins (Figure S7b)

In [ ]:
# ============================================================
# Figure S7 b: dense-core vesicle (DCV) proteins in iPSC and brain.
#   Rows    = DCV cargo, DCV machinery and SNCA, grouped
#   Columns = 6 concepts x {iPSC, brain}: each concept is one comparison per
#             system, e.g. "Soma stress" = iPSC soma PFF vs PBS and brain DAB−
#             neurons disease vs control
#   Colour  = log2 fold change; SNCA is left out of the colour range, so the
#             range is set by the other genes
#   Text    = */**/*** for FDR < 0.05 / 0.01 / 0.001
# Drawn by concept_pair_heatmap (src/crosssystem_plots.py).
# ============================================================
dcv_proteins = {
    "cargo": {
        "granins":       ["CHGA", "CHGB", "SCG2", "SCG3", "SCG5"],
        "neuropeptides": ["VGF", "PCSK1N"],
        "processing":    ["CPE", "PAM", "PCSK1"],
    },
    "machinery": {
        "rab3":           ["RAB3A", "RAB3B", "RAB3C", "RAB3D"],
        "rab27":          ["RAB27A", "RAB27B"],
        "sv2":            ["SV2A", "SV2B", "SV2C"],
        "synaptotagmins": ["SYT1", "SYT4", "SYT11"],
        "priming":        ["CADPS", "CADPS2"],
        "vsnare":         ["VAMP4", "VAMP2"],
        "biogenesis":     ["HID1"],
    },
    "snca": {
        "α-syn":          ["SNCA"],
    },
}

gene_order = []
for _, subs in dcv_proteins.items():
    for sub_lbl, genes in subs.items():
        gene_order.extend(genes)

# --- Two brain contrasts used only here: neuropil vs DAB− neurons, in control
#     and in disease donors (eBayes on donor pseudobulk, as in section 3.3) ---
EXTRA = [
    dict(name="Neuropil_vs_NeuronDABneg_Control",
         group_a={"fraction": "Neuropil",                    "disease_binary": "Control"},
         group_b={"fraction": "Neuron | Without Aggregate",  "disease_binary": "Control"}),
    dict(name="Neuropil_vs_NeuronDABneg_Disease",
         group_a={"fraction": "Neuropil",                    "disease_binary": "Disease"},
         group_b={"fraction": "Neuron | Without Aggregate",  "disease_binary": "Disease"}),
]
extra_brain_results = {}
for c in EXTRA:
    ad_sub = adata_eb_brain.copy()
    pb, n_a, n_b = _pseudobulk_pair(ad_sub,
                                    _mask_for(ad_sub, c["group_a"]),
                                    _mask_for(ad_sub, c["group_b"]))
    if pb is None:
        print(f"{c['name']}: SKIPPED (donors A={n_a}, B={n_b})")
        continue
    _, res = at.tl.diff_exp_ebayes(pb, "__contrast", ("A", "B"))
    res = res.rename(columns={"logFC": "log2fc", "P.Value": "p_value",
                              "adj.P.Val": "fdr", "pvalue": "p_value",
                              "padj": "fdr", "adj_pvalue": "fdr"})
    if "protein" not in res.columns:
        res = res.reset_index().rename(columns={"index": "protein"})
    if "single_gene_name" not in res.columns:
        res["single_gene_name"] = res["protein"].map(_gene_map_brain)
    extra_brain_results[c["name"]] = res
    print(f"{c['name']}: A={n_a} B={n_b}, tested={len(res)}")

brain_all_results = {**ebayes_results_brain, **extra_brain_results}

# --- 6 concepts x (iPSC contrast, brain contrast) ---
concepts = [
    ("Baseline split",  ("PBS_Neurites_vs_Soma",       "Neuropil_vs_NeuronDABneg_Control")),
    ("Disease split",   ("PFF_Neurites_vs_Soma",       "Neuropil_vs_NeuronDABneg_Disease")),
    ("Soma stress",     ("Soma_PFF_vs_PBS",            "Neuron_DABneg_Disease_vs_Control")),
    ("Neurite stress",  ("Neurites_PFF_vs_PBS",        "Neuropil_Disease_vs_Control")),
    ("Neuritic agg.",   ("Neuritic_Agg_vs_Neurites",   "Elong_vs_Neuropil_Disease")),
    ("Somatic agg.",    ("Somatic_Agg_vs_Soma",        "Round_vs_NeuronDABpos_Disease")),
]
# Parenthetical under each concept header, drawn once over neighbouring concepts
# that share it
SUB_TITLES = {
    "Baseline split":  "(neurite/soma)",
    "Disease split":   "(neurite/soma)",
    "Soma stress":     "(disease/control)",
    "Neurite stress":  "(disease/control)",
    "Neuritic agg.":   "(agg/host)",
    "Somatic agg.":    "(agg/host)",
}

# --- log2FC and FDR per gene and contrast ---
lfc_data, fdr_data, col_order, col_short = {}, {}, [], []
for c_name, (ip_key, br_key) in concepts:
    col_ip, col_br = f"{c_name}|iPSC", f"{c_name}|brain"
    col_order.extend([col_ip, col_br]);  col_short.extend(["iPSC", "brain"])

    if ip_key in ttest_results_ipsc:
        lfc_data[col_ip] = _gene_values(ttest_results_ipsc[ip_key], det_ipsc, "log2fc", gene_order)
        fdr_data[col_ip] = _gene_values(ttest_results_ipsc[ip_key], det_ipsc, "fdr", gene_order)
    else:
        print(f"Missing iPSC contrast: {ip_key}")
        lfc_data[col_ip] = pd.Series(np.nan, index=gene_order)
        fdr_data[col_ip] = pd.Series(np.nan, index=gene_order)

    if br_key in brain_all_results:
        lfc_data[col_br] = _gene_values(brain_all_results[br_key], det_brain, "log2fc", gene_order)
        fdr_data[col_br] = _gene_values(brain_all_results[br_key], det_brain, "fdr", gene_order)
    else:
        print(f"Missing brain contrast: {br_key}")
        lfc_data[col_br] = pd.Series(np.nan, index=gene_order)
        fdr_data[col_br] = pd.Series(np.nan, index=gene_order)

lfc_matrix = pd.DataFrame(lfc_data).loc[gene_order, col_order]
fdr_matrix = pd.DataFrame(fdr_data).loc[gene_order, col_order]

def _sig_label(fdr):
    if not np.isfinite(fdr): return ""
    if fdr < 0.001: return "***"
    if fdr < 0.01:  return "**"
    if fdr < 0.05:  return "*"
    return ""
# DataFrame.applymap is deprecated; per-column map works on every pandas version
annot_matrix = fdr_matrix.apply(lambda col: col.map(_sig_label))

# --- Colour range from every gene except SNCA; SNCA cells beyond it saturate ---
SNCA_EXCLUDE  = ["SNCA"]
VMAX_SHRINK   = 0.7   # the colour saturates at 70 % of the largest |log2FC|
non_snca_mask = ~lfc_matrix.index.isin(SNCA_EXCLUDE)
vmax = np.nanmax(np.abs(lfc_matrix.loc[non_snca_mask].values)) * VMAX_SHRINK
norm = TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)

# Row groups: cargo and machinery are labelled, SNCA is not.
concept_pair_heatmap(
    lfc_matrix, annot_matrix, [c[0] for c in concepts], SUB_TITLES, col_short,
    [(m, [g for gs in subs.values() for g in gs]) for m, subs in dcv_proteins.items()],
    {"cargo", "machinery"}, "b", "DCV_crosssystem_heatmap", SI7_DIR,
    size=(2, 2), norm=norm,
)

lfc_matrix.to_csv(f"{output_dir}/{today}_DCV_log2FC.csv")
fdr_matrix.to_csv(f"{output_dir}/{today}_DCV_FDR.csv")

# 8. FTD/ALS RNA-binding proteins (Figure S7d)

In [ ]:
# ============================================================
# Figure S7 d: FTD/ALS RNA-binding proteins in the five aggregate-vs-host
# contrasts.
#   Columns: HEK EA, iPSC SAgg, iPSC NAgg, brain Round/Neuron DAB+,
#            brain Elong/Neuropil.
#   Check before the plot: per-donor sign consistency of the brain contrasts.
#   The check passes if >= 70 % of the genes have the same sign in at least 4
#   of 5 paired donors.
# ============================================================
panel_D_genes = {
    "": [
        "TARDBP", "FUS", "HNRNPA1", "HNRNPA2B1",
        "MATR3", "EWSR1", "TAF15",
    ],
}
ftd_genes = panel_D_genes[""]

# --- contrast specs: (col_label, result_df, det_df) ---
contrast_specs_D = [
    ("HEK EA",       ttest_results_autoph["PureAggs_vs_RestCells_minusBafA1"], det_autoph),
    ("iPSC SAgg",    ttest_results_ipsc["Somatic_Agg_vs_Soma"],                det_ipsc),
    ("iPSC NAgg",    ttest_results_ipsc["Neuritic_Agg_vs_Neurites"],           det_ipsc),
    ("brain Round",  ebayes_results_brain["Round_vs_NeuronDABpos_Disease"],    det_brain),
    ("brain Elong",  ebayes_results_brain["Elong_vs_Neuropil_Disease"],        det_brain),
]

col_labels_D = [c[0] for c in contrast_specs_D]

# ============================================================
# Check: per-donor log2FC sign consistency of the brain contrasts.
# Per gene (best-detected protein), the log2FC of every donor with samples in
# both groups, and how many of these donors agree in sign.
# ============================================================
def _per_donor_log2fc_genes(adata, mask_a, mask_b, genes, det_df,
                             donor_col=DONOR_COL_BRAIN):
    obs = adata.obs
    donors_a = set(obs.loc[mask_a, donor_col].astype(str).unique())
    donors_b = set(obs.loc[mask_b, donor_col].astype(str).unique())
    paired   = sorted(donors_a & donors_b)
    best_protein = det_df["protein"].to_dict()

    results = {}
    for gene in genes:
        pid = best_protein.get(gene)
        if pid is None or pid not in adata.var.index:
            results[gene] = pd.Series(np.nan, index=paired);  continue
        col_idx = adata.var.index.get_loc(pid)
        x = adata.X[:, col_idx]
        donor_lfcs = {}
        for d in paired:
            sel_a = mask_a & (obs[donor_col].astype(str).values == d)
            sel_b = mask_b & (obs[donor_col].astype(str).values == d)
            if sel_a.sum() == 0 or sel_b.sum() == 0:
                donor_lfcs[d] = np.nan;  continue
            donor_lfcs[d] = float(np.nanmean(x[sel_a]) - np.nanmean(x[sel_b]))
        results[gene] = pd.Series(donor_lfcs)
    return pd.DataFrame(results).T, paired   # genes × donors

def _sign_consistency_report(per_donor_df, paired, label):
    print(f"\n=== {label} — paired donors ({len(paired)}): {paired} ===")
    rows = []
    for gene in per_donor_df.index:
        vals = per_donor_df.loc[gene].dropna()
        n_total = len(vals)
        if n_total == 0:
            print(f"  {gene:<10}  no data");  rows.append((gene, 0, 0));  continue
        signs = np.sign(vals.values)
        n_max = int(max((signs > 0).sum(), (signs < 0).sum()))
        majority = "+" if (signs > 0).sum() >= (signs < 0).sum() else "-"
        per_donor_str = "  ".join(f"{d}: {v:+.2f}" for d, v in vals.items())
        print(f"  {gene:<10} {n_max}/{n_total} same sign (majority {majority})   |  {per_donor_str}")
        rows.append((gene, n_max, n_total))

    # A gene passes with >= 4 donors and at most one donor of the other sign
    # (4 of 5); with more than 5 donors, n_total - 1 must agree.
    def _passes(n_max, n_total):
        return n_total >= 4 and n_max >= max(4, n_total - 1)
    n_pass = sum(1 for g, nmax, ntot in rows if _passes(nmax, ntot))
    n_total_genes = len(rows)
    pct = n_pass / n_total_genes if n_total_genes else 0
    verdict = "✅ pass" if pct >= 0.70 else "❌ FAIL (< 70%) — panel does not merit main-figure space"
    print(f"  → {n_pass}/{n_total_genes} genes pass ≥{max(4, len(paired)-1)}/{len(paired)} consistency "
          f"({pct:.0%}) — {verdict}")

for label, group_a, group_b in [
    ("brain Round vs NeuronDAB+", {"fraction": "Round Aggregate",        "disease_binary": "Disease"},
                                   {"fraction": "Neuron | DAB Positive", "disease_binary": "Disease"}),
    ("brain Elong vs Neuropil",   {"fraction": "Elongated Aggregate",    "disease_binary": "Disease"},
                                   {"fraction": "Neuropil",              "disease_binary": "Disease"}),
]:
    mA = _mask_for(adata_eb_brain, group_a)
    mB = _mask_for(adata_eb_brain, group_b)
    per_donor_df, paired = _per_donor_log2fc_genes(adata_eb_brain, mA, mB, ftd_genes, det_brain)
    _sign_consistency_report(per_donor_df, paired, label)

# ============================================================
# log2FC and FDR matrices, and the plot
# ============================================================
lfc_data_D = {col: _gene_values(rdf, ddf, "log2fc", ftd_genes)
              for col, rdf, ddf in contrast_specs_D}
fdr_data_D = {col: _gene_values(rdf, ddf, "fdr", ftd_genes)
              for col, rdf, ddf in contrast_specs_D}
df_lfc_D = pd.DataFrame(lfc_data_D, index=ftd_genes)[col_labels_D]
df_fdr_D = pd.DataFrame(fdr_data_D, index=ftd_genes)[col_labels_D]

cross_system_dotplot(
    df_lfc_D, df_fdr_D, col_labels_D, [("", ftd_genes)],
    "d", "crosssystem_dotplot_FTD_ALS_RBPs", SI7_DIR, size=(1, 1),
    legend_corner=True,   # the dot-size key in the corner under the gene names
)

In [ ]:
# ============================================================
# Control for Figure S7 d: does the log2FC of the seven FTD/ALS RBPs differ from
# that of other nuclear proteins in the same contrast, with and without matching
# the background on abundance?
#
#   All seven genes are nuclear. If an aggregate fraction contains less nucleus
#   than its host fraction, every nuclear protein has a negative log2FC, so the
#   panel alone does not show that these seven differ from the rest of the
#   nucleus. Each gene's log2FC is therefore compared with that of the other
#   nuclear proteins of the same contrast.
#
#   Two backgrounds, both reported:
#     unmatched  every detected nuclear gene;
#     matched    the nuclear genes within ABUND_CAL log2 intensity of the panel
#                gene, with the local trend removed: the background log2FC is
#                regressed on abundance inside the window and the fit is read at
#                the gene's own abundance. The seven genes are abundant; if the
#                depletion depends on abundance, the unmatched background is not
#                comparable. A window median does not remove the trend: the
#                background has more genes below an abundant gene than above it,
#                so the median of the window lies at a lower abundance than the
#                gene. The matched values are residuals around the fit.
#
#   Nuclear set: GO:0005634 and its part_of descendants (the closure rule of the
#   organelle modules), restricted to the genes detected in that system, the
#   seven panel genes removed.
#
#   Per gene: log2FC minus the median of the nuclear background, and log2FC
#   minus the local fit. Each has a two-sided empirical p (the rank of the
#   deviation among the background deviations of the same kind), BH-corrected
#   over the seven genes of one contrast.
#
#   Per set: Mann-Whitney test against the unmatched background, and a matched
#   permutation: ABUND_B draws, each taking one residual from each panel gene's
#   window, so the null has the abundance profile of the panel; the statistic is
#   the median residual. With seven genes the per-gene FDR has little power; the
#   set-level test is the test of the compartment question.
#
#   This is a competitive test, not a covariate adjustment. A covariate
#   adjustment would fit log2 intensity ~ group + nuclear score per sample, which
#   needs a design matrix that at.tl.diff_exp_ebayes does not accept.
# ============================================================
NUC_ROOT   = "GO:0005634"   # nucleus
NUC_MIN_BG = 200            # minimum number of nuclear background genes per contrast
ABUND_K    = 300            # at most this many nuclear genes per window
ABUND_CAL  = 1.0            # a window reaches at most this far in log2 intensity.
                            # Without this limit a panel gene more abundant than
                            # the whole background would get a window from the top
                            # of the distribution, and the abundance trend that the
                            # window does not match would appear as a panel effect.
ABUND_MINP = 50             # with fewer neighbours inside ABUND_CAL the gene gets
                            # no matched background and no matched p value
ABUND_B    = 10_000         # draws of the matched permutation
ABUND_SEED = 0              # fixed seed: reproducible p values

# Abundance matrix of each system, found by the detection table of the contrast.
# These are the log2 matrices, as in the complex cells: adata_sf_* is linear, and
# the local fit uses abundance as a numeric covariate, so the scale changes the fit.
_ABUND_SRC = [(det_autoph, adata_log_autoph), (det_ipsc, adata_log_ipsc),
              (det_brain, adata_log_brain)]
ABUND_MIN_COVER = 0.5   # if fewer of the detection table's genes have a protein in
                        # the abundance matrix, the two come from different
                        # quantifications


def _abundance(det_df):
    """Mean log2 intensity per gene, from that system's log matrix."""
    _ad = next((a for d, a in _ABUND_SRC if d is det_df), None)
    if _ad is None:
        raise KeyError("this contrast's detection table is not in _ABUND_SRC; "
                       "add its matrix there")
    _X = np.asarray(_ad.X, dtype=float)
    if np.nanmax(_X) >= 100:      # linear scale: take log2
        with np.errstate(invalid="ignore", divide="ignore"):
            _X = np.log2(np.where(_X > 0, _X, np.nan))
        print(f"  abundance matrix was on a linear scale, taking log2 "
              f"(max now {np.nanmax(_X):.1f})")
    with np.errstate(invalid="ignore"):
        _m = pd.Series(np.nanmean(_X, axis=0), index=_ad.var.index)
    _out = det_df["protein"].map(_m).dropna()
    _cover = len(_out) / max(1, len(det_df))
    assert _cover >= ABUND_MIN_COVER, (
        f"only {_cover:.0%} of the detection table's genes have a protein in "
        f"the abundance matrix; the two are not the same quantification")
    print(f"  abundance for {len(_out)} of {len(det_df)} genes ({_cover:.0%})")
    return _out                                    # gene -> mean log2 intensity


# Nuclear gene set: part_of descendants only; the is_a closure would add subtypes
# of the nucleus (see the organelle-module cell).
_nuc_terms = _closure(NUC_ROOT, _part_kids)
NUC_GENES = {g.upper() for t in _nuc_terms for g in _term2genes.get(t, ())}
print(f"nuclear background: {len(_nuc_terms)} terms under {NUC_ROOT}, "
      f"{len(NUC_GENES)} annotated genes ({ORG_VERSION_NOTE})")

_panel = [g.upper() for g in ftd_genes]
_missing_ann = [g for g in _panel if g not in NUC_GENES]
assert not _missing_ann, (
    f"{_missing_ann} are not annotated under {NUC_ROOT}; the premise of this "
    f"control is that every panel gene is nuclear")


def _emp_p(values, centre, observed):
    """Two-sided rank of `observed` inside `values`, both read from `centre`."""
    dev = np.abs(np.asarray(values) - centre)
    return (1 + int((dev >= abs(observed - centre)).sum())) / (1 + len(dev))


_rng = np.random.default_rng(ABUND_SEED)
rows, summary = [], []
for _label, _res, _det in contrast_specs_D:
    _lfc = _gene_values(_res, _det, "log2fc").dropna()   # every gene of the contrast
    _abu = _abundance(_det)
    _detected = set(_det[_det["frac"] >= MIN_DETECTION_FRAC].index)
    _usable = _detected & set(_lfc.index) & set(_abu.index)
    _bg_genes = sorted((NUC_GENES & _usable) - set(_panel))
    _bg, _bg_a = _lfc[_bg_genes], _abu[_bg_genes]
    assert len(_bg) >= NUC_MIN_BG, (
        f"{_label}: only {len(_bg)} nuclear background genes, fewer than "
        f"{NUC_MIN_BG}; this contrast cannot support the control")
    _k = min(ABUND_K, len(_bg))

    # background sorted by abundance once; each panel gene's window is the _k
    # entries nearest its own abundance
    _ord = np.argsort(_bg_a.to_numpy())
    _a_sorted = _bg_a.to_numpy()[_ord]
    _l_sorted = _bg.to_numpy()[_ord]

    _med = float(_bg.median())
    # Shift of the nucleus: the nuclear median against the median of the
    # non-nuclear genes. The reference leaves out the nucleus, which is a large
    # share of the detected genes.
    _rest = _lfc[sorted(_usable - NUC_GENES)]
    _all_med = float(_rest.median())

    _p_un, _p_mt, _pools, _obs = [], [], [], []
    for _g in _panel:
        _v = float(_lfc.get(_g, np.nan))
        _ga = float(_abu.get(_g, np.nan))
        if not (np.isfinite(_v) and np.isfinite(_ga)):
            _p_un.append(np.nan);  _p_mt.append(np.nan);  _pools.append(None)
            _obs.append(np.nan)
            rows.append(dict(contrast=_label, gene=_g, abundance=_ga,
                             abund_pct=np.nan, lfc=np.nan, nuc_median=_med,
                             corrected=np.nan, p_emp=np.nan,
                             matched_median=np.nan, corrected_matched=np.nan,
                             p_matched=np.nan))
            continue
        # window: the _k nearest neighbours within ABUND_CAL
        _near = np.where(np.abs(_a_sorted - _ga) <= ABUND_CAL)[0]
        _obs.append(_v)
        _p_un.append(_emp_p(_bg.to_numpy(), _med, _v))
        _row = dict(contrast=_label, gene=_g, abundance=_ga,
                    abund_pct=float((_bg_a.to_numpy() < _ga).mean()),
                    lfc=_v, nuc_median=_med, corrected=_v - _med,
                    p_emp=_p_un[-1], n_window=len(_near))
        if len(_near) < ABUND_MINP:
            # fewer than ABUND_MINP nuclear genes at this abundance: no matched value
            _pools.append(None);  _p_mt.append(np.nan)
            _row.update(matched_ref=np.nan, corrected_matched=np.nan,
                        abund_gap=np.nan, local_slope=np.nan, p_matched=np.nan)
            rows.append(_row)
            continue
        _sel = _near[np.argsort(np.abs(_a_sorted[_near] - _ga))[:_k]]
        _pool, _pool_a = _l_sorted[_sel], _a_sorted[_sel]
        # linear fit of the background log2FC on abundance inside the window,
        # read at this gene's abundance: the expected log2FC of a nuclear protein
        # of this abundance
        _slope, _icept = np.polyfit(_pool_a, _pool, 1)
        _pred = _icept + _slope * _ga
        _resid = _pool - (_icept + _slope * _pool_a)
        _pools.append(_resid)
        _p_mt.append(_emp_p(_resid, float(np.median(_resid)), _v - _pred))
        _row.update(matched_ref=float(_pred), corrected_matched=_v - _pred,
                    abund_gap=float(np.median(_pool_a) - _ga),
                    local_slope=float(_slope), p_matched=_p_mt[-1],
                    n_window=len(_sel))
        rows.append(_row)
    _fdr_un, _fdr_mt = nan_safe_bh_correction(_p_un), nan_safe_bh_correction(_p_mt)
    for _i in range(len(_panel)):
        rows[-len(_panel) + _i]["fdr_emp"] = _fdr_un[_i]
        rows[-len(_panel) + _i]["fdr_matched"] = _fdr_mt[_i]

    # --- set level -------------------------------------------------------
    _obs = np.array(_obs, dtype=float)
    _ok = np.isfinite(_obs)
    _panel_med = float(np.nanmedian(_obs))
    _u, _p_mw = (mannwhitneyu(_obs[_ok], _bg.to_numpy(), alternative="two-sided")
                 if _ok.sum() >= 3 else (np.nan, np.nan))
    # matched permutation: one draw takes a random background gene from each
    # panel gene's window, so the null set has the panel's abundance profile
    _use = [p for p in _pools if p is not None]
    _matchable = [g for g, p in zip(_panel, _pools) if p is not None]
    if _use:
        # each pool holds the residuals of its window around the local fit, so a
        # draw is one residual per matchable panel gene; the statistic is their
        # median, compared with the panel's own median residual
        _draw = np.stack([p[_rng.integers(0, len(p), ABUND_B)] for p in _use])
        _null = np.median(_draw, axis=0)
        _null_med = float(np.median(_null))
        _res_obs = [r["corrected_matched"] for r in rows[-len(_panel):]
                    if np.isfinite(r.get("corrected_matched", np.nan))]
        _panel_med_m = float(np.median(_res_obs))
        _p_perm = _emp_p(_null, _null_med, _panel_med_m)
    else:
        _null_med, _panel_med_m, _p_perm = np.nan, np.nan, np.nan

    summary.append(dict(
        contrast=_label, n_background=len(_bg), n_matched=_k,
        panel_abund_pct=float(np.nanmean([r["abund_pct"] for r in rows[-len(_panel):]])),
        nonnuclear_median=_all_med, nuclear_median=_med,
        nuclear_shift=_med - _all_med, panel_median=_panel_med,
        panel_vs_nucleus=_panel_med - _med, p_set_unmatched=float(_p_mw),
        n_matchable=len(_use), unmatchable=",".join(
            g for g, p in zip(_panel, _pools) if p is None),
        matched_null_median=_null_med, panel_vs_matched=_panel_med_m,
        p_set_matched=_p_perm,
        local_slope=float(np.nanmean([r.get("local_slope", np.nan)
                                      for r in rows[-len(_panel):]])),
        n_fdr05_unmatched=int(np.nansum(np.asarray(_fdr_un) < 0.05)),
        n_fdr05_matched=int(np.nansum(np.asarray(_fdr_mt) < 0.05))))

rbp_nuclear = pd.DataFrame(rows)
rbp_nuclear_summary = pd.DataFrame(summary)

# --- report: per contrast, then per gene ----------------------------------
print("\nPer contrast. nuclear_shift is the nucleus against the rest of the "
      "proteome; panel_vs_nucleus uses the whole compartment as reference, "
      "panel_vs_matched only nuclear genes of the panel's own abundance.")
print(rbp_nuclear_summary[
    ["contrast", "n_background", "panel_abund_pct", "nuclear_shift",
     "local_slope", "panel_median", "panel_vs_nucleus", "p_set_unmatched",
     "n_matchable", "panel_vs_matched", "p_set_matched", "n_fdr05_matched"]]
    .to_string(index=False, float_format=lambda v: f"{v:.3g}"))

print("\nPer gene. corrected removes the compartment median; "
      "corrected_matched removes the local fit of background log2FC on "
      "abundance, read at that gene's abundance. local_slope is that fit's "
      "slope, in log2 per log2. Both FDRs are BH over the seven of one "
      "contrast.")
for _label in [c[0] for c in contrast_specs_D]:
    _t = rbp_nuclear[rbp_nuclear.contrast == _label]
    print(f"\n  {_label}  (nuclear median {_t.nuc_median.iloc[0]:+.2f})")
    print(_t[["gene", "abundance", "abund_pct", "lfc", "corrected", "fdr_emp",
              "n_window", "local_slope", "corrected_matched", "fdr_matched"]]
          .to_string(index=False, float_format=lambda v: f"{v:.3g}"))

# --- CHECKS --------------------------------------------------------------
# every window's median abundance lies within ABUND_CAL of its gene
_gap = rbp_nuclear.abund_gap.dropna().abs()
assert _gap.empty or _gap.max() <= ABUND_CAL, \
    f"a window's median abundance is {_gap.max():.2f} from its gene, beyond " \
    f"the caliper; the window is not matched"
print(f"\nCHECK matched windows sit within {_gap.max() if not _gap.empty else 0:.2f} "
      f"log2 of their gene (caliper {ABUND_CAL})")
_un = rbp_nuclear_summary[rbp_nuclear_summary.n_matchable < len(_panel)]
for _r in _un.itertuples():
    print(f"  NOTE {_r.contrast}: no matched background for {_r.unmatchable} "
          f"(fewer than {ABUND_MINP} nuclear genes within {ABUND_CAL} log2). "
          f"The matched result rests on {_r.n_matchable} of {len(_panel)} genes.")
print(f"CHECK panel abundance sits at the "
      f"{100 * rbp_nuclear.abund_pct.min():.0f} to "
      f"{100 * rbp_nuclear.abund_pct.max():.0f} percentile of the nuclear "
      f"background across contrasts, mean "
      f"{100 * rbp_nuclear.abund_pct.mean():.0f}; that spread is the reason "
      f"the unmatched background is not sufficient on its own")
# subtracting one median per contrast keeps the order of the genes
for _label in [c[0] for c in contrast_specs_D]:
    _t = rbp_nuclear[rbp_nuclear.contrast == _label].dropna(subset=["lfc"])
    assert (_t.lfc.rank().to_numpy() == _t.corrected.rank().to_numpy()).all(), \
        f"{_label}: the compartment correction reordered the genes"
print(f"CHECK the compartment correction is one shift per contrast; the "
      f"matched correction is per gene and can reorder the genes")

# --- set-level result, as two lists ----------------------------------------
_sig = rbp_nuclear_summary[rbp_nuclear_summary.p_set_matched < 0.05]
_ns = rbp_nuclear_summary[rbp_nuclear_summary.p_set_matched >= 0.05]
print(f"\nSET LEVEL, against an abundance-matched nuclear background "
      f"({ABUND_B} draws, windows of {ABUND_K} genes):")
print(f"  DIFFERENT from comparable nuclear proteins (p < 0.05): "
      + ("; ".join(f"{r.contrast} ({r.panel_vs_matched:+.2f} log2, "
                   f"p {r.p_set_matched:.2g})" for r in _sig.itertuples())
         if len(_sig) else "none"))
print(f"  NOT distinguishable (p >= 0.05): "
      + ("; ".join(f"{r.contrast} ({r.panel_vs_matched:+.2f} log2, "
                   f"p {r.p_set_matched:.2g})" for r in _ns.itertuples())
         if len(_ns) else "none"))
_flip = rbp_nuclear_summary[(rbp_nuclear_summary.p_set_unmatched < 0.05)
                            & (rbp_nuclear_summary.p_set_matched >= 0.05)]
if len(_flip):
    print(f"  abundance matching removed the set-level claim in: "
          f"{', '.join(_flip.contrast)}")

rbp_nuclear.to_csv(f"{output_dir}/{today}_FTD_RBP_nuclear_control.csv",
                   index=False)
rbp_nuclear_summary.to_csv(
    f"{output_dir}/{today}_FTD_RBP_nuclear_control_summary.csv", index=False)
print(f"written: {today}_FTD_RBP_nuclear_control.csv and its summary")

# 9. V-ATPase subunits (Figure S7a)

In [ ]:
# ============================================================
# Figure S7 a: every V-ATPase subunit in the five contrasts of Figure S7 d (HEK
# EA, iPSC SAgg, iPSC NAgg, brain Round, brain Elong).
#
#   The three-system shared-UP set contains ATP6V0A1, ATP6V0C, ATP6V0D1 and
#   WDR7: three subunits of the V0 sector and one mRAVE subunit. The panel shows
#   every subunit, so it shows whether the whole pump or only the membrane
#   sector is enriched in the aggregates.
#
#   The groups follow the structure of the complex. The ubiquitous subunits are
#   grouped by sector; the tissue-restricted paralogues (kidney a4, the
#   osteoclast a3 TCIRG1, testis and other isoforms) form their own block below
#   a solid line, because their absence from neuronal or HEK samples is expected.
#
#   A gene with no value in any contrast is not drawn; the cell prints its name.
# ============================================================

# Short group names in the panel; the long forms are for the figure legend:
# V0 membrane sector, V0 accessory subunits, V1 catalytic head, mRAVE /
# rabconnectin-3.
VATPASE_GENES = {
    "core": {
        "V0 sector": ["ATP6V0A1", "ATP6V0A2", "ATP6V0C", "ATP6V0D1"],
        "Accessory": ["ATP6AP1", "ATP6AP2"],
        "V1 head":   ["ATP6V1A", "ATP6V1B2", "ATP6V1C1", "ATP6V1D",
                      "ATP6V1E1", "ATP6V1F", "ATP6V1G1", "ATP6V1G2",
                      "ATP6V1H"],
        "mRAVE":     ["DMXL1", "DMXL2", "WDR7", "ROGDI"],
    },
    "other": {
        "Other subunits and\ntissue-restricted isoforms":
            ["ATP6V0B", "ATP6V0E1", "ATP6V0E2", "ATP6V0A4", "TCIRG1",
             "ATP6V0D2", "ATP6V1B1", "ATP6V1C2", "ATP6V1E2", "ATP6V1G3"],
    },
}
V_SIZE = (1, 2)     # grid units; the rows do not fit into one unit

_all = [g for sect in VATPASE_GENES.values() for grp in sect.values() for g in grp]
assert len(_all) == len(set(_all)), \
    f"a subunit is listed twice: {sorted({g for g in _all if _all.count(g) > 1})}"
print(f"{len(_all)} subunits listed: "
      + ", ".join(f"{k} {len(v)}" for sect in VATPASE_GENES.values()
                  for k, v in sect.items()).replace("\n", " "))

# --- number of contrasts with a value, per subunit -------------------------
_cov = pd.DataFrame(
    {col: _gene_values(rdf, ddf, "log2fc", _all)
     for col, rdf, ddf in contrast_specs_D})[col_labels_D]
_n_seen = _cov.notna().sum(axis=1)
_empty = sorted(_n_seen[_n_seen == 0].index)
print(f"\nquantified in how many of the {len(col_labels_D)} contrasts:")
print(_n_seen.sort_values(ascending=False).to_string())
if _empty:
    print(f"\ndropped, no value in any contrast ({len(_empty)}): "
          f"{', '.join(_empty)}")

V_PANEL = {sect_name: {lbl: [g for g in genes if g not in _empty]
                       for lbl, genes in sect.items()
                       if any(g not in _empty for g in genes)}
           for sect_name, sect in VATPASE_GENES.items()}
V_PANEL = {k: v for k, v in V_PANEL.items() if v}

system_results_V = {lbl: (rdf, ddf) for lbl, rdf, ddf in contrast_specs_D}
g_order_v, grps_v, breaks_v, df_lfc_v, df_fdr_v = build_panel_data(
    V_PANEL, system_results_V)

# --- CHECKS before the figure -------------------------------------------
assert list(df_lfc_v.columns) == col_labels_D, \
    f"column order changed: {list(df_lfc_v.columns)} against {col_labels_D}"
assert len(g_order_v) == len(_all) - len(_empty), \
    "the drawn gene list is not the listed one minus the unmeasured"
print(f"\nCHECK {len(g_order_v)} rows over {len(grps_v)} groups, "
      f"{len(breaks_v)} solid separator(s), columns {col_labels_D}")

_sig = (df_fdr_v < 0.05) & df_lfc_v.notna()
print(f"CHECK {int(_sig.to_numpy().sum())} of "
      f"{int(df_lfc_v.notna().to_numpy().sum())} drawn dots are at FDR < 0.05; "
      f"per contrast: { {c: int(_sig[c].sum()) for c in col_labels_D} }")

print("\nlog2FC per subunit (blank = not quantified in that contrast):")
print(df_lfc_v.loc[g_order_v].round(2).to_string())

cross_system_dotplot(
    df_lfc_v, df_fdr_v, col_labels_D, grps_v,
    "a", "crosssystem_dotplot_VATPase", SI7_DIR, size=V_SIZE,
    section_breaks=breaks_v,
    legend_corner=True,   # the dot-size key in the corner under the gene names
)

# 10. Gene set enrichment (Figure 5c)

Gene set enrichment analysis with gseapy prerank on one aggregate-versus-host contrast per system, against Reactome and GO biological process from MSigDB (cached in `src/gsea_genesets_cache.parquet`). The computation is in `src/gsea_pipeline.py`.

- **[1/3]** runs the enrichment.
- **[2/3]** clusters redundant terms by leading-edge overlap and writes the decision file `review.csv` of each contrast.
- **[3/3]** draws Figure 5c from the decisions in the three `review.csv` files.

The decisions in `review.csv` (representative term, exclusion and display label of each high-priority cluster) were made by the authors with the help of an AI review (Claude, Anthropic) that followed the rules in `GSEA_REVIEW_PROMPT.md`; they are an input of this analysis. A re-run of [2/3] carries every decision over by term name.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# GSEA [1/3]: the three aggregate-vs-host contrasts of Figure 5 c, one per system.
# Ranking: Welch t (HEK, iPSC) or the moderated t of the donor-level eBayes
# (brain). Null: gene permutation (prerank). The phenotype permutation needs the
# expression matrix and the sample classes, which the brain contrast does not
# provide, so no contrast uses it and the three columns share one null.
# ═══════════════════════════════════════════════════════════════════════
# --- Gene sets of cell types or tissues absent from a system are excluded per
# --- system, so artifact_pathways is a field of each system's config. A term
# --- impossible in a HEK monolayer can be correct in brain tissue, which has
# --- vessels and microglia.
_ABSENT_EVERYWHERE = {
    # Embryonic tissue programmes: no system is a gastrulating embryo
    "REACTOME_FORMATION_OF_PARAXIAL_MESODERM",
    "REACTOME_GASTRULATION",
    "REACTOME_SOMITOGENESIS",
    # Melanocytes: absent cell type
    "REACTOME_MITF_M_REGULATED_MELANOCYTE_DEVELOPMENT",
    "REACTOME_TRANSCRIPTIONAL_AND_POST_TRANSLATIONAL_REGULATION_OF_MITF_M_EXPRESSION_AND_ACTIVITY",
    # Cochlear hair cells: absent from cortex, nucleus accumbens and both cultures
    "REACTOME_SENSORY_PROCESSING_OF_SOUND",
}
_NO_VESSELS_NO_IMMUNE = {
    # Vascular / blood: no endothelium, no vessels, no plasma in a culture
    "GOBP_ENDOTHELIAL_CELL_MIGRATION",
    "GOBP_REGULATION_OF_BLOOD_CIRCULATION",
    "GOBP_REGULATION_OF_SYSTEMIC_ARTERIAL_BLOOD_PRESSURE",
    "GOBP_VASCULAR_PROCESS_IN_CIRCULATORY_SYSTEM",
    "REACTOME_CELL_SURFACE_INTERACTIONS_AT_THE_VASCULAR_WALL",
    "GOCC_BLOOD_MICROPARTICLE",
    # Immune: the named effector cell types are absent
    "GOBP_GRANULOCYTE_MIGRATION",
    "GOBP_POSITIVE_REGULATION_OF_IMMUNE_EFFECTOR_PROCESS",
    "GOBP_T_CELL_MEDIATED_IMMUNITY",
}
ARTIFACTS_BY_SYSTEM = {
    # HEK293 monolayer: no vessels, no immune cells, no synapses
    "HEK":   frozenset(_ABSENT_EVERYWHERE | _NO_VESSELS_NO_IMMUNE),
    # iPSC-derived dopaminergic neurons: synapses are present, vessels and
    # immune cells are not
    "iPSC":  frozenset(_ABSENT_EVERYWHERE | _NO_VESSELS_NO_IMMUNE),
    # Post-mortem brain: vessels and microglia are in the tissue, so vascular
    # and immune terms are kept
    "Brain": frozenset(_ABSENT_EVERYWHERE),
}

cfg_base = GSEAConfig(
    output_root=gsea_dir,
    n_review_clusters=40,
    run_phenotype=False,
    sentinel_action="drop",     # terms with a constant permutation null are
                                # dropped; their number belongs in the legend
    # Font sizes of the tick labels, axis labels and the two legend keys of
    # plot_combined, from src/figure_style.py
    tick_fontsize=TICK_FS,
    label_fontsize=LABEL_FS,
    title_fontsize=TITLE_FS,
    cmap_colors=(DOWN_COLOR, "white", UP_COLOR),
)
CFG = {sys_name: replace(cfg_base, artifact_pathways=arts)
       for sys_name, arts in ARTIFACTS_BY_SYSTEM.items()}

gene_sets = load_gene_sets(cfg_base)     # cached in src/gsea_genesets_cache.parquet

# --- Contrasts. kind "adata": the Welch t is computed from the matrix here;
# --- kind "rank": the ranking is the moderated t of the eBayes table.
contrast_plan = [
    dict(name="HEK_PA_vs_RC", label="HEK EA vs Rest", system="HEK", kind="adata",
         adata=adata_log_autoph, group_col=biological_autoph,
         cond_a="-BafA1 PFF Pure Aggregates", cond_b="-BafA1 PFF Rest Cells"),
    dict(name="iPSC_SAgg_vs_Soma", label="iPSC SAgg vs Soma", system="iPSC", kind="adata",
         adata=adata_strict_no_contaminants_ipsc, group_col=biological_ipsc,
         cond_a="iDA D56 PFF N2 Aggregates Somatic",
         cond_b="iDA D56 PFF N2 Soma Without Aggregates"),
    dict(name="Brain_Round_vs_DABpos", label="Brain Round vs DAB+", system="Brain", kind="rank",
         result=ebayes_results_brain["Round_vs_NeuronDABpos_Disease"]),
]


def _rank_from_ebayes(res):
    """Moderated t per gene symbol, from a donor-level eBayes table.

    One row per symbol, the one with the highest mean abundance (AveExpr), as in
    GSEA_PIPELINE.md §1. The row with the largest |t| is not used: the most
    extreme row of a symbol is usually the least reliable one.
    """
    r = res.dropna(subset=["single_gene_name", "stat", "AveExpr"]).copy()
    r = (r.sort_values("AveExpr", ascending=False)
          .drop_duplicates(subset=["single_gene_name"], keep="first"))
    s = r.set_index("single_gene_name")["stat"].astype(float)
    return s[np.isfinite(s)].sort_values(ascending=False)


gsea_res_xs = {}
for c in contrast_plan:
    print(f"\n{'='*78}\n{c['name']}  [{c['system']}]\n{'='*78}")
    cfg = CFG[c["system"]]
    if c["kind"] == "adata":
        gsea_res_xs[c["name"]] = run_contrast(
            c["adata"], c["group_col"], c["cond_a"], c["cond_b"],
            c["name"], gene_sets, cfg)
    else:
        rank = _rank_from_ebayes(c["result"])
        gsea_res_xs[c["name"]] = run_contrast_from_rank(
            rank, c["name"], gene_sets, cfg,
            ranking_description=(
                "limma eBayes moderated t on donor-level pseudobulk "
                "(one row per donor per group), symbols collapsed by highest AveExpr"),
            extra_params={"system": c["system"], "design": "donor-level pseudobulk"})

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# GSEA [2/3]: cluster redundant terms by leading-edge overlap and write one
# decision file review.csv per contrast. Nothing is dropped here. Each contrast
# is clustered against its own system's excluded terms, so a vascular term stays
# in brain and is removed in HEK. Decisions already in review.csv are carried
# over by term name.
# ═══════════════════════════════════════════════════════════════════════
# The decisions are an input. Without the file, build_review would write a new one
# and [3/3] would plot unreviewed clusters under their raw term names.
# build_review rewrites review.csv and gives no decision to a term whose cluster
# changed (MOVED), so each run first keeps a time-stamped copy of the file.
for c in contrast_plan:
    cfg = CFG[c["system"]]
    _review = cfg.contrast_dir(c["name"]) / "review.csv"
    assert _review.exists(), f"decision file missing: {_review}; check gsea_dir"
    shutil.copy2(_review, _review.with_name(f"review_before_{time.strftime('%Y%m%d_%H%M%S')}.csv"))
    build_review(gsea_res_xs[c["name"]], cfg)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# GSEA [3/3]: Figure 5 c, the three contrasts side by side.
#   HEK EA vs RC, iPSC SAgg vs Soma, Brain Round vs DAB+
#
# plot_combined builds the rows (one display label each, from the review.csv
# decisions), the term drawn per label and the grid from the contrasts it is
# given, reading each contrast's review.csv and raw table.
#
# cell_terms, anchor_rule and min_rows_per_contrast are passed explicitly. The
# defaults of plot_combined ("purity", "coverage", 3) score every candidate term
# against a gene list per label and stop with a KeyError for a label without one;
# the vocabulary of this notebook has gene lists for few labels. "canonical"
# draws one term per label, the one with the largest |NES| across the three
# contrasts.
#
# The panel is 2 x 2 units. The layout is plot_combined's own; the display
# settings (dot sizes, line widths, key position, label rotation) are set on a
# copy of the config in this cell.
# ═══════════════════════════════════════════════════════════════════════
XS_THREE = ["HEK_PA_vs_RC", "iPSC_SAgg_vs_Soma", "Brain_Round_vs_DABpos"]
XS3_SIZE = (2, 2)                    # panel size in units (columns, rows)
# Dot area at the smallest and the largest detected set size, from printed
# diameters of 2.0 and 5.0 pt, as in the iPSC and Autophagy GSEA panels.
XS3_DOT_AREA = (native_pt(2.0) ** 2, native_pt(5.0) ** 2)

_by_name = {c["name"]: c for c in contrast_plan}
_missing = [n for n in XS_THREE if n not in _by_name]
if _missing:
    raise KeyError(f"{_missing} not in contrast_plan; names are {sorted(_by_name)}")
xs3_labels = [_by_name[n]["label"] for n in XS_THREE]
print(f"Plotting {len(XS_THREE)} contrasts: {list(zip(XS_THREE, xs3_labels))}")

# Line widths from the style: plot_combined's defaults print below 0.25 pt on
# this canvas.
cfg_xs3 = replace(cfg_base,
                  dot_area_min=XS3_DOT_AREA[0], dot_area_max=XS3_DOT_AREA[1],
                  edge_lw_sig=MARKER_LW, edge_lw_ns=AXES_LW,
                  cross_lw=AXES_LW, grid_lw=THIN_LW,
                  key_edge_lw=MARKER_LW, key_marker_size=native_pt(3.0),
                  key_mew_sig=MARKER_LW, key_mew_ns=AXES_LW,
                  keys_position="corner", contrast_label_rotation=90)

# Two lines per contrast label: "HEK EA vs" above "Rest".
fig3, ax3 = plot_combined(XS_THREE, cfg_xs3,
                          contrast_labels=["\n".join(l.rsplit(" ", 1)) for l in xs3_labels],
                          rows_up=20, rows_down=5, save=False,
                          figsize=figsize(*XS3_SIZE),
                          save_stem="crosssystem_3contrasts_dotplot",
                          cell_terms="canonical",
                          anchor_rule="peak",
                          min_rows_per_contrast=0)

# ---------------------------------------------------------- CHECKS
# Printed, not asserted: the panel is saved either way, and save_panel reports
# anything drawn past the canvas edge (legends, clipped row labels).
if fig3 is not None:
    fig3.canvas.draw()
    _rows = len(ax3.get_yticklabels())
    _pitch = ax3.get_window_extent().height / fig3.dpi * 72 / _rows
    _dmax = np.sqrt(XS3_DOT_AREA[1])
    print(f"CHECK {_rows} rows; largest dot {_dmax / DRAW_SCALE:.1f} pt across, "
          f"row pitch {_pitch / DRAW_SCALE:.1f} pt (printed)")
    if _dmax >= _pitch:
        print("  !! dots are taller than the row pitch; lower XS3_DOT_AREA[1]")

    save_panel(fig3, "c", "gsea_crosssystem_3contrasts", FIG5_DIR)

# 11. Published aggregate proteomes (Figure S7e, f)

Gene lists of published α-synuclein aggregate proteomes from `LITERATURE_MASTER.xlsx`: the share of each list detected in the aggregate fractions (Figure S7e), the share in the UP sets (Figure S7f), a consensus list, and the genes of the UP sets that no list reports (the "Novel" groups of Figure 5b and Figure S7c).

In [ ]:
# ============================================================
# Literature gene lists drawn in Figure S7 e and f: one list per study.
# Per sheet: the first column whose name contains "gene", "symbol" or "hgnc"
# (else the first column), upper case, first symbol of "GENE1;GENE2", unique.
# ============================================================
panel_e_datasets = [
    # === Tier 1: primary αSyn aggregate composition (≥30 proteins, MS-based, non-redundant) ===
    "Choi2024", "Greally2024", "Laferriere2022", "MahulMellier2020",
    "Conod2023", "Lumpkin2024", "Shahmoradian2019",
    "Tanudjojo2021", "Killinger2022",
    # === Tier 2: IHC canon (a compendium, drawn separately because it is not MS) ===
    "Wakabayashi2013",
]

xl = pd.ExcelFile(literature_list)

lit_lists = {}
for sheet in panel_e_datasets:
    if sheet not in xl.sheet_names:
        print(f"  ⚠  '{sheet}' not in xlsx — skipping");  continue
    df_sheet = pd.read_excel(xl, sheet_name=sheet)
    gene_cols = [c for c in df_sheet.columns
                 if any(k in str(c).lower() for k in ["gene", "symbol", "hgnc"])]
    col = gene_cols[0] if gene_cols else df_sheet.columns[0]
    genes = (df_sheet[col].dropna().astype(str)
                 .str.upper().str.strip()
                 .str.split(";").str[0])      # "GENE1;GENE2" -> GENE1
    genes = sorted({g for g in genes if g and g != "NAN"})
    lit_lists[sheet] = set(genes)
    print(f"  {sheet:<18} → {len(genes):>3} genes from column {col!r}")

In [ ]:
# Shared-UP sets of the literature comparison (Figure S7 f)
#   up_3sys = shared-UP in df_cs     (HEK, iPSC and brain)
#   up_2sys = shared-UP in df_cs_ib  (iPSC and brain)
up_3sys = set(df_cs[df_cs["category"] == "shared-UP"].index.astype(str).str.upper())
up_2sys = set(df_cs_ib[df_cs_ib["category"] == "shared-UP"].index.astype(str).str.upper())

print(f"up_3sys: {len(up_3sys)}")
print(f"up_2sys: {len(up_2sys)}")

In [ ]:
# ============================================================
# Consensus list of the literature (the last list of Figure S7 e and f): genes
# reported by at least MIN_LISTS independent units of composition evidence.
#
#   The INDEX sheet gives each sheet a tier and notes.
#   tier   only "primary" and "canon" sheets are composition evidence of
#          α-synuclein aggregates and are counted. "cross_disease_control" sheets
#          (tau and AD lists) are reported as a specificity control below.
#          Interactome sheets are not counted, because a proximity list places a
#          protein near α-synuclein, not in the aggregate; discussion sheets are
#          not counted, because they report single proteins.
#   notes  some sheets are the same experiment twice. DUPLICATE_GROUPS merges each
#          such group into one unit, so an experiment counts once.
# ============================================================
MIN_LISTS = 3
KEEP_TIERS = {"primary", "canon"}
CONTROL_TIERS = {"cross_disease_control"}      # reported, never counted

# One experiment per group, as stated in the notes of the INDEX sheet
DUPLICATE_GROUPS = [
    {"Leverenz2007", "Xia2008"},               # identical lists (38 of 38 genes)
    {"Choi2024", "Killinger2025_topHits"},     # one dataset (13 of the smaller list's 15)
    {"Lougee2025_all", "Lougee2025_fibril"},   # the fibril list is a subset of the full list
]

idx = pd.read_excel(xl, sheet_name="INDEX").set_index("sheet_name")
_missing_idx = [s for s in xl.sheet_names[1:] if s not in idx.index]
if _missing_idx:
    raise KeyError(f"sheets absent from INDEX, so their tier is unknown: {_missing_idx}")

def _genes(sheet):
    d = pd.read_excel(xl, sheet_name=sheet)
    gc = [c for c in d.columns
          if any(k in str(c).lower() for k in ("gene", "symbol", "hgnc"))]
    col = gc[0] if gc else d.columns[0]
    g = (d[col].dropna().astype(str).str.upper().str.strip().str.split(";").str[0])
    return {x for x in g if x and x != "NAN"}

_sel = [s for s in xl.sheet_names[1:] if idx.loc[s, "tier"] in KEEP_TIERS]
G = {s: _genes(s) for s in xl.sheet_names[1:]}

# ---- each duplicate group becomes one unit -------------------------------
units, _used = [], set()
for grp in DUPLICATE_GROUPS:
    inside = sorted(grp & set(_sel))
    if not inside:
        continue
    units.append(("+".join(inside), set().union(*[G[s] for s in inside]), inside))
    _used |= set(inside)
for s in _sel:
    if s not in _used:
        units.append((s, G[s], [s]))

print(f"Composition evidence: {len(_sel)} sheets -> {len(units)} independent units")
print(f"{'unit':<34} {'tier':<9} {'genes':>6}  merged from")
for name, g, members in sorted(units, key=lambda u: -len(u[1])):
    tiers = "/".join(sorted({str(idx.loc[m, 'tier']) for m in members}))
    print(f"  {name[:32]:<32} {tiers:<9} {len(g):>6}  "
          f"{members if len(members) > 1 else ''}")
_dropped = [s for s in xl.sheet_names[1:] if s not in _sel]
print(f"\nNot counted ({len(_dropped)}), by tier:")
for t in sorted({str(idx.loc[s, 'tier']) for s in _dropped}):
    print(f"  {t:<22} {[s for s in _dropped if str(idx.loc[s,'tier']) == t]}")

# ---- count units per gene ------------------------------------------------
unit_counts = Counter()
for _, g, _ in units:
    unit_counts.update(g)
consensus_genes = {g for g, c in unit_counts.items() if c >= MIN_LISTS}
print(f"\nConsensus (>= {MIN_LISTS} of {len(units)} independent units): "
      f"{len(consensus_genes)} genes")

# ---- specificity control: overlap with the tau and AD lists ---------------
_ctl = {s: G[s] for s in xl.sheet_names[1:]
        if idx.loc[s, "tier"] in CONTROL_TIERS}
if _ctl:
    _any = set().union(*_ctl.values())
    print(f"\nSpecificity control — consensus genes also in a non-aSyn list:")
    for s, g in sorted(_ctl.items(), key=lambda kv: -len(kv[1] & consensus_genes)):
        ov = len(g & consensus_genes)
        print(f"  {s:<20} {ov:>4} of {len(consensus_genes)} "
              f"({100 * ov / max(len(consensus_genes), 1):.1f}%)   "
              f"[{str(idx.loc[s, 'notes'])[:52]}]")
    print(f"  any control list  {len(_any & consensus_genes):>4} of "
          f"{len(consensus_genes)} "
          f"({100 * len(_any & consensus_genes) / max(len(consensus_genes), 1):.1f}%)")
    print("  This is a result, not a contaminant: a high fraction means the "
          "consensus is aggregate-generic rather than aSyn-specific.")

# ----------------------------------------------------------------- CHECKS
print(f"\nCHECK — SNCA in the consensus: {'SNCA' in consensus_genes}")
_partial = [sorted(grp) for grp in DUPLICATE_GROUPS
            if 0 < len(grp & set(_sel)) < len(grp)]
if _partial:
    print(f"CHECK — duplicate groups only partly inside the selection: {_partial}")
    print("        the excluded member is in another tier, so no double count "
          "arises; re-check the group if the tiers change.")
# A duplicate missing from DUPLICATE_GROUPS would be counted twice, so the notes
# of the INDEX sheet are searched for duplication keywords.
_flag = idx.index[idx["notes"].astype(str).str.contains(
    "duplicat|SAME PRIMARY|Same MS dataset|subset", case=False, na=False)]
_unhandled = [s for s in _flag if s in _sel
              and not any(s in grp for grp in DUPLICATE_GROUPS)]
print(f"CHECK — sheets whose notes flag duplication but are not in "
      f"DUPLICATE_GROUPS: {_unhandled if _unhandled else 'none'}")
_maxc = max(unit_counts.values()) if unit_counts else 0
print(f"CHECK — highest unit count for any gene: {_maxc} of {len(units)}")

# ---- the consensus becomes the last list of the two plots ---------------
CONSENSUS_NAME = f"Consensus"
lit_lists[CONSENSUS_NAME] = consensus_genes
if CONSENSUS_NAME not in panel_e_datasets:
    panel_e_datasets.append(CONSENSUS_NAME)
print(f"\nAdded '{CONSENSUS_NAME}' (n={len(consensus_genes)}) to lit_lists and "
      f"panel_e_datasets.")

In [ ]:
# ============================================================
# Detected genes per system, for Figure S7 e.
# system_any[sys] = symbols with at least one value anywhere in that system
# system_agg[sys] = symbols with at least one value in its aggregate fractions
# Both come from the same filtered AnnData per system, so "detected" means
# "passed that system's completeness filter and was measured", not "present in
# the DIA-NN output". Symbols in upper case, as in lit_lists.
# ============================================================
AGG_LEVELS = {
    "HEK":   (adata_log_autoph, biological_autoph,
              ["-BafA1 PFF Pure Aggregates", "+BafA1 PFF Pure Aggregates"]),
    "iPSC":  (adata_strict_no_contaminants_ipsc, biological_ipsc,
              ["iDA D56 PFF N2 Aggregates Somatic",
               "iDA D56 PFF N2 Aggregates Neuritic"]),
    "Brain": (adata_log_brain, "CellType_Aggregate",
              ["Round Aggregate Aggregate", "Elongated Aggregate Aggregate"]),
}

def _detected(ad_sub):
    """Symbols with >=1 non-NaN value in this subset."""
    seen = np.asarray(~np.isnan(ad_sub.X)).any(axis=0)
    return set(ad_sub.var["single_gene_name"][seen]
                     .dropna().astype(str).str.upper().str.strip()) - {"", "NAN"}

system_any, system_agg = {}, {}
for _sys, (_ad, _col, _levels) in AGG_LEVELS.items():
    present = set(_ad.obs[_col].astype(str))
    missing = [l for l in _levels if l not in present]
    assert not missing, f"{_sys}: {_col} has no level(s) {missing}. Present: {sorted(present)}"
    mask = _ad.obs[_col].astype(str).isin(_levels).values
    system_any[_sys] = _detected(_ad)
    system_agg[_sys] = _detected(_ad[mask, :])
    print(f"{_sys:6s} {_ad.n_obs:4d} samples ({int(mask.sum())} aggregate)  "
          f"any={len(system_any[_sys]):5d}  aggregate={len(system_agg[_sys]):5d} symbols")

In [ ]:
# ============================================================
# Figure S7 e: share of each literature list detected in the aggregate
# fractions, per system.
# ============================================================
systems = ["HEK", "iPSC", "Brain"]
bar_h, group_pad = 0.7, 0.3   # group_pad: gap between two lists, in the units of bar_h

shown = [s for s in panel_e_datasets if s in lit_lists]

# 1 x 2 units, as Figure S7 f, so the two compose side by side.
fig, axm = create_figure(1, 1, figsize=figsize(1, 2))
ax = axm.next()
ax.grid(False)

y_ticks, y_labels = [], []
last_y_pos = 0
xmax_seen = 0

for i, sheet in enumerate(shown):
    L = lit_lists[sheet]
    n_total = len(L)
    if n_total == 0:
        continue
    y_base = -i * (3 * bar_h + group_pad)

    for j, sys in enumerate(systems):
        y_pos      = y_base - j * bar_h
        last_y_pos = y_pos
        pct_agg    = 100 * len(L & system_agg[sys]) / n_total
        ax.barh(y_pos, pct_agg, height=bar_h * 0.85,
                color=SYSTEM_COLORS[sys], edgecolor="black", linewidth=MARKER_LW)
        ax.text(pct_agg + 1, y_pos, f"{pct_agg:.0f}%",
                va="center", ha="left", fontsize=ANNOT_FS)
        xmax_seen = max(xmax_seen, pct_agg)

    y_ticks.append(y_base - bar_h)
    y_labels.append(f"{sheet}\n(n={n_total})")

ax.set_ylim(last_y_pos - bar_h * 0.6, bar_h * 0.6)
ax.set_xlim(0, min(105, xmax_seen + 12))
ax.set_xticks([0, 50, 100])   # at 6 pt on a 20 mm axis, "75" and "100" touched
ax.set_yticks(y_ticks)
ax.set_yticklabels(y_labels)
ax.set_xlabel("% of list detected")

legend_handles = [Patch(facecolor=SYSTEM_COLORS[s], edgecolor="black",
                        linewidth=MARKER_LW, label=s) for s in systems]

# Legend in the corner below the list names and left of the x tick labels, so
# the bars keep the full height. Two columns keep it no lower than the x tick
# labels and the x label.
ax.legend(handles=legend_handles, loc="upper right", bbox_to_anchor=(0, 0), ncol=2,
          frameon=False, borderaxespad=0.6, borderpad=0.2, labelspacing=0.3,
          handlelength=0.7, handletextpad=0.3, columnspacing=0.8)

save_panel(fig, "e", "lit_overlap_detection", SI7_DIR)

In [ ]:
# ============================================================
# Figure S7 f: share of each literature list in three UP sets.
# Per list three bars: 3-system shared-UP, 2-system shared-UP, and brain UP
# (log2FC > UNIV_LFC in Round or Elong, fold change only). Layout as Figure S7 e.
# ============================================================
def _up_genes(res, lfc_thresh=UNIV_LFC):
    m = res["log2fc"] > lfc_thresh
    return set(res.loc[m, "single_gene_name"].dropna()
                  .astype(str).str.upper().str.strip())

brain_up_set = (_up_genes(ebayes_results_brain["Round_vs_NeuronDABpos_Disease"]) |
                _up_genes(ebayes_results_brain["Elong_vs_Neuropil_Disease"]))

print(f"Brain UP    (log2FC > {UNIV_LFC} in Round or Elong): {len(brain_up_set)} genes")
print(f"2-sys UP    (shared-UP in df_cs_ib):              {len(up_2sys)} genes")
print(f"3-sys UP    (shared-UP in df_cs):                 {len(up_3sys)} genes")

# One hue in three shades: the three bars are one category (an UP set) at three
# levels of cross-system agreement, so they differ in lightness, and UP_COLOR is
# the hue of "up" in every panel. Top to bottom within a list: 3-sys, 2-sys, Brain.
UP_TIERS = [
    ("3-sys", up_3sys,      shade(UP_COLOR, 0.00)),   # the x label says "UP sets";
    ("2-sys", up_2sys,      shade(UP_COLOR, 0.28)),   # with "UP" in each name the legend
    ("Brain", brain_up_set, shade(UP_COLOR, 0.55)),   # is 24 mm wide, the corner 20 mm
]

bar_h, group_pad = 0.7, 0.3   # group_pad: gap between two lists, in the units of bar_h

shown = [s for s in panel_e_datasets if s in lit_lists]

fig, axm = create_figure(1, 1, figsize=figsize(1, 2))
ax = axm.next()
ax.grid(False)

y_ticks, y_labels = [], []
last_y_pos = 0

for i, sheet in enumerate(shown):
    L = lit_lists[sheet]
    n_total = len(L)
    if n_total == 0:
        continue
    y_base = -i * (3 * bar_h + group_pad)

    for j, (cat_name, cat_set, color) in enumerate(UP_TIERS):
        y_pos      = y_base - j * bar_h
        last_y_pos = y_pos
        n_overlap  = len(L & cat_set)
        pct        = 100 * n_overlap / n_total
        ax.barh(y_pos, pct, height=bar_h * 0.85,
                color=color, edgecolor="black", linewidth=MARKER_LW)
        ax.text(pct + 0.4, y_pos, f"{pct:.0f}%",
                va="center", ha="left", fontsize=ANNOT_FS)

    y_ticks.append(y_base - bar_h)
    y_labels.append(f"{sheet}\n(n={n_total})")

ax.set_ylim(last_y_pos - bar_h * 0.6, bar_h * 0.6)
ax.set_yticks(y_ticks)
ax.set_yticklabels(y_labels)
ax.set_xlabel("% of list in UP sets")

legend_handles = [Patch(facecolor=c, edgecolor="black", linewidth=MARKER_LW, label=n)
                  for n, _, c in UP_TIERS]

# Legend in the corner below the list names and left of the x tick labels, so
# the bars keep the full height. Two columns keep it no lower than the x tick
# labels and the x label.
ax.legend(handles=legend_handles, loc="upper right", bbox_to_anchor=(0, 0), ncol=2,
          frameon=False, borderaxespad=0.6, borderpad=0.2, labelspacing=0.3,
          handlelength=0.7, handletextpad=0.3, columnspacing=0.8)

# x range that fits every percentage label: a label starting at data x with pixel
# width w needs the axis to run to x*W/(W-w), W being the axis width in pixels.
fig.canvas.draw()
_W = ax.get_window_extent().width
_need = 0.0
for _t in ax.texts:
    _w = _t.get_window_extent().width
    _x = _t.get_position()[0]
    if _W > _w:
        _need = max(_need, _x * _W / (_W - _w))
ax.set_xlim(0, _need * 1.03)

save_panel(fig, "f", "lit_overlap_enrichment", SI7_DIR)

In [ ]:
# ============================================================
# The consensus list gene by gene, in both directions: which of its genes the UP
# sets contain, which the DOWN sets contain, and at which level of cross-system
# agreement. Figure S7 f gives percentages for the UP side; this table names the
# genes behind them and does the same for DOWN.
#
#   Three sets per direction, in increasing order of the agreement they require:
#     brain      log2FC beyond +/- UNIV_LFC in Round or Elong, brain only and
#                fold change only, the rule of Figure S7 f, so the counts here are
#                the counts the bars draw;
#     2-sys      shared-UP / shared-DOWN in df_cs_ib, iPSC and brain;
#     3-sys      shared-UP / shared-DOWN in df_cs, HEK and iPSC and brain.
#   The sets are not nested, so each gene carries all three flags, and `tier`
#   names the strictest set it is in, within its direction.
#
#   The brain set is a union over two contrasts and thresholds a fold change only,
#   so a gene can be in both directions (UP in one contrast, DOWN in the other).
#   Both memberships are kept, so the counts still match the bars; the tier
#   records whether the brain FDR of that side is below UNIV_FDR, and the genes in
#   both directions are printed with both FDRs.
#
#   The DOWN side: the literature reports these proteins in aggregates, so a gene
#   that the three systems place below its host fraction disagrees with the
#   literature.
#
#   A gene in neither direction may not have been tested. tested_3sys and
#   tested_2sys record whether the gene has a value in df_cs and df_cs_ib, which
#   separates "not measured" from "measured and not enriched".
#
#   One row per gene and direction, one CSV for both.
# ============================================================
CONSENSUS_KEY = "Consensus"
assert CONSENSUS_KEY in lit_lists, \
    f"{CONSENSUS_KEY!r} is not in lit_lists; it holds {sorted(lit_lists)}"
for _name, _df in (("df_cs", df_cs), ("df_cs_ib", df_cs_ib)):
    _seen = set(_df["category"].unique())
    assert _seen <= {"shared-UP", "shared-DOWN", "none"}, (
        f"{_name}.category holds {sorted(_seen)}; expected shared-UP, "
        f"shared-DOWN and none")
assert up_3sys and up_2sys, "up_3sys or up_2sys is empty"

BRAIN_KEYS = ("Round_vs_NeuronDABpos_Disease", "Elong_vs_Neuropil_Disease")


def _brain_dir(sign, lfc_thresh=UNIV_LFC):
    """Genes past +/- lfc_thresh in either brain contrast, fold change only.

    The rule of Figure S7 f, so these flags reproduce the bars. The FDR enters
    the table, not the set (_brain_fdr below).
    """
    out = set()
    for _k in BRAIN_KEYS:
        _r = ebayes_results_brain[_k]
        _m = (_r["log2fc"] > lfc_thresh) if sign > 0 else (_r["log2fc"] < -lfc_thresh)
        out |= set(_r.loc[_m, "single_gene_name"].dropna()
                    .astype(str).str.upper().str.strip())
    return out


def _brain_fdr(sign, lfc_thresh=UNIV_LFC):
    """Per gene, the smallest FDR among the brain contrasts that put it past
    the threshold on this side.

    A gene UP in one brain contrast and DOWN in the other is in both
    directions under the fold-change rule; the FDR of each side shows which
    side is significant, without changing the counts.
    """
    best = {}
    for _k in BRAIN_KEYS:
        _r = ebayes_results_brain[_k]
        _m = (_r["log2fc"] > lfc_thresh) if sign > 0 else (_r["log2fc"] < -lfc_thresh)
        _sub = _r.loc[_m, ["single_gene_name", "fdr"]].dropna()
        for _g, _f in zip(_sub["single_gene_name"].astype(str).str.upper().str.strip(),
                          _sub["fdr"].astype(float)):
            if _g not in best or _f < best[_g]:
                best[_g] = _f
    return best


brain_up,   brain_up_fdr   = _brain_dir(+1), _brain_fdr(+1)
brain_down, brain_down_fdr = _brain_dir(-1), _brain_fdr(-1)
BRAIN_FDR = {"UP": brain_up_fdr, "DOWN": brain_down_fdr}
assert brain_up == brain_up_set, (
    f"the brain UP set built here has {len(brain_up)} genes, the one Figure S7 f "
    f"uses has {len(brain_up_set)}. Both must apply the same rule.")
down_3sys = set(df_cs[df_cs["category"] == "shared-DOWN"].index.astype(str).str.upper())
down_2sys = set(df_cs_ib[df_cs_ib["category"] == "shared-DOWN"].index.astype(str).str.upper())
_both_brain = brain_up & brain_down
_both_sig = {g for g in _both_brain
             if brain_up_fdr.get(g, 1) < UNIV_FDR and brain_down_fdr.get(g, 1) < UNIV_FDR}
print(f"brain UP {len(brain_up)}, brain DOWN {len(brain_down)}, "
      f"{len(_both_brain)} in both on fold change, of which {len(_both_sig)} "
      f"are significant on BOTH sides (a real difference between the two "
      f"aggregate morphologies); the rest carry one supported side and one "
      f"unsupported point estimate")
print(f"2-sys UP {len(up_2sys)}, 2-sys DOWN {len(down_2sys)}; "
      f"3-sys UP {len(up_3sys)}, 3-sys DOWN {len(down_3sys)}")

_cons = sorted({g.upper() for g in lit_lists[CONSENSUS_KEY]})
_i3 = {g.upper() for g in df_cs.index}
_i2 = {g.upper() for g in df_cs_ib.index}
_lfc3 = {g.upper(): r for g, r in df_cs[["HEK", "iPSC", "brain"]].iterrows()}
_lfc2 = {g.upper(): r for g, r in df_cs_ib[["iPSC", "brain"]].iterrows()}

DIRECTIONS = [("UP", brain_up, up_2sys, up_3sys),
              ("DOWN", brain_down, down_2sys, down_3sys)]

rows = []
for _dir, _brain, _two, _three in DIRECTIONS:
    for _g in _cons:
        _b, _2, _3 = _g in _brain, _g in _two, _g in _three
        _f = BRAIN_FDR[_dir].get(_g, np.nan)
        _bsig = bool(np.isfinite(_f) and _f < UNIV_FDR)
        _tier = (f"3-sys {_dir}" if _3 else f"2-sys {_dir}" if _2 else
                 (f"brain {_dir} only, significant" if _bsig else
                  f"brain {_dir} only, fold change not significant") if _b else
                 f"not in any {_dir} set")
        _r3, _r2 = _lfc3.get(_g), _lfc2.get(_g)
        rows.append(dict(
            gene=_g, direction=_dir, tier=_tier,
            brain=_b, brain_fdr=_f, brain_significant=_bsig,
            two_sys=_2, three_sys=_3,
            tested_3sys=_g in _i3, tested_2sys=_g in _i2,
            lfc_HEK=float(_r3["HEK"]) if _r3 is not None else np.nan,
            lfc_iPSC=float(_r3["iPSC"]) if _r3 is not None
                     else (float(_r2["iPSC"]) if _r2 is not None else np.nan),
            lfc_brain=float(_r3["brain"]) if _r3 is not None
                      else (float(_r2["brain"]) if _r2 is not None else np.nan)))
consensus_members = pd.DataFrame(rows)

print(f"\n{CONSENSUS_KEY}: {len(_cons)} genes, "
      f"{int(consensus_members.tested_3sys.sum() / 2)} tested in all three "
      f"systems, {int(consensus_members.tested_2sys.sum() / 2)} in iPSC and "
      f"brain")

for _dir, *_ in DIRECTIONS:
    _tiers = [f"3-sys {_dir}", f"2-sys {_dir}",
              f"brain {_dir} only, significant",
              f"brain {_dir} only, fold change not significant",
              f"not in any {_dir} set"]
    print(f"\n--- {_dir} ---")
    for _t in _tiers:
        _sub = consensus_members[(consensus_members.direction == _dir)
                                 & (consensus_members.tier == _t)]
        print(f"{_t} ({len(_sub)}, {len(_sub) / len(_cons):.0%} of the list):")
        print(f"  {', '.join(sorted(_sub.gene)) if len(_sub) else 'none'}\n")

# consensus genes that are shared-DOWN in all three systems disagree most with
# the literature; they are printed separately
_contra = consensus_members[(consensus_members.direction == "DOWN")
                            & consensus_members.three_sys]
_up_rows = consensus_members[consensus_members.direction == "UP"].set_index("gene")
_dn_rows = consensus_members[consensus_members.direction == "DOWN"].set_index("gene")
_conflict = sorted(set(_up_rows[_up_rows.brain].index)
                   & set(_dn_rows[_dn_rows.brain].index))
if _conflict:
    print(f"\nNOTE {len(_conflict)} consensus gene(s) pass the fold-change "
          f"threshold in BOTH directions, because the brain tier is a union "
          f"over two contrasts:")
    for _g in _conflict:
        _fu, _fd = _up_rows.loc[_g, "brain_fdr"], _dn_rows.loc[_g, "brain_fdr"]
        _verdict = ("both sides significant: the two aggregate morphologies "
                    "differ for this gene" if _fu < UNIV_FDR and _fd < UNIV_FDR
                    else f"only the {'UP' if _fu < _fd else 'DOWN'} side is "
                         f"supported")
        print(f"  {_g}: UP FDR {_fu:.3g}, DOWN FDR {_fd:.3g} - {_verdict}")
print(f"consensus genes our three systems put DOWN (literature says these are "
      f"in aggregates): {', '.join(sorted(_contra.gene)) if len(_contra) else 'none'}")

_never = consensus_members[(consensus_members.direction == "UP")
                           & ~consensus_members.tested_3sys
                           & ~consensus_members.tested_2sys]
print(f"\nnever tested in either aligned matrix ({len(_never)} of "
      f"{len(_cons)}): {', '.join(sorted(_never.gene)) if len(_never) else 'none'}")

# --- CHECKS --------------------------------------------------------------
assert len(consensus_members) == 2 * len(_cons), \
    "the table is not one row per gene and direction"
for _dir, _brain, _two, _three in DIRECTIONS:
    _d = consensus_members[consensus_members.direction == _dir]
    assert (_d.loc[_d.tier == f"3-sys {_dir}", "three_sys"]).all(), \
        f"a gene is tiered 3-sys {_dir} without the flag"
_summary = (consensus_members.groupby(["direction", "tier"]).size()
            .rename("genes").reset_index())
print(f"\nCHECK {len(consensus_members)} rows = 2 x {len(_cons)} consensus "
      f"genes; tier counts:")
print(_summary.to_string(index=False))
_recount = {f"{d} {lab}": int(consensus_members[
                (consensus_members.direction == d)][col].sum())
            for d, _b, _t, _th in DIRECTIONS
            for lab, col in (("brain", "brain"), ("2-sys", "two_sys"),
                             ("3-sys", "three_sys"))}
print(f"CHECK the UP counts are the ones Figure S7 f draws for {CONSENSUS_KEY} "
      f"({ {k: f'{100 * v / len(_cons):.0f}%' for k, v in _recount.items() if k.startswith('UP')} }"
      f" of {len(_cons)} genes): {_recount}")

_path = f"{output_dir}/{today}_consensus_membership.csv"
consensus_members.sort_values(["direction", "tier", "gene"]).to_csv(_path, index=False)
print(f"\nwritten: {_path}")

In [ ]:
# ============================================================
# Novel genes: genes of the UP sets that no α-synuclein list of the workbook
# reports (the "Novel" groups of Figure 5 b and Figure S7 c).
#
# Three backgrounds are computed; novelty is judged against the middle one:
#
#   composition   tier primary and canon, the sheets the consensus counts.
#   asyn_all      composition, interactome and discussion: every sheet about
#                 α-synuclein aggregates. This is the background of "novel": a
#                 gene named in a proximity list has been reported, even if it is
#                 not in the consensus.
#   everything    asyn_all and the tau and AD control lists. Reported per gene
#                 (in_control_lists): a gene known from tau is not thereby known
#                 in α-synuclein, but it may be generic to aggregates.
# ============================================================
_sheets = list(G)
BG = {
    "composition": {s for s in _sheets if idx.loc[s, "tier"] in KEEP_TIERS},
    "asyn_all":    {s for s in _sheets if idx.loc[s, "tier"] not in CONTROL_TIERS},
    "everything":  set(_sheets),
}
_union = {k: set().union(*[G[s] for s in v]) if v else set() for k, v in BG.items()}
for k in ("composition", "asyn_all", "everything"):
    print(f"background {k:<12} {len(BG[k]):>2} sheets -> "
          f"{len(_union[k]):>5} unique genes")
KNOWN = _union["asyn_all"]          # the background novelty is judged against
print(f"\nNovelty is judged against 'asyn_all': {len(KNOWN)} genes\n")

# ---- UP sets of this study -----------------------------------------------
brain_round = ebayes_results_brain["Round_vs_NeuronDABpos_Disease"].copy()
brain_round["gene_u"] = (brain_round["single_gene_name"].astype(str)
                         .str.upper().str.strip())
brain_round_up = set(brain_round.loc[brain_round["log2fc"] > UNIV_LFC, "gene_u"])
brain_round_map = (brain_round.sort_values("log2fc", ascending=False)
                   .drop_duplicates("gene_u").set_index("gene_u")[["log2fc", "fdr"]])

novel_3sys  = up_3sys - KNOWN
novel_2sys  = up_2sys - KNOWN
novel_round = brain_round_up - KNOWN
print(f"3-sys shared-UP        : {len(up_3sys):>4}   novel: {len(novel_3sys):>4}")
print(f"2-sys shared-UP (iP+B) : {len(up_2sys):>4}   novel: {len(novel_2sys):>4}")
print(f"brain Round vs DAB+ UP    : {len(brain_round_up):>4}   novel: {len(novel_round):>4}")

# ---- per-gene literature provenance -------------------------------------
_comp_units = {}
for name, g, members in units:
    for x in g:
        _comp_units.setdefault(x, set()).add(name)

def _annotate(genes, source_set, source_df=None):
    rows = []
    for g in sorted(genes):
        row = {"gene": g, "source": source_set}
        if source_df is not None and g in source_df.index:
            for col in source_df.columns:
                if col != "category":
                    row[f"LFC_{col}"] = source_df.loc[g, col]
        if g in brain_round_map.index:
            row["brain_Round_LFC"] = brain_round_map.loc[g, "log2fc"]
            row["brain_Round_FDR"] = brain_round_map.loc[g, "fdr"]
        # literature provenance: 0 composition units for a novel gene by
        # construction; the control lists can still contain it
        row["n_composition_units"] = len(_comp_units.get(g, ()))
        _in_ctl = sorted(s for s in BG["everything"] - BG["asyn_all"] if g in G[s])
        row["in_control_lists"] = ";".join(_in_ctl)
        row["known_in_tau_or_AD_only"] = bool(_in_ctl)
        rows.append(row)
    return pd.DataFrame(rows)

df_3sys  = _annotate(novel_3sys,  "3-sys shared-UP", df_cs[["HEK", "iPSC", "brain"]])
df_2sys  = _annotate(novel_2sys,  "2-sys shared-UP", df_cs_ib[["iPSC", "brain"]])
df_round = _annotate(novel_round, "brain Round vs DAB+")

for _lab, _df, _sort in (("3-sys shared-UP", df_3sys, None),
                         ("2-sys shared-UP", df_2sys, None),
                         ("brain Round vs DAB+", df_round, "brain_Round_LFC")):
    print(f"\n=== Novel in {_lab} ===")
    if not len(_df):
        print("  (none)"); continue
    _lfc = [c for c in _df.columns if c.startswith("LFC_")]
    if _lfc:
        _df["max_LFC"] = _df[_lfc].max(axis=1)
    key = _sort or ("max_LFC" if _lfc else "gene")
    _show = _df.sort_values(key, ascending=False)
    if _lab.startswith("brain"):
        print(f"  (top 30 of {len(_df)})"); _show = _show.head(30)
    print(_show.to_string(index=False, float_format=lambda v: f"{v:+.3f}"))

# ----------------------------------------------------------------- CHECKS
print(f"\nCHECK — novel genes with a composition-unit hit (must be 0): "
      f"{int(sum(r for d in (df_3sys, df_2sys, df_round) if len(d) for r in d['n_composition_units']))}")
for _lab, _s in (("3-sys", novel_3sys), ("2-sys", novel_2sys),
                 ("round", novel_round)):
    _ctl = {g for g in _s if any(g in G[s] for s in BG["everything"] - BG["asyn_all"])}
    print(f"CHECK — {_lab}: {len(_ctl)} of {len(_s)} 'novel' genes are already "
          f"reported in a tau or AD list — novel for aSyn, not unreported")
_alt = {k: (len(up_3sys - _union[k]), len(brain_round_up - _union[k])) for k in _union}
print(f"CHECK — novel counts under the other backgrounds (3-sys, brain round):")
for k, v in _alt.items():
    print(f"    {k:<12} {v[0]:>4}, {v[1]:>4}")

all_novel = pd.concat([df_3sys.assign(set="3-sys"),
                       df_2sys.assign(set="2-sys"),
                       df_round.assign(set="round-vs-DABpos")], ignore_index=True)
all_novel.to_csv(f"{output_dir}/{today}_novel_markers.csv", index=False)
print(f"\nSaved -> {output_dir}/{today}_novel_markers.csv")

# 12. Protein complexes (Figure 5e, f; Figure S8n)

Inputs in `data/CrossSystem/Complexes/`, downloaded 2026-09-10: `9606.tsv`, the human complexes of the Complex Portal (https://ftp.ebi.ac.uk/pub/databases/intact/complex/current/complextab/), and `uniprot_human_covariates_v2.tsv`, the reviewed human UniProt entries with gene names, subcellular location, transmembrane features, mass and length (https://rest.uniprot.org/uniprotkb/stream?query=reviewed:true+AND+organism_id:9606&fields=accession,gene_primary,gene_synonym,cc_subcellular_location,ft_transmem,mass,length&format=tsv).

## 12.1 Complexes per contrast: coherence and direction

In [ ]:
# ============================================================
# Complexes from the Complex Portal (human, 9606.tsv), for Figure 5 e and f and
# Figure S8 n -> complexes_normalised.csv
#
#   participants -> bare UniProt accession -> primary gene symbol
#   >= MIN_SUBUNITS distinct symbols per complex
#   variants of one complex merged into one row (without this the V-ATPase is
#   counted four times)
# ============================================================

MIN_SUBUNITS = 4
MAX_UNION_INFLATION = 2.0      # see the UNION REJECTED block below
DROP_PREFIX = ("CPX-", "CHEBI:", "URS")

_cp  = pd.read_csv(f"{CX_DIR}/9606.tsv", sep="\t", dtype=str)
_uni = pd.read_csv(f"{CX_DIR}/uniprot_human_covariates_v2.tsv", sep="\t", dtype=str)
_acc2sym = (_uni.dropna(subset=["Entry", "Gene Names (primary)"])
                .assign(_s=lambda d: d["Gene Names (primary)"].str.strip())
                .query("_s != ''").set_index("Entry")["_s"].to_dict())
print(f"Complex Portal {len(_cp)} complexes | UniProt {len(_acc2sym)} accession->symbol")

_unmapped = set()

def _participants(field):
    """Split on |, text before (, strip from the first -, map to symbol.

    Bracketed tokens such as [P15313,P21281] are interchangeable subunits at
    one position and are expanded into their accessions. As one unmappable
    token they would remove positions such as V0e and V1B of the V-ATPase, and
    complexes would fall below MIN_SUBUNITS. The per-contrast restriction (next
    cell) keeps the paralogue that a system quantifies.
    """
    out = set()
    for part in str(field).split("|"):
        tok = part.split("(")[0].strip()
        if not tok or tok.startswith(DROP_PREFIX):
            continue
        accs = ([a.strip() for a in tok.strip("[]").split(",")]
                if tok.startswith("[") else [tok])
        for a in accs:
            if a.startswith(DROP_PREFIX):       # before the - split, or CPX-1 becomes CPX
                continue
            a = a.split("-")[0]
            if not a:
                continue
            sym = _acc2sym.get(a)
            out.add(sym) if sym else _unmapped.add(a)
    return out

_cp["subunits"] = _cp["Identifiers (and stoichiometry) of molecules in complex"].map(_participants)
_cp["name"]  = _cp["Recommended name"].astype(str).str.strip()
_cp["n_raw"] = _cp["subunits"].map(len)
print(f"unmapped accessions: {len(_unmapped)} -> {sorted(_unmapped)}")

_big = _cp[_cp["n_raw"] >= MIN_SUBUNITS].copy()
print(f"complexes with >= {MIN_SUBUNITS} distinct symbols: {len(_big)} of {len(_cp)}")

def _base_name(nm):
    """Strip a trailing variant/isoform suffix, then everything after the last
    TOP-LEVEL comma.

    The suffix pattern allows a trailing token, so "COP9 signalosome variant 1"
    loses "variant 1"; without it, COP9 would enter twice.

    The comma must be at paren depth zero, otherwise
    "B(0,+)AT1-rBAT heteromeric amino acid transporter complex" truncates
    to "B(0".
    """
    nm = re.sub(r"\s+(variant|isoform)(\s+[\w.\-]{1,12})?$", "",
                str(nm).strip(), flags=re.I)
    depth, cut = 0, -1
    for i, ch in enumerate(nm):
        if ch == "(":
            depth += 1
        elif ch == ")":
            depth -= 1
        elif ch == "," and depth == 0:
            cut = i
    return nm[:cut].strip() if cut >= 0 else nm


_big["base"] = _big["name"].map(_base_name)
complexes = (_big.groupby("base")
             .agg(n_merged=("#Complex ac", "size"),
                  cx_acs=("#Complex ac", lambda s: ";".join(sorted(s))),
                  members=("name", lambda s: " | ".join(sorted(set(s)))),
                  subunits=("subunits", lambda s: set().union(*s)),
                  largest_member=("n_raw", "max")).reset_index())
complexes["n_subunits"] = complexes["subunits"].map(len)
complexes["union_inflation"] = complexes["n_subunits"] / complexes["largest_member"]
print(f"after paralogue collapse: {len(complexes)} complexes "
      f"({int((complexes.n_merged > 1).sum())} merged from >1 Complex Portal row)")

# The merge takes the union of the variants' subunits. This is correct when the
# variants share a core and differ in one subunit. When they share only a core
# and differ in the rest, the union is a set that no single complex contains
# (union_inflation > MAX_UNION_INFLATION). Such a group is kept as its largest
# single Complex Portal entry.
_over = complexes[complexes.union_inflation > MAX_UNION_INFLATION].copy()
print(f"\nUNION REJECTED — inflation > {MAX_UNION_INFLATION}x. These are "
      f"subunit-swap families, not one particle, so each falls back to its "
      f"largest single Complex Portal entry: {len(_over)}")
_fix = []
for _, r in _over.iterrows():
    _m = (_big[_big.base == r.base]
          .sort_values(["n_raw", "name"], ascending=[False, True]).iloc[0])
    _fix.append(dict(base=r.base, n_merged=1, cx_acs=_m["#Complex ac"],
                     members=_m["name"], subunits=_m["subunits"],
                     largest_member=int(_m.n_raw), n_subunits=int(_m.n_raw),
                     union_inflation=1.0))
    print(f"    {r.base}: {r.n_merged} rows unioned to {r.n_subunits} subunits "
          f"({r.union_inflation:.1f}x) -> kept '{_m['name']}' ({int(_m.n_raw)})")
complexes = pd.concat(
    [complexes[complexes.union_inflation <= MAX_UNION_INFLATION],
     pd.DataFrame(_fix)], ignore_index=True).reset_index(drop=True)

# ----------------------------------------------------------------- CHECKS
assert complexes.n_subunits.min() >= MIN_SUBUNITS
assert not complexes.base.duplicated().any()
print(f"\nCHECK size range {complexes.n_subunits.min()}-{complexes.n_subunits.max()}, "
      f"median {int(complexes.n_subunits.median())}")
for _p in ("Vacuolar proton translocating ATPase", "Spliceosom"):
    _m = complexes[complexes.base.str.contains(_p, case=False, regex=False)]
    print(f"CHECK {_p!r}: {len(_m)} rows, merged from {int(_m.n_merged.sum())}")

_out = complexes.assign(subunits=complexes.subunits.map(lambda s: ";".join(sorted(s))))
_out.drop(columns=["union_inflation"]).to_csv(f"{output_dir}/complexes_normalised.csv", index=False)
print(f"\nSaved {len(_out)} complexes -> {output_dir}/complexes_normalised.csv")

In [ ]:
# ============================================================
# One representative per cluster of near-identical complexes, then each complex
# restricted to the subunits quantified in each of the five contrasts.
#
#   The representative is chosen once, on the full Complex Portal subunit sets,
#   and applied to every contrast. A choice per contrast would bias the
#   cross-system comparison: when two contrasts keep different members of one
#   cluster, that complex is missing from the shared set, so the shared set
#   would contain only the clusters where both contrasts picked the same member.
#
#   Cluster rule: Jaccard > CX_JACCARD_MAX, the largest complex is kept. Removing
#   identical sets only is not enough: the spliceosome stages share most of their
#   subunits but are not identical, and would each count as a complex.
# ============================================================

CX_JACCARD_MAX = 0.8

# (name, system, result, adata for abundance, group column, (cond_a, cond_b))
# The last two are None for the brain contrasts: those are donor-level eBayes
# fits whose own AveExpr column already is the contrast's mean log2 abundance.
CX_CONTRASTS = [
    ("HEK_PA_vs_RC", "HEK",
     ttest_results_autoph["PureAggs_vs_RestCells_minusBafA1"], adata_log_autoph,
     biological_autoph, ("-BafA1 PFF Pure Aggregates", "-BafA1 PFF Rest Cells")),
    ("iPSC_SAgg_vs_Soma", "iPSC",
     ttest_results_ipsc["Somatic_Agg_vs_Soma"], adata_log_ipsc, biological_ipsc,
     ("iDA D56 PFF N2 Aggregates Somatic", "iDA D56 PFF N2 Soma Without Aggregates")),
    ("iPSC_NAgg_vs_Neurites", "iPSC",
     ttest_results_ipsc["Neuritic_Agg_vs_Neurites"], adata_log_ipsc, biological_ipsc,
     ("iDA D56 PFF N2 Aggregates Neuritic", "iDA D56 PFF N2 Neurites Without Aggregates")),
    ("Brain_Round_vs_DABpos", "Brain",
     ebayes_results_brain["Round_vs_NeuronDABpos_Disease"], adata_log_brain, None, None),
    ("Brain_Elong_vs_Neuropil", "Brain",
     ebayes_results_brain["Elong_vs_Neuropil_Disease"], adata_log_brain, None, None),
]
CX_CONTRAST_SYSTEM = {c[0]: c[1] for c in CX_CONTRASTS}


def _cx_symbols(x):
    """Upper-case, stripped symbols, as in det_*."""
    return (pd.Series(x, dtype="object").dropna().astype(str)
            .str.upper().str.strip().pipe(lambda s: s[~s.isin(["", "NAN", "NONE"])]))


def _quantified(res, label):
    miss = [c for c in ("single_gene_name", "log2fc") if c not in res.columns]
    if miss:
        raise KeyError(f"{label}: missing {miss}; columns are {list(res.columns)}")
    r = res.replace([np.inf, -np.inf], np.nan).dropna(subset=["single_gene_name", "log2fc"])
    return set(_cx_symbols(r["single_gene_name"]))


def _jaccard(a, b):
    return len(a & b) / len(a | b)


complexes["subunits_u"] = complexes.subunits.map(lambda s: set(_cx_symbols(list(s))))
assert (complexes.subunits_u.map(len) == complexes.subunits.map(len)).all(), \
    "upper-casing merged two distinct symbols inside one complex"

# --- global cluster collapse, on the full annotation ---------------------
_order = complexes.sort_values(["n_subunits", "base"], ascending=[False, True])
_reps, _cluster_log = [], []
for _, r in _order.iterrows():
    _hit = next((k for k in _reps if _jaccard(r.subunits_u, k.subunits_u) > CX_JACCARD_MAX),
                None)
    if _hit is None:
        _reps.append(r)
    else:
        _cluster_log.append(dict(dropped=r.base, representative=_hit.base,
                                 jaccard=round(_jaccard(r.subunits_u, _hit.subunits_u), 3),
                                 n_dropped=r.n_subunits, n_rep=_hit.n_subunits))
complexes_rep = pd.DataFrame(_reps).reset_index(drop=True)
_cl = pd.DataFrame(_cluster_log)
print(f"global cluster collapse at Jaccard > {CX_JACCARD_MAX}: "
      f"{len(complexes)} -> {len(complexes_rep)} representatives "
      f"({len(_cl)} folded in)")
if len(_cl):
    print(_cl.nlargest(8, "jaccard").to_string(index=False))
    _cl.to_csv(f"{output_dir}/{today}_complexes_clusters_folded.csv", index=False)

_ws = list(complexes_rep.subunits_u)
_worst = max((_jaccard(_ws[i], _ws[j]) for i in range(len(_ws)) for j in range(i)),
             default=0.0)
assert _worst <= CX_JACCARD_MAX + 1e-12, f"representatives still pair at J={_worst:.3f}"
print(f"CHECK worst pair among representatives J = {_worst:.3f} "
      f"(rule: <= {CX_JACCARD_MAX})")
print(f"CHECK spliceosome representatives: "
      f"{int(complexes_rep.base.str.contains('Spliceosom', case=False).sum())} "
      f"(from {int(complexes.base.str.contains('Spliceosom', case=False).sum())})")

# --- per-contrast restriction, no further deduplication -----------------
complexes_by_contrast, _summary = {}, []
for _name, _system, _res, _ad, _gcol, _conds in CX_CONTRASTS:
    _q = _quantified(_res, _name)
    _df = (complexes_rep.assign(q=complexes_rep.subunits_u.map(lambda s: s & _q))
                        .assign(n_q=lambda d: d.q.map(len))
                        .query(f"n_q >= {MIN_SUBUNITS}")
                        .sort_values(["n_q", "base"], ascending=[False, True])
           [["base", "n_subunits", "n_q", "q", "n_merged", "cx_acs"]]
           .rename(columns={"base": "complex", "q": "quantified_subunits"})
           .reset_index(drop=True))
    complexes_by_contrast[_name] = _df
    _dupsets = len(_df) - len({frozenset(s) for s in _df.quantified_subunits})
    _summary.append(dict(contrast=_name, system=_system, quantified_genes=len(_q),
                         complexes=len(_df), median_subunits=int(_df.n_q.median()),
                         identical_quantified_sets=_dupsets,
                         spliceosome_rows=int(_df["complex"].str.contains(
                             "Spliceosom", case=False).sum())))
    print(f"{_name:<26} {len(_q):>5} genes -> {len(_df):>3} complexes")

print()
print(pd.DataFrame(_summary).to_string(index=False))

# ----------------------------------------------------------------- CHECKS
# Complexes present in one contrast and absent from another
_sets = {n: set(d["complex"]) for n, d in complexes_by_contrast.items()}
_all = set().union(*_sets.values())
print(f"\nCHECK presence across contrasts, {len(_all)} distinct complexes total")
print(f"    in all {len(_sets)} contrasts: "
      f"{len(set.intersection(*_sets.values()))}")
for _a in _sets:
    for _b in _sets:
        if _a < _b:
            print(f"    {_a:<26} vs {_b:<26} "
                  f"shared {len(_sets[_a] & _sets[_b]):>3}, "
                  f"only A {len(_sets[_a] - _sets[_b]):>3}, "
                  f"only B {len(_sets[_b] - _sets[_a]):>3}")
for _name, _df in complexes_by_contrast.items():
    assert _df.n_q.min() >= MIN_SUBUNITS, _name
    assert not _df["complex"].duplicated().any(), _name

_long = pd.concat([d.assign(contrast=n) for n, d in complexes_by_contrast.items()],
                  ignore_index=True)
_long["quantified_subunits"] = _long.quantified_subunits.map(lambda s: ";".join(sorted(s)))
_long.to_csv(f"{output_dir}/{today}_complexes_by_contrast.csv", index=False)
print(f"\nSaved -> {output_dir}/{today}_complexes_by_contrast.csv")

In [ ]:
# ============================================================
# Complex-level fold change, coherence and direction (numbers quoted in the
# Results; the gene universes _uni_store are also read by the split cells).
#
#   coherence, sign     all members share the sign of their log2FC
#   coherence, spread   observed SD of the member log2FC against the mean SD of
#                       CX_N_DRAWS matched random sets: each member replaced by a
#                       gene of the same abundance quartile and compartment class.
#                       The matching answers the objection that complexes look
#                       tight only because their members have similar abundance.
#   direction           complex median log2FC below or above zero, tested with a
#                       one-sample t test of the member log2FC against zero, BH
#                       within contrast
# ============================================================
CX_N_DRAWS   = 200
CX_AB_BINS   = 4        # quartiles; with 5 bins x 12 compartment classes, strata hold few genes
CX_MIN_POOL  = 10       # a stratum with fewer spare genes is widened; the count is printed
CX_SEED      = 0

# --- compartment class, from the UniProt table of the complex cell ---------
COMPARTMENT_RULES = [
    ("mitochondrion",   ("mitochondrion", "mitochondrial")),
    ("nucleus",         ("nucleus", "nucleolus", "nuclear ", "chromosome")),
    ("ER_Golgi",        ("endoplasmic reticulum", "golgi", "sarcoplasmic reticulum")),
    ("endolysosome",    ("lysosome", "endosome", "vacuole", "autophagosome", "phagosome")),
    ("plasma_membrane", ("cell membrane", "plasma membrane", "cell surface",
                         "cell projection", "synapse", "postsynap", "presynap")),
    ("secreted",        ("secreted", "extracellular")),
    ("cytoskeleton",    ("cytoskeleton", "microtubule", "actin", "centrosome", "spindle")),
    ("peroxisome",      ("peroxisome",)),
    ("cytoplasm",       ("cytoplasm", "cytosol")),
]

def _compartment(cc, tm):
    s = str(cc).lower()
    if s in ("", "nan"):
        return "membrane_other" if isinstance(tm, str) and tm.strip() else "unannotated"
    for nm, keys in COMPARTMENT_RULES:
        if any(k in s for k in keys):
            return nm
    return "membrane_other" if "membrane" in s else "other"

_cov = pd.read_csv(f"{CX_DIR}/uniprot_human_covariates_v2.tsv", sep="\t", dtype=str)
_cov["gene"] = _cov["Gene Names (primary)"].astype(str).str.upper().str.strip()
_cov["compartment"] = [_compartment(a, b) for a, b in
                       zip(_cov["Subcellular location [CC]"], _cov["Transmembrane"])]
GENE_COMPARTMENT = (_cov[_cov.gene.ne("") & _cov.gene.ne("NAN")]
                    .drop_duplicates("gene").set_index("gene")["compartment"].to_dict())
print(f"compartment classes for {len(GENE_COMPARTMENT)} symbols: "
      f"{pd.Series(GENE_COMPARTMENT).value_counts().to_dict()}")


def _protein_ids(res):
    for c in ("protein", "id", "Protein.Group"):
        if c in res.columns:
            return res[c].astype(str)
    return pd.Series(res.index.astype(str), index=res.index)


def _gene_universe(res, adata, gcol, conds, label):
    """One row per symbol: log2FC, mean log2 abundance, compartment, stratum.

    Abundance comes from the contrast's own samples: for the cell contrasts the
    median over the samples of the two compared groups; for the brain contrasts
    the AveExpr column of the donor-level eBayes table, the mean log2 abundance
    over the donors of the fit.

    One row per symbol, the most abundant protein, as in _rank_from_ebayes
    (GSEA). The protein with the largest |log2FC| is not used: the most extreme
    protein of a symbol is the least reliable one.
    """
    t = pd.DataFrame({
        "protein": _protein_ids(res).to_numpy(),
        "gene": res["single_gene_name"].astype(str).str.upper().str.strip().to_numpy(),
        "log2fc": pd.to_numeric(res["log2fc"], errors="coerce").to_numpy(),
    })
    if gcol is None:
        if "AveExpr" not in res.columns:
            raise KeyError(f"{label}: no group column given and no AveExpr column")
        t["abundance"] = pd.to_numeric(res["AveExpr"], errors="coerce").to_numpy()
        print(f"  {label}: abundance from AveExpr ({len(res)} rows)")
    else:
        m = adata.obs[gcol].isin(conds).to_numpy()
        if m.sum() < 4:
            raise ValueError(f"{label}: only {m.sum()} samples match {conds} "
                             f"in adata.obs[{gcol!r}]")
        ab = pd.DataFrame({
            "protein": adata.var_names.astype(str),
            "abundance": np.nanmedian(np.asarray(adata.X[m], dtype=float), axis=0)})
        n_before = len(t)
        t = t.merge(ab, on="protein", how="inner")
        if len(t) < 0.5 * n_before:
            raise ValueError(f"{label}: only {len(t)} of {n_before} result rows "
                             f"matched adata.var_names on the protein id")
        print(f"  {label}: abundance from {int(m.sum())} of {adata.n_obs} samples "
              f"({len(t)} of {n_before} rows joined)")
    t = (t.replace([np.inf, -np.inf], np.nan)
          .dropna(subset=["gene", "log2fc", "abundance"]))
    t = t[~t.gene.isin(["", "NAN", "NONE"])]
    t = (t.sort_values("abundance", ascending=False)
          .drop_duplicates("gene", keep="first").set_index("gene"))
    t["compartment"] = [GENE_COMPARTMENT.get(g, "unannotated") for g in t.index]
    t["ab_bin"] = pd.qcut(t.abundance, CX_AB_BINS, labels=False, duplicates="drop")
    t["stratum"] = t.compartment + "|" + t.ab_bin.astype(str)
    return t


def _null_sds(mat, lfc):
    return lfc[mat].std(axis=1, ddof=1)


def _n_dup_rows(mat):
    srt = np.sort(mat, axis=1)
    return int((np.diff(srt, axis=1) == 0).any(axis=1).sum())


def _draw_group(pool, m, n_draws, rng):
    """(n_draws, m) picks from pool, distinct within every row.

    Rejection sampling is not used: when many members of one complex share a
    stratum, the pool is small relative to m and a random row without a repeated
    gene becomes improbable (for 24 members drawn from a pool of 24 genes the
    probability is 24!/24^24, about 5e-10). Ranking random keys and taking the
    first m gives a draw without replacement in one step.
    """
    if m > len(pool):
        raise ValueError(f"cannot draw {m} distinct genes from a pool of {len(pool)}")
    if m == len(pool):
        return np.tile(pool, (n_draws, 1))
    return pool[np.argsort(rng.random((n_draws, len(pool))), axis=1)[:, :m]]


def _fix_cross_group(mat, groups, n_draws, rng):
    """Two widened groups can share genes; rows with a repeated gene are drawn
    again. The widened pools are large, so rejection sampling works here."""
    for _ in range(40):
        srt = np.sort(mat, axis=1)
        bad = np.flatnonzero((np.diff(srt, axis=1) == 0).any(axis=1))
        if not len(bad):
            return mat, 0
        for pool, cols_j in groups:
            mat[np.ix_(bad, cols_j)] = _draw_group(pool, len(cols_j), len(bad), rng)
    return mat, _n_dup_rows(mat)


cx_stats, _uni_store, _pool_notes = {}, {}, []
for _name, _system, _res, _ad, _gcol, _conds in CX_CONTRASTS:
    uni = _gene_universe(_res, _ad, _gcol, _conds, _name)
    _uni_store[_name] = uni
    pos = {g: i for i, g in enumerate(uni.index)}
    lfc = uni.log2fc.to_numpy(float)
    all_idx = np.arange(len(uni))
    pool_strat = {s: np.array([pos[g] for g in gg.index])
                  for s, gg in uni.groupby("stratum")}
    pool_ab = {b: np.array([pos[g] for g in gg.index])
               for b, gg in uni.groupby("ab_bin")}
    rng = np.random.default_rng(CX_SEED)

    rows, n_widened, n_members = [], 0, 0
    for _, cx in complexes_by_contrast[_name].iterrows():
        mem = sorted(cx.quantified_subunits)
        mi = np.array([pos[g] for g in mem if g in pos])
        if len(mi) < MIN_SUBUNITS:      # symbol present in the result but not the adata
            continue
        mem = [uni.index[i] for i in mi]
        k = len(mi)
        v = lfc[mi]
        obs_sd = float(v.std(ddof=1))

        # --- abundance and compartment matched null ---
        # Members are grouped by the pool they draw from, and each group is drawn
        # without replacement: the members of one stratum are the ones whose
        # draws could repeat a gene.
        strat_i = [uni.stratum.iloc[i] for i in mi]
        need = pd.Series(strat_i).value_counts()
        groups, widened_here = [], 0
        for s_key in need.index:
            cols_j = [j for j, s2 in enumerate(strat_i) if s2 == s_key]
            m_g = len(cols_j)
            avail = np.setdiff1d(pool_strat[s_key], mi)
            # The pool needs at least twice as many spare genes as members (and
            # CX_MIN_POOL), so the draws vary; otherwise the abundance bin alone,
            # then the whole universe.
            if len(avail) < max(CX_MIN_POOL, 2 * m_g):
                b = uni.ab_bin.iloc[mi[cols_j[0]]]
                avail = np.setdiff1d(pool_ab[b], mi)
                widened_here += m_g
                if len(avail) < 2 * m_g:
                    avail = np.setdiff1d(all_idx, mi)
            groups.append((avail, np.asarray(cols_j)))
        n_widened += widened_here
        n_members += k
        m_mat = np.empty((CX_N_DRAWS, k), dtype=int)
        for pool, cols_j in groups:
            m_mat[:, cols_j] = _draw_group(pool, len(cols_j), CX_N_DRAWS, rng)
        m_mat, _left = _fix_cross_group(m_mat, groups, CX_N_DRAWS, rng)
        sd_matched = float(_null_sds(m_mat, lfc).mean())

        rows.append(dict(
            complex=cx["complex"], n_q=k, members=";".join(mem),
            member_lfc=";".join(f"{x:.3f}" for x in v),
            median_lfc=float(np.median(v)), mean_lfc=float(v.mean()), sd_lfc=obs_sd,
            all_same_sign=bool(np.all(v > 0) or np.all(v < 0)),
            frac_up=float((v > 0).mean()),
            sd_null_matched=sd_matched,
            n_widened_members=widened_here,
            tight_vs_matched=obs_sd < sd_matched,
            dup_rows_left=_left))
    d = pd.DataFrame(rows)

    # --- direction: one-sample t test of the member log2FC against zero ---
    # Members share samples and normalisation, so they are not independent and
    # this p value is optimistic; the matched null is reported next to it.
    p_zero = []
    for _, r in d.iterrows():
        mi = np.array([pos[g] for g in r.members.split(";")])
        v = lfc[mi]
        p_zero.append(ttest_1samp(v, 0.0).pvalue if np.ptp(v) > 0 else 1.0)
    d["p_vs_zero"] = p_zero
    d["q_vs_zero"] = false_discovery_control(np.asarray(p_zero), method="bh")
    d["vs_zero_dir"] = np.where(d.median_lfc > 0, "enriched", "depleted")
    d["enriched"] = (d.q_vs_zero < 0.05) & (d.median_lfc > 0)
    d["depleted"] = (d.q_vs_zero < 0.05) & (d.median_lfc < 0)
    cx_stats[_name] = d
    _pool_notes.append(dict(contrast=_name, members=n_members, widened=n_widened,
                            frac_widened=round(n_widened / max(n_members, 1), 3)))
    print(f"{_name:<26} {len(d):>3} complexes, "
          f"{n_widened}/{n_members} members needed a widened stratum")

print("\n" + "=" * 78)
print("COHERENCE AND DIRECTION")
print("=" * 78)
report = []
for _name, d in cx_stats.items():
    report.append(dict(
        contrast=_name, n=len(d),
        same_sign=int(d.all_same_sign.sum()),
        same_sign_pct=round(100 * d.all_same_sign.mean(), 1),
        tight_matched_pct=round(100 * d.tight_vs_matched.mean(), 1),
        median_complex_lfc=round(d.median_lfc.median(), 3),
        proteome_median_lfc=round(float(np.median(_uni_store[_name].log2fc)), 3),
        pct_below_zero=round(100 * (d.median_lfc < 0).mean(), 1),
        enriched_vs_zero=int(d.enriched.sum()),
        depleted_vs_zero=int(d.depleted.sum())))
cx_report = pd.DataFrame(report)
print(cx_report.to_string(index=False))

# ----------------------------------------------------------------- CHECKS
for _name, d in cx_stats.items():
    assert d.sd_lfc.notna().all() and (d.sd_lfc >= 0).all(), _name
    assert d["q_vs_zero"].between(0, 1).all(), f"{_name} q_vs_zero"
    assert (d["q_vs_zero"] >= d["p_vs_zero"] - 1e-12).all(), f"{_name}: q_vs_zero below p_vs_zero"
    assert not (d.enriched & d.depleted).any(), _name
    assert d.dup_rows_left.sum() == 0, \
        f"{_name}: {int(d.dup_rows_left.sum())} null draws kept a repeated gene"
print(f"\nCHECK all contrasts: SD finite and non-negative, q >= p, "
      f"no null draw contains a repeated gene")
print(pd.DataFrame(_pool_notes).to_string(index=False))
_g = pd.concat([d.assign(contrast=n) for n, d in cx_stats.items()], ignore_index=True)
_g.to_csv(f"{output_dir}/{today}_complex_coherence.csv", index=False)
print(f"Saved -> {output_dir}/{today}_complex_coherence.csv")

In [ ]:
# Complex coherence and direction per contrast, the counts quoted in the Results:
#   tested         complexes with >= MIN_SUBUNITS quantified subunits
#   same_sign      every member has the same sign of log2FC
#   tight_vs_null  member SD below the mean SD of the matched null
#   sign_dep/enr   complex median log2FC below / above zero
#   q_dep/enr      members differ from zero at BH q < 0.05 (one-sample t test)
_rows = []
for _n, _d in cx_stats.items():
    _t = len(_d)
    _rows.append({
        "contrast": _n, "tested": _t,
        "same_sign": f"{int(_d.all_same_sign.sum())} ({100*_d.all_same_sign.mean():.0f}%)",
        "tight_vs_null": f"{int(_d.tight_vs_matched.sum())} ({100*_d.tight_vs_matched.mean():.0f}%)",
        "both": int((_d.all_same_sign & _d.tight_vs_matched).sum()),
        "median_lfc": round(_d.median_lfc.median(), 3),
        "sign_dep": int((_d.median_lfc < 0).sum()),
        "sign_enr": int((_d.median_lfc > 0).sum()),
        "q_dep": int(_d.depleted.sum()),
        "q_enr": int(_d.enriched.sum()),
    })
cx_summary = pd.DataFrame(_rows)
print(cx_summary.to_string(index=False))
cx_summary.to_csv(f"{output_dir}/{today}_complex_summary.csv", index=False)

# Display names of the five contrasts, used by the split cells
CX_LABELS = {
    "HEK_PA_vs_RC":            "HEK EA vs Rest",
    "iPSC_SAgg_vs_Soma":       "iPSC SAgg vs Soma",
    "iPSC_NAgg_vs_Neurites":   "iPSC NAgg vs Neurites",
    "Brain_Round_vs_DABpos":   "Brain Round vs DAB+",
    "Brain_Elong_vs_Neuropil": "Brain Elong vs Neuropil",
}

## 12.2 Parts of one assembly that move apart (Figure 5e)

Inputs: the Gene Ontology and its human annotations as parsed in section 4, and `data/CrossSystem/Complexes/uniprot_human_pdb.tsv`, the PDB cross-references of the reviewed human UniProt entries (https://rest.uniprot.org/uniprotkb/stream?query=reviewed:true+AND+organism_id:9606&fields=accession,gene_primary,xref_pdb&format=tsv).

In [ ]:
# ============================================================
# Candidate part pairs for Figure 5 e: two parts of one assembly whose log2FC
# could differ.
#
#   route GO   Cellular Component graph, part_of edges between complex terms
#              only. is_a is excluded: it is a subtype relation whose hubs are
#              generic ("catalytic complex", >100 children), and with it about
#              half of all pairs test significant.
#   route PDB  two Complex Portal complexes without a shared subunit, joined if
#              a PDB entry contains >= 2 subunits of each. One subunit per side
#              is not enough: B2M alone is in over 1300 structures and would join
#              every complex to every other.
#
#   The pairs do not depend on the contrast. The measurability rule, the p
#   floor and the collapses are applied per contrast in the next cells.
#   The ontology and the protein annotations are the ones parsed by the organelle
#   module cell (_terms, _term2genes): GAF rows of object type protein,
#   alternative GO IDs mapped to the main ID.
# ============================================================

CX_ROOT_COMPLEX = "GO:0032991"          # protein-containing complex
PDB_TSV = f"{CX_DIR}/uniprot_human_pdb.tsv"

# ---------------- ontology ----------------
name     = {t: d["name"] for t, d in _terms.items()}
isa      = {t: set(d["is_a"]) for t, d in _terms.items()}
partof   = {t: set(d["part_of"]) for t, d in _terms.items()}
obsolete = {t for t, d in _terms.items() if d["obsolete"]}
print(f"OBO: {len(name)} terms, {sum(d['ns'] == 'cellular_component' for d in _terms.values())} CC")

_children_isa = {}
for c, ps in isa.items():
    for p in ps:
        _children_isa.setdefault(p, set()).add(c)
complex_terms, _stack = set(), [CX_ROOT_COMPLEX]
while _stack:
    t = _stack.pop()
    for c in _children_isa.get(t, ()):
        if c not in complex_terms:
            complex_terms.add(c); _stack.append(c)
complex_terms -= obsolete
print(f"complex terms (is_a under {CX_ROOT_COMPLEX}): {len(complex_terms)}")

po_children = {}
for c, ps in partof.items():
    if c not in complex_terms:
        continue
    for p in ps:
        if p in complex_terms:
            po_children.setdefault(p, set()).add(c)
print(f"part_of edges between complex terms: {sum(len(v) for v in po_children.values())}; "
      f"parents with >= 2 children: {sum(len(v) >= 2 for v in po_children.values())}")

# ---------------- annotations ----------------
go2gene = _term2genes
print(f"GAF: {sum(len(v) for v in go2gene.values())} CC annotations, {len(go2gene)} terms")

# Genes of a part: annotations to the term and to every descendant over is_a and
# part_of, since an annotation to a descendant implies the ancestor.
_children_all = {}
for d, ps in list(isa.items()) + list(partof.items()):
    for p in ps:
        _children_all.setdefault(p, set()).add(d)
_closure_cache = {}
def _genes_with_descendants(t):
    if t in _closure_cache:
        return _closure_cache[t]
    seen, stack, out = {t}, [t], set()
    while stack:
        u = stack.pop()
        out |= go2gene.get(u, set())
        for c in _children_all.get(u, ()):
            if c not in seen:
                seen.add(c); stack.append(c)
    _closure_cache[t] = out
    return out

# ---------------- GO candidate pairs ----------------
go_pairs = []
for parent, kids in po_children.items():
    if len(kids) < 2:
        continue
    gsets = {k: _genes_with_descendants(k) for k in kids}
    for a, b in combinations(sorted(kids), 2):
        ga, gb = gsets[a], gsets[b]
        shared = ga & gb
        # Genes in both parts would be on both sides of the test and are removed
        # from both; a part left with fewer than 3 genes drops the pair.
        A, B = ga - shared, gb - shared
        if len(A) < 3 or len(B) < 3:
            continue
        go_pairs.append(dict(route="GO", assembly=f"{name.get(parent,parent)} [{parent}]",
                             part_a=name.get(a, a), part_b=name.get(b, b),
                             genes_a=A, genes_b=B, n_shared_removed=len(shared)))
print(f"\nGO route: {len(go_pairs)} candidate pairs from "
      f"{len({p['assembly'] for p in go_pairs})} parent assemblies")

# ---------------- PDB route ----------------
_pdb = pd.read_csv(PDB_TSV, sep="\t", dtype=str)
_pdb["gene"] = _pdb["Gene Names (primary)"].astype(str).str.upper().str.strip()
_pdb = _pdb[_pdb.gene.ne("") & _pdb.gene.ne("NAN") & _pdb["PDB"].notna()]
pdb2genes = {}
for g, ids in zip(_pdb.gene, _pdb["PDB"]):
    for e in str(ids).split(";"):
        e = e.strip()
        if e:
            pdb2genes.setdefault(e, set()).add(g)
print(f"PDB: {len(pdb2genes)} entries, "
      f"{len({g for v in pdb2genes.values() for g in v})} genes")

_cx = {r.base: r.subunits_u for _, r in complexes_rep.iterrows()}
_pairs_pdb = {}
for e, genes in pdb2genes.items():
    hits = [c for c, s in _cx.items() if len(s & genes) >= 2]     # >= 2 per side
    for a, b in combinations(sorted(hits), 2):
        if _cx[a] & _cx[b]:                                        # no shared subunit
            continue
        _pairs_pdb.setdefault((a, b), []).append(e)
print(f"PDB route: {len(_pairs_pdb)} raw pairs")

# --- evidence floor, applied before any test ---
# A pair needs at least PDB_MIN_ENTRIES structures with subunits of both sides.
#
# Among several partners of one complex, the choice is made at the collapse to
# one row per machine (cell above Figure 5 e), ordered by the same entry count.
# Choosing here would keep one partner per complex, and among partners tied on
# entry count the iteration order would decide.
PDB_MIN_ENTRIES = 2
_n_ent = {k: len(v) for k, v in _pairs_pdb.items()}
_keep_pdb = [k for k, v in _n_ent.items() if v >= PDB_MIN_ENTRIES]
print(f"  dropped for < {PDB_MIN_ENTRIES} PDB entries: "
      f"{len(_n_ent) - len(_keep_pdb)} of {len(_n_ent)}")

# assembly is a fixed string: the entry count is in its own column, so the key
# of a pair does not change between runs
pdb_pairs = [dict(route="PDB", assembly="PDB-coupled",
                  part_a=a, part_b=b, genes_a=_cx[a], genes_b=_cx[b],
                  n_shared_removed=0, n_pdb_entries=_n_ent[(a, b)],
                  pdb_example=sorted(_pairs_pdb[(a, b)])[0])
             for a, b in _keep_pdb]
print(f"PDB route: {len(pdb_pairs)} candidate pairs kept")
for d in go_pairs:
    d["n_pdb_entries"] = np.nan
    d["pdb_example"] = ""

cx_pair_candidates = pd.DataFrame(go_pairs + pdb_pairs)
cx_pair_candidates["n_a"] = cx_pair_candidates.genes_a.map(len)
cx_pair_candidates["n_b"] = cx_pair_candidates.genes_b.map(len)

# ----------------------------------------------------------------- CHECKS
assert (cx_pair_candidates.genes_a.map(len) >= 3).all()
assert (cx_pair_candidates.genes_b.map(len) >= 3).all()
_ov = cx_pair_candidates.apply(lambda r: len(r.genes_a & r.genes_b), axis=1)
assert (_ov == 0).all(), f"{int((_ov>0).sum())} pairs still share a gene"
print(f"\nCHECK {len(cx_pair_candidates)} candidates, no pair shares a gene, "
      f"every part has >= 3 annotated members")
print(cx_pair_candidates.route.value_counts().to_string())
print(f"CHECK is_a excluded: 'catalytic complex' GO:1902494 has "
      f"{len(_children_isa.get('GO:1902494', ()))} is_a children and "
      f"{len(po_children.get('GO:1902494', ()))} part_of children")

In [ ]:
# ============================================================
# Tier of a candidate pair of Figure 5 e, from the part names only, written by
# hand from the literature. It uses no p value, so the split test applies it
# before the near-duplicate collapse: U6 against the major pre-B particle and U6
# against the other spliceosome states are near-duplicates by union overlap, only
# one of them is kept, and the tier in the sort key keeps the tier-1 partner.
#
#   tier 1  documented as present together in one particle
#   tier 2  consecutive states of one process, or a transient association
#   tier 3  not assessed
#   confounder  a reason in the third field; the row is dropped by name before
#               the figure, with the reason recorded
# Returns (tier, note, confounder, machine name).
# ============================================================
def _tier_of(a, b):
    p = {str(a).lower(), str(b).lower()}

    def has(*subs):
        return all(any(s in x for x in p) for s in subs)

    # ---- confounded: dropped by name before the figure ----
    if has("lsm1-7"):
        return (1, "LSM1-7 shares LSM2-LSM7 with the nuclear LSM2-8 ring",
                "the >=2 subunits matched in the spliceosome structures are "
                "LSM2-8, not the cytoplasmic LSM1-7 ring",
                "LSM1-7 (confounded)")
    if has("processome") or has("rixosome"):
        return (1, "mature particle against its own biogenesis precursor",
                "the two are the same machine at two points in its assembly "
                "and never coexist", "biogenesis precursor (confounded)")
    if has("immunoproteasome"):
        return (1, "quantified set is PSMA1-7 + PSMB1-4, the constitutive core",
                "the GO core-vs-accessory row is the same comparison without "
                "the misleading name", "20S immunoproteasome (confounded)")
    if has("minor spliceosomal", "u6 small nuclear"):
        return (1, "U6 against a minor-spliceosome state",
                "the major pre-B particle is the better-supported partner for "
                "U6 and is tier 1", "U6 vs minor spliceosome (confounded)")

    # ---- tier 1: present together in one particle ----
    if has("proteasome core complex", "proteasome accessory complex"):
        return (1, "19S regulatory particle docks on the 20S core to form the "
                "26S", "", "Proteasome 19S vs 20S core")
    if has("prefoldin", "t-complex"):
        return (1, "prefoldin hands substrate directly to TRiC/CCT (6NR8)", "",
                "TRiC vs prefoldin")
    if has("complex i", "complex iii"):
        return (1, "respirasome supercomplex I+III2+IV (5XTH)", "",
                "Respirasome I vs III")
    if has("40s cytosolic", "60s cytosolic"):
        return (1, "the two subunits of one 80S ribosome", "",
                "Ribosome 40S vs 60S")
    if has("initiation factor 3", "40s cytosolic"):
        return (1, "eIF3 on the 40S is the 43S and 48S initiation complex, "
                "directly and extensively characterised", "",
                "43S initiation (eIF3 vs 40S)")
    if has("u6 small nuclear", "major spliceosomal pre-b"):
        return (1, "U6 arrives as U4/U6.U5 in the pre-B particle and the "
                "contact is resolved in pre-B structures", "",
                "U6 snRNP vs spliceosome pre-B")

    # ---- tier 2: consecutive states or transient association ----
    if has("u6 small nuclear", "spliceosomal"):
        return (2, "U6 snRNP is a constituent of successive spliceosome states",
                "", "U6 snRNP vs other spliceosome states")
    if has("nineteen core"):
        return (2, "NTC/Prp19 joins the spliceosome transiently at activation",
                "", "Spliceosome vs NTC")
    if has("survival motor neuron"):
        return (2, "SMN loads the Sm core onto snRNAs, then leaves (5XJL)", "",
                "Spliceosome vs SMN complex")
    if has("initiation factor 3"):
        return (2, "eIF3 binds the 40S; a contact with the other subunit is "
                "indirect", "", "Ribosome vs eIF3 (other subunit)")
    if has("oligosaccharyltransferase"):
        return (2, "OST contacts the ribosome through the translocon", "",
                "Ribosome vs OST (translocon)")
    if has("u1 small nuclear") or has("u4 small nuclear"):
        return (2, "snRNPs co-resident in the assembled spliceosome", "",
                "snRNP vs snRNP")
    return (3, "not assessed", "", "")

In [ ]:
# ============================================================
# Split test for Figure 5 e: do the two parts of one assembly differ in log2FC?
#
#   Filters, in this order, all before any test:
#     measurability   >= 3 quantified members per part (>= 4 to display)
#     attainability   exact two-sided Mann-Whitney p floor = 2 / C(n1+n2, n1).
#                     The largest BH threshold is 0.05, so a comparison whose
#                     floor is at or above 0.05 can never be significant and only
#                     enlarges the BH family. 3v3 = 0.100 and 3v4 = 0.057 are
#                     not testable; 3v5 = 0.036 and 4v4 = 0.029 are.
#     identical sets  comparisons whose quantified parts are the same two gene
#                     sets are one test (e.g. the 19S particle against several
#                     named 20S variants).
#     near-duplicate  across both routes, on the union of the two parts,
#                     Jaccard > CX_JACCARD_MAX; kept: lower tier, then larger,
#                     then by name. No p value enters.
#   Then Mann-Whitney, one BH family per contrast.
#
#   A row is one comparison, not one finding: one part tested against several
#   near-identical spliceosome states is several rows and one finding, so both
#   counts are printed.
# ============================================================

CX_MIN_PART, CX_DISPLAY_PART, CX_P_FLOOR_MAX = 3, 4, 0.05


def _p_floor(n1, n2):
    return 2.0 / comb(n1 + n2, n2)


def _pair_dup(r1, r2, thr):
    """Two comparisons are near-duplicates if the unions of their two parts
    overlap with Jaccard > thr.

    The union is compared, because the common case is one fixed part tested
    against several near-identical partners, which a side-against-side rule
    does not catch. The two CRL4 complexes share DDB1, CUL4A, CUL4B and RBX1
    and differ in the substrate receptor (Jaccard 0.667); joined with the 8
    COP9 subunits of the other side, the two comparisons reach 0.857 and
    collapse to one test.
    """
    u1, u2 = r1["set_a"] | r1["set_b"], r2["set_a"] | r2["set_b"]
    return len(u1 & u2) / len(u1 | u2) > thr




cx_split, _drop_log = {}, []
for _name in _uni_store:
    uni = _uni_store[_name]
    lfc = uni.log2fc
    q = set(uni.index)
    rows = []
    for _, c in cx_pair_candidates.iterrows():
        A, B = c.genes_a & q, c.genes_b & q
        if len(A) < CX_MIN_PART or len(B) < CX_MIN_PART:
            continue
        fl = _p_floor(len(A), len(B))
        if fl >= CX_P_FLOOR_MAX:
            _drop_log.append(dict(contrast=_name, reason="p floor", route=c.route,
                                  assembly=c.assembly, n_a=len(A), n_b=len(B),
                                  p_floor=round(fl, 4)))
            continue
        rows.append(dict(route=c.route, assembly=c.assembly,
                         part_a=c.part_a, part_b=c.part_b,
                         set_a=frozenset(A), set_b=frozenset(B),
                         n_a=len(A), n_b=len(B), p_floor=fl,
                         n_pdb_entries=c.get("n_pdb_entries", np.nan)))
        rows[-1].update(dict(zip(("tier", "tier_note", "confounder", "machine"),
                                 _tier_of(c.part_a, c.part_b))))
        if not rows[-1]["machine"]:
            rows[-1]["machine"] = f"{c.assembly} | {c.part_a}"
    d = pd.DataFrame(rows)
    n_measurable = len(d)
    if not n_measurable:
        cx_split[_name] = d
        print(f"{_name:<26} no testable comparison")
        continue

    # --- identical-set collapse ---
    d["key"] = [frozenset((a, b)) for a, b in zip(d.set_a, d.set_b)]
    n_before = len(d)
    d = (d.sort_values(["tier", "n_a", "n_b", "assembly"],
                       ascending=[True, False, False, True])
          .drop_duplicates("key", keep="first"))
    n_identical = n_before - len(d)

    # --- near-duplicate collapse, both routes at once, no p values involved ---
    # sorted by tier first, so a tier-1 comparison is kept over a tier-2 one
    d = d.assign(_tot=d.n_a + d.n_b).sort_values(
        ["tier", "_tot", "assembly", "part_a"],
        ascending=[True, False, True, True])
    kept = []
    for _, r in d.iterrows():
        if not any(_pair_dup(r, k, CX_JACCARD_MAX) for k in kept):
            kept.append(r)
    n_near = len(d) - len(kept)
    d = pd.DataFrame(kept).drop(columns=["_tot", "key"]).reset_index(drop=True)

    # --- test ---
    stats = []
    for _, r in d.iterrows():
        va, vb = lfc[list(r.set_a)].to_numpy(), lfc[list(r.set_b)].to_numpy()
        u = mannwhitneyu(va, vb, alternative="two-sided")
        stats.append(dict(
            p=u.pvalue,
            rank_biserial=2 * u.statistic / (len(va) * len(vb)) - 1,
            median_a=float(np.median(va)), median_b=float(np.median(vb)),
            median_diff=float(np.median(va) - np.median(vb))))
    d = pd.concat([d, pd.DataFrame(stats)], axis=1)
    d["spared"] = np.where(d.median_a >= d.median_b, d.part_a, d.part_b)
    d["lost"] = np.where(d.median_a >= d.median_b, d.part_b, d.part_a)
    d["q"] = false_discovery_control(d.p.to_numpy(), method="bh")
    d["displayable"] = (d.n_a >= CX_DISPLAY_PART) & (d.n_b >= CX_DISPLAY_PART)
    d["sig"] = d.q < 0.05
    cx_split[_name] = d.sort_values("q").reset_index(drop=True)
    _s = d[d.sig]
    print(f"{_name:<26} {n_measurable:>4} measurable -> "
          f"-{n_identical} identical -{n_near} near-dup = {len(d):>3} tests, "
          f"{len(_s)} rows at q<0.05 across {_s.spared.nunique()} findings "
          f"({int((d.sig & d.displayable).sum())} displayable rows)")

# ----------------------------------------------------------------- CHECKS
for _name, d in cx_split.items():
    if not len(d):
        continue
    assert (d.n_a >= CX_MIN_PART).all() and (d.n_b >= CX_MIN_PART).all(), _name
    assert (d.p_floor < CX_P_FLOOR_MAX).all(), _name
    assert (d.p >= d.p_floor - 1e-12).all(), \
        f"{_name}: a p value below its own exact floor"
    assert (d.q >= d.p - 1e-12).all(), _name
    assert len({frozenset((a, b)) for a, b in zip(d.set_a, d.set_b)}) == len(d), _name
    assert all(not _pair_dup(d.iloc[i], d.iloc[j], CX_JACCARD_MAX)
               for i in range(len(d)) for j in range(i)), f"{_name}: near-dup survived"
print(f"\nCHECK every p at or above its own exact floor, q >= p, "
      f"no identical or near-duplicate comparison left")
for _name, d in cx_split.items():
    if len(d):
        print(f"CHECK {_name:<26} tiers among significant rows: "
              f"{d[d.sig].tier.value_counts().sort_index().to_dict()}")
_fl = pd.DataFrame(_drop_log)
if len(_fl):
    print(f"\nDropped for an unattainable floor: {len(_fl)} "
          f"({_fl.groupby(['n_a','n_b']).size().to_dict()})")

_out = pd.concat([d.assign(contrast=n) for n, d in cx_split.items() if len(d)],
                 ignore_index=True)
for _c in ("set_a", "set_b"):
    _out[_c] = _out[_c].map(lambda s: ";".join(sorted(s)))
_out.to_csv(f"{output_dir}/{today}_complex_split_tests.csv", index=False)
print(f"Saved -> {output_dir}/{today}_complex_split_tests.csv")

print("\n=== significant rows grouped by the higher-sitting part ===")
for _name, d in cx_split.items():
    if not len(d) or not d.sig.any():
        continue
    _s = d[d.sig]
    print(f"\n{_name}: {len(_s)} rows, {_s.spared.nunique()} findings")
    for _sp, g in sorted(_s.groupby("spared"), key=lambda kv: -len(kv[1])):
        print(f"    {_sp}  ({len(g)} rows)")
        for _, r in g.iterrows():
            print(f"        vs {r.lost}   tier {r.tier}  q={r.q:.4f}  "
                  f"{r.median_a:+.2f}/{r.median_b:+.2f}  n={r.n_a}v{r.n_b}")

In [ ]:
# ============================================================
# Rows of Figure 5 e: the pairs drawn, one row per machine.
#
#   CX_TRUSTED names the two contrasts drawn in the figure, brain round
#   inclusions and iPSC somatic aggregates; the caption printed with the figure
#   lists the contrasts left out.
#
#   Tiers, confounders and machine names are the columns that the split test
#   attached from _tier_of.
#
#   Only tier 1 is drawn. Tier 2 pairs are consecutive states of one process or
#   transient associations: real contacts, but not two parts of one assembly
#   that exists at one time, so a difference between them is not a split of an
#   assembly. Within a machine the partner is chosen by structural evidence.
# ============================================================
CX_TRUSTED = ("Brain_Round_vs_DABpos", "iPSC_SAgg_vs_Soma")
CX_EXCLUDED = [c for c in cx_split if c not in CX_TRUSTED]

_missing = [c for c in CX_TRUSTED if c not in cx_split]
if _missing:
    raise KeyError(f"CX_TRUSTED names {_missing}, which the split test did not produce. "
                   f"Available: {list(cx_split)}")
for _c in CX_TRUSTED:
    _need = {"tier", "confounder", "machine"} - set(cx_split[_c].columns)
    if _need:
        raise KeyError(f"{_c} is missing {_need}")


def _pair_id(r):
    return f"{r.route}|{r.assembly}|" + "||".join(sorted([r.part_a, r.part_b]))


# --- long frame: one row per (pair, trusted contrast) --------------------
_recs = []
for _c in CX_TRUSTED:
    d = cx_split[_c]
    for _, r in d.iterrows():
        _recs.append(dict(pair_id=_pair_id(r), contrast=_c, route=r.route,
                          assembly=r.assembly, part_a=r.part_a, part_b=r.part_b,
                          set_a=r.set_a, set_b=r.set_b,
                          n_pdb=float(getattr(r, "n_pdb_entries", np.nan)),
                          tier=int(r.tier), confounder=str(r.confounder),
                          machine=str(r.machine),
                          median_a=r.median_a, median_b=r.median_b,
                          n_a=r.n_a, n_b=r.n_b, q=r.q, sig=bool(r.sig),
                          displayable=bool(r.displayable)))
cx_split_long = pd.DataFrame(_recs)
print(f"trusted contrasts {CX_TRUSTED}, excluded {CX_EXCLUDED}")
print(f"{len(cx_split_long)} (pair, contrast) results over "
      f"{cx_split_long.pair_id.nunique()} distinct pairs")

# --- tier gate: tier 1 only, confounded pairs removed ---------------------
_tier_of_pair = dict(zip(cx_split_long.pair_id, cx_split_long.tier))
_machine = dict(zip(cx_split_long.pair_id, cx_split_long.machine))
_conf = {p: c for p, c in zip(cx_split_long.pair_id, cx_split_long.confounder) if c}
_t1 = {p for p, t in _tier_of_pair.items() if t == 1}
print(f"tiers over {len(_tier_of_pair)} pairs: "
      f"{pd.Series(list(_tier_of_pair.values())).value_counts().sort_index().to_dict()}")

_keep = cx_split_long[cx_split_long.pair_id.isin(_t1)].copy()
print(f"\ntier 1: {_keep.pair_id.nunique()} pairs")
_keep["confounder"] = _keep.pair_id.map(_conf).fillna("")
_dropped_conf = sorted({(p, c) for p, c in zip(_keep.pair_id, _keep.confounder) if c})
_keep = _keep[_keep.confounder.eq("")]
print(f"after confounder drop: {_keep.pair_id.nunique()} pairs "
      f"({len(_dropped_conf)} dropped by name)")
for p, c in _dropped_conf:
    print(f"    DROPPED {p}  ->  {c}")

_sig_any = set(_keep.loc[_keep.sig, "pair_id"])
_keep = _keep[_keep.pair_id.isin(_sig_any)]
print(f"significant in >= 1 trusted contrast: {_keep.pair_id.nunique()} pairs")

# --- one row per machine ------------------------------------------------
_per_pair = (_keep.groupby(["machine", "pair_id"], as_index=False)
             .agg(n_sig=("sig", "sum"), tier=("tier", "min")))
_ev = _keep.groupby("pair_id").n_pdb.max()          # NaN for the GO route
_szs = _keep.groupby("pair_id")[["n_a", "n_b"]].max().min(axis=1)
_per_pair["evidence"] = _per_pair.pair_id.map(_ev).fillna(-1)
_per_pair["min_part"] = _per_pair.pair_id.map(_szs)
_chosen, _alts = {}, []
for m, g in _per_pair.groupby("machine"):
    # significant in both contrasts first, then the most PDB entries, then the
    # larger of the two smaller parts, then the name. No q value enters beyond
    # the count of contrasts with q < 0.05.
    g = g.sort_values(["tier", "n_sig", "evidence", "min_part", "pair_id"],
                      ascending=[True, False, False, False, True])
    _chosen[m] = g.iloc[0].pair_id
    for p in g.iloc[1:].pair_id:
        _alts.append((m, p, _chosen[m]))
_keep = _keep[_keep.pair_id.isin(set(_chosen.values()))]
print(f"one row per machine: {_keep.machine.nunique()} machines, "
      f"{len(_alts)} alternative partners set aside")
for m, p, kept in _alts:
    print(f"    {m}: kept {kept.split('||')[-1]}, also had {p.split('||')[-1]}")

# --- orientation --------------------------------------------------------
# The reference contrast is the significant one (lowest q if both are). The
# part with the higher median there is "spared" (higher in the aggregate), the
# other "lost"; the assignment is kept in the other contrast.
rows = []
for pid, g in _keep.groupby("pair_id"):
    ref = g[g.sig].nsmallest(1, "q").iloc[0]
    spared_is_a = ref.median_a >= ref.median_b
    rec = dict(pair_id=pid, machine=_machine[pid], route=ref.route,
               assembly=ref.assembly,
               spared=ref.part_a if spared_is_a else ref.part_b,
               lost=ref.part_b if spared_is_a else ref.part_a,
               ref_contrast=ref.contrast, reversed_in_other=False,
               tier=int(_tier_of_pair.get(pid, 3)))
    for _c in CX_TRUSTED:
        h = g[g.contrast == _c]
        if not len(h):
            rec[f"{_c}__measured"] = False
            continue
        h = h.iloc[0]
        ms = h.median_a if spared_is_a else h.median_b
        ml = h.median_b if spared_is_a else h.median_a
        rec.update({f"{_c}__measured": True, f"{_c}__median_spared": ms,
                    f"{_c}__median_lost": ml,
                    f"{_c}__n_spared": int(h.n_a if spared_is_a else h.n_b),
                    f"{_c}__n_lost": int(h.n_b if spared_is_a else h.n_a),
                    f"{_c}__q": float(h.q), f"{_c}__sig": bool(h.sig)})
        if _c != rec["ref_contrast"] and ml > ms:
            rec["reversed_in_other"] = True
    rec["gap"] = max(abs(rec[f"{c}__median_spared"] - rec[f"{c}__median_lost"])
                     for c in CX_TRUSTED if rec.get(f"{c}__measured"))
    rows.append(rec)
cx_split_rows = (pd.DataFrame(rows)
                 .sort_values("gap", ascending=False).reset_index(drop=True))

# ----------------------------------------------------------------- CHECKS
_src = {(_pair_id(r), c) for c in CX_TRUSTED for _, r in cx_split[c].iterrows()}
for _, r in cx_split_rows.iterrows():
    for _c in CX_TRUSTED:
        if r[f"{_c}__measured"]:
            if (r.pair_id, _c) not in _src:
                raise AssertionError(f"{r.pair_id} / {_c} does not resolve to a "
                                     f"row in cx_split; refusing to draw it")
            assert pd.notna(r[f"{_c}__median_spared"]) and \
                   pd.notna(r[f"{_c}__median_lost"]), \
                f"{r.pair_id} / {_c} is marked measured but a median is missing"
    assert any(r.get(f"{c}__sig", False) for c in CX_TRUSTED), \
        f"{r.pair_id} is drawn but significant in neither contrast"
CX_N_SIG_BOTH = int(sum(all(r.get(f"{c}__sig", False) for c in CX_TRUSTED)
                        for _, r in cx_split_rows.iterrows()))
print(f"\nCHECK {len(cx_split_rows)} rows, every pair resolves to a source row, "
      f"both medians present wherever measured, every row significant in >= 1")
print(f"CHECK significant in BOTH, recomputed from the drawn rows: {CX_N_SIG_BOTH}")
if cx_split_rows.reversed_in_other.any():
    print(f"FLAG {int(cx_split_rows.reversed_in_other.sum())} row(s) where the "
          f"non-reference contrast reverses the order:")
    for _, r in cx_split_rows[cx_split_rows.reversed_in_other].iterrows():
        print(f"    {r.spared} vs {r.lost}")

CX_CAPTION_DROPPED, CX_CAPTION_ALTS = _dropped_conf, _alts
assert (cx_split_rows.tier == 1).all(), "a non-tier-1 row reached the figure"
print(f"CHECK every drawn row is tier 1: "
      f"{cx_split_rows.tier.value_counts().to_dict()}")
print(cx_split_rows[["spared", "lost", "gap", "ref_contrast",
                     "reversed_in_other"]].to_string(index=False))

In [ ]:
# ============================================================
# Figure 5 e: one row per machine, one dumbbell per contrast, markers at the
# median log2FC of each part. Every row is a tier-1 pair (two parts documented as
# present together in one particle). Drawn by split_dumbbell
# (src/crosssystem_plots.py).
# ============================================================
if not len(cx_split_rows):
    raise ValueError("cx_split_rows is empty — nothing passed the tier-1 and "
                     "significance filters, so there is no figure to draw.")

CC = {c: SYSTEM_COLORS[CX_CONTRAST_SYSTEM[c]] for c in CX_TRUSTED}   # line colour per contrast

# --- row labels ----------------------------------------------------------
# Complex Portal names are too long for a row label: explicit short names first,
# then general rules, so a name without an entry is shortened too. The check
# below prints every name still longer than CX_NAME_MAX.
CX_NAME_MAX = 22
CX_SHORT = {
    "Chaperonin-containing T-complex":                    "TRiC/CCT",
    "Prefoldin co-chaperone complex":                     "prefoldin",
    "proteasome core complex":                            "20S core",
    "proteasome accessory complex":                       "19S cap",
    "19S proteasome regulatory complex":                  "19S cap",
    "20S immunoproteasome complex":                       "20S immuno",
    "40S cytosolic small ribosomal subunit":              "40S",
    "60S cytosolic large ribosomal subunit":              "60S",
    "Oligosaccharyltransferase complex A":                "OST-A",
    "Oligosaccharyltransferase complex B":                "OST-B",
    "Eukaryotic translation initiation factor 3 complex": "eIF3",
    "Rixosome RNA degradation complex":                   "rixosome",
    "Small ribosomal subunit processome":                  "SSU processome",
    "Mitochondrial respiratory chain complex I":           "CI",       # long forms in the
    "Mitochondrial respiratory chain complex III":         "CIII",     # figure legend text
    "Major Spliceosomal Pre-B complex":                    "Pre-B",
    "U11/U12 di-small nuclear ribonucleoprotein complex": "U11/U12 di-snRNP",
    "NineTeen Core complex":                              "NTC (Prp19)",
    "Survival motor neuron complex":                      "SMN",
    "LSM1-7":                                             "LSM1-7",
}
_NAME_RULES = [
    (r"^Major Spliceosomal ",                            "spliceosome "),
    (r"^Minor Spliceosomal ",                            "minor spliceosome "),
    (r"^Mitochondrial respiratory chain ",               ""),
    (r" small nuclear ribonucleoprotein complex$",       " snRNP"),
    (r"^Oligosaccharyltransferase complex ",             "OST-"),
    (r"[ ]+complex$",                                    ""),
]


def _short(s):
    s = str(s)
    if s in CX_SHORT:
        return CX_SHORT[s]
    for pat, rep in _NAME_RULES:
        s = re.sub(pat, rep, s)
    return s.strip()


n = len(cx_split_rows)
_labels = [f"{_short(r.spared)}\nvs {_short(r.lost)}"
           for _, r in cx_split_rows.iterrows()]

# ----------------------------------------------------------------- CHECKS
for _, r in cx_split_rows.iterrows():
    assert any(r.get(f"{c}__sig", False) for c in CX_TRUSTED)
assert (cx_split_rows.tier == 1).all(), "a non-tier-1 row reached the figure"
print(f"CHECK {n} rows drawn, all tier 1 and significant in >= 1 contrast")
_longnames = sorted({nm for _, r in cx_split_rows.iterrows()
                     for nm in (_short(r.spared), _short(r.lost))
                     if len(nm) > CX_NAME_MAX}, key=len, reverse=True)
print(f"CHECK display names over {CX_NAME_MAX} chars (add to CX_SHORT): "
      f"{_longnames if _longnames else 'none'}")

split_dumbbell(cx_split_rows, list(CX_TRUSTED), CC, CX_LABELS, CX_CONTRAST_SYSTEM,
               _labels, "e", "complex_splits", FIG5_DIR)

# --- caption, counts from the drawn rows --------------------------------
_cap = [
    f"Parts of {n} assemblies that move apart, in "
    f"{' and '.join(CX_LABELS.get(c, c) for c in CX_TRUSTED)}. Each row is one "
    f"machine and every pair is documented as simultaneously present in one "
    f"particle; the two dumbbells are the two contrasts, markers at the median "
    f"log2 fold change of each part's quantified members, joined by a line. "
    f"Rows are oriented by the part enriched in the aggregate in the contrast "
    f"where the pair is significant, and that orientation is carried into the "
    f"other contrast unchanged. {CX_N_SIG_BOTH} of {n} are significant in both "
    f"contrasts.",
    f"Not shown: {', '.join(CX_LABELS.get(c, c) for c in CX_EXCLUDED)}.",
]
if CX_CAPTION_ALTS:
    _cap.append("Where a part had several partners the best-supported one is "
                "shown: "
                + "; ".join(f"{m} -> {_short(k.split('||')[-1])}"
                            for m, _p, k in CX_CAPTION_ALTS) + ".")
if CX_CAPTION_DROPPED:
    _cap.append("Dropped for a confounder: "
                + "; ".join(f"{p.split('|')[1]} ({c})"
                            for p, c in CX_CAPTION_DROPPED) + ".")
if cx_split_rows.reversed_in_other.any():
    _cap.append("Order reverses in the non-reference contrast for: "
                + "; ".join(f"{_short(r.spared)} vs {_short(r.lost)}" for _, r in
                            cx_split_rows[cx_split_rows.reversed_in_other].iterrows())
                + ".")
print("\n=== CAPTION ===\n" + "\n".join(_cap))
plt.show()

## 12.3 Physicochemical properties against fold change (Figure S8n)

Inputs in `data/CrossSystem/Physicochemical_Properties/`: `uniprot_human_sequences.tsv`, the sequences of the reviewed human UniProt entries, and `camsol_all_quantified_proteins_n7170.txt`, the scores of the CamSol web server for the sequences of the quantified proteins (`all_quantified_proteins.fasta`).

In [ ]:
# ============================================================
# Physicochemical properties per gene symbol (Figure 5 f, Figure S8 n).
#
#   CamSol solubility from the output of the CamSol web server, sequence
#   descriptors computed here from the longest UniProt sequence of each symbol,
#   transmembrane-helix count from the UniProt feature string.
#
#   Every property is one number per gene symbol, so the four levels of the next
#   cell and the split heatmap use the same values.
# ============================================================


# --- CamSol: only the score column, the profile column is ~90 MB ---------
_cam = pd.read_csv(f"{PP_DIR}/camsol_all_quantified_proteins_n7170.txt", sep="\t",
                   usecols=["Name", "protein variant score"])
_cam["accession"] = _cam.Name.str.split("_").str[0]
_cam["gene"] = _cam.Name.str.split(" ").str[0].str.split("_").str[1]
_cam = _cam.rename(columns={"protein variant score": "camsol"})
print(f"CamSol: {len(_cam)} proteins, score range "
      f"{_cam.camsol.min():.2f} to {_cam.camsol.max():.2f}")

# --- sequences ----------------------------------------------------------
_seq = pd.read_csv(f"{PP_DIR}/uniprot_human_sequences.tsv", sep="\t", dtype=str)
_seq["gene"] = _seq["Gene Names (primary)"].astype(str).str.upper().str.strip()
_seq = _seq[_seq.gene.ne("") & _seq.gene.ne("NAN") & _seq.Sequence.notna()]
_seq = _seq.sort_values("Length", key=lambda s: s.astype(int), ascending=False) \
           .drop_duplicates("gene")
print(f"sequences: {len(_seq)} gene symbols")

# --- transmembrane-helix count -----------------------------------------
_tm = pd.read_csv(f"{CX_DIR}/uniprot_human_covariates_v2.tsv", sep="\t", dtype=str)
_tm["gene"] = _tm["Gene Names (primary)"].astype(str).str.upper().str.strip()
_tm["n_tm"] = _tm["Transmembrane"].fillna("").str.count("TRANSMEM")
_tm = _tm[_tm.gene.ne("") & _tm.gene.ne("NAN")].drop_duplicates("gene")
print(f"transmembrane: {int((_tm.n_tm > 0).sum())} of {len(_tm)} genes have "
      f">= 1 helix, max {int(_tm.n_tm.max())}")

# --- sequence descriptors ----------------------------------------------
KD = {"A": 1.8, "R": -4.5, "N": -3.5, "D": -3.5, "C": 2.5, "Q": -3.5, "E": -3.5,
      "G": -0.4, "H": -3.2, "I": 4.5, "L": 3.8, "K": -3.9, "M": 1.9, "F": 2.8,
      "P": -1.6, "S": -0.8, "T": -0.7, "W": -0.9, "Y": -1.3, "V": 4.2}
# Uversky order-promoting set; higher fraction = more ordered
ORDER_AA = set("WCFIYVLN")
AROM = set("FWY")
POS, NEG = set("KR"), set("DE")
BLOB = 25


def _descriptors(s):
    s = re.sub(r"[^ACDEFGHIKLMNPQRSTVWY]", "", s.upper())
    n = len(s)
    if n < BLOB:
        return None
    a = np.frombuffer(s.encode(), dtype="S1")
    gravy = float(np.mean([KD[c] for c in s]))
    pos = np.isin(a, [c.encode() for c in POS])
    neg = np.isin(a, [c.encode() for c in NEG])
    ncpr = float((pos.sum() - neg.sum()) / n)
    fcr = float((pos.sum() + neg.sum()) / n)
    # charge blockiness: mean absolute net charge per residue in 25-residue
    # windows minus the absolute net charge per residue of the whole protein.
    # Not the kappa parameter of Das and Pappu, which is normalised differently.
    q = pos.astype(float) - neg.astype(float)
    w = np.convolve(q, np.ones(BLOB), mode="valid") / BLOB
    blocky = float(np.mean(np.abs(w)) - abs(ncpr))
    return dict(
        gravy=gravy,
        net_charge_per_res=ncpr,
        frac_charged=fcr,
        charge_blockiness=blocky,
        order_propensity=float(np.mean([c in ORDER_AA for c in s])),
        aromaticity=float(np.mean([c in AROM for c in s])),
        log10_length=float(np.log10(n)),
    )


_rows = []
for g, sq in zip(_seq.gene, _seq.Sequence):
    d = _descriptors(sq)
    if d:
        _rows.append(dict(gene=g, **d))
_desc = pd.DataFrame(_rows)
print(f"descriptors: {len(_desc)} genes ({len(_seq) - len(_desc)} sequences "
      f"shorter than the {BLOB}-residue window, dropped)")

# --- assemble -----------------------------------------------------------
prop_table = (_desc
              .merge(_cam.groupby("gene", as_index=False).camsol.mean(), on="gene",
                     how="left")
              .merge(_tm[["gene", "n_tm"]], on="gene", how="left")
              .set_index("gene"))
PROPERTIES = ["camsol", "gravy", "order_propensity", "charge_blockiness",
              "frac_charged", "net_charge_per_res", "aromaticity",
              "log10_length", "n_tm"]
PROPERTY_LABEL = {"camsol": "CamSol solubility", "gravy": "hydrophobicity (GRAVY)",
                  "order_propensity": "order propensity",
                  "charge_blockiness": "charge blockiness",
                  "frac_charged": "fraction charged",
                  "net_charge_per_res": "net charge / residue",
                  "aromaticity": "aromaticity", "log10_length": "log10 length",
                  "n_tm": "TM helices"}

# ----------------------------------------------------------------- CHECKS
print(f"\nprop_table: {len(prop_table)} genes x {len(PROPERTIES)} properties")
print(prop_table[PROPERTIES].notna().sum().to_frame("n_with_value").to_string())
assert prop_table.index.is_unique
assert set(PROPERTIES) <= set(prop_table.columns)
assert set(PROPERTIES) == set(PROPERTY_LABEL)
# Sign check of the descriptors: hydrophobicity and the number of TM helices
# correlate positively across the proteome.
_c = prop_table[["gravy", "n_tm"]].dropna()
_r = spearmanr(_c.gravy, _c.n_tm)
print(f"CHECK sign control, GRAVY vs TM-helix count: rho = {_r.statistic:+.3f} "
      f"(p = {_r.pvalue:.1e}, n = {len(_c)}) — must be clearly positive")
assert _r.statistic > 0.2, "GRAVY and TM count disagree; check the descriptors"
_c2 = prop_table[["camsol", "gravy"]].dropna()
print(f"CHECK CamSol vs GRAVY: rho = {spearmanr(_c2.camsol, _c2.gravy).statistic:+.3f} "
      f"(expected negative: hydrophobic proteins score as less soluble)")
prop_table.to_csv(f"{output_dir}/{today}_protein_properties.csv")
print(f"Saved -> {output_dir}/{today}_protein_properties.csv")

In [ ]:
# ============================================================
# Rank correlation of each property with the fold change at four levels of
# aggregation (Figure S8 n; rho quoted in the Results).
#
#   L1 subunits within their own complex. Per complex, rank both vectors
#      across that complex's members, standardise each, sum the products and
#      divide by n-1, which is that complex's own Spearman rho; average over
#      complexes. Each complex is its own control, so differences between
#      complexes do not contribute, and the statistic is bounded like L2 to L4.
#   L2 proteins. Spearman over every quantified protein.
#   L3 complexes. Mean property of the members against median fold change
#      of the members, Spearman across complexes.
#   L4 extreme complexes. The same rho inside the top and bottom quintile
#      of complex fold change only.
#
#   L1 null: fold changes permuted within each complex. L2-L4 null: the fold
#   changes permuted. PP_N_PERM draws, two-sided empirical p.
#   BH within each level and contrast, not pooled across levels.
#
#   All four levels use a rank correlation. An AUC is not used: a raw AUC below
#   0.5 and a model AUC constrained above 0.5 are different quantities and are
#   not comparable between levels.
# ============================================================

PP_N_PERM = 10_000
PP_MIN_MEMBERS = 4        # per complex, per property
PP_MIN_COMPLEXES = 20     # else the property is not computable at that level
PP_MIN_PROTEINS = 200
PP_SEED = 0
PP_LEVELS = ["L1 subunits", "L2 proteins", "L3 complexes", "L4 extremes"]


def _z(x):
    x = np.asarray(x, float)
    s = x.std(ddof=1)
    return (x - x.mean()) / s if s > 0 else np.zeros_like(x)


def _perm_p(obs, null):
    """Two-sided empirical p with the observed value included, so p is never 0."""
    null = np.asarray(null, float)
    return (1 + int((np.abs(null) >= abs(obs) - 1e-12).sum())) / (1 + len(null))


def _rho_null(zp, zf, B, rng, chunk=1000):
    """Null of (1/n) * zp . permuted(zf), which is Spearman when both are
    standardised ranks; the fold changes (zf) are permuted."""
    n = len(zp)
    out = np.empty(B)
    for i in range(0, B, chunk):
        b = min(chunk, B - i)
        M = rng.permuted(np.tile(zf, (b, 1)), axis=1)
        out[i:i + b] = M @ zp / n
    return out


def _rho(zp, zf):
    return float(zp @ zf / len(zp))


pp_results, _t0 = [], time.time()
for _c in _uni_store:
    lfc_all = _uni_store[_c].log2fc
    cx = complexes_by_contrast[_c]
    # members sorted: the order of a set changes with every Python process, and
    # the member order sets the permutation draws of the L1 null
    members = {r["complex"]: [g for g in sorted(r.quantified_subunits) if g in lfc_all.index]
               for _, r in cx.iterrows()}
    for _p in PROPERTIES:
        pv_all = prop_table[_p]
        rng = np.random.default_rng(PP_SEED)

        # ---------- L1: within complex ----------
        per_cx = []
        for name, mem in members.items():
            g = [x for x in mem if x in pv_all.index and np.isfinite(pv_all.get(x, np.nan))]
            if len(g) < PP_MIN_MEMBERS:
                continue
            zp = _z(rankdata(pv_all.loc[g].to_numpy()))
            zf = _z(rankdata(lfc_all.loc[g].to_numpy()))
            if zp.std() == 0 or zf.std() == 0:
                continue
            per_cx.append((zp, zf))
        if len(per_cx) >= PP_MIN_COMPLEXES:
            # Divided by n-1, the standardised sum of products is the complex's
            # Spearman rho, bounded at +-1 like L2 to L4. Undivided, it grows
            # with complex size and is not comparable with the other levels.
            obs1 = float(np.mean([zp @ zf / (len(zp) - 1) for zp, zf in per_cx]))
            null1 = np.zeros(PP_N_PERM)
            for zp, zf in per_cx:
                M = rng.permuted(np.tile(zf, (PP_N_PERM, 1)), axis=1)
                null1 += (M @ zp) / (len(zp) - 1)
            null1 /= len(per_cx)
            pp_results.append(dict(contrast=_c, level=PP_LEVELS[0], prop=_p,
                                   stat=obs1, n=len(per_cx),
                                   p=_perm_p(obs1, null1)))

        # ---------- L2: proteins ----------
        d = pd.DataFrame({"p": pv_all, "f": lfc_all}).dropna()
        if len(d) >= PP_MIN_PROTEINS:
            zp, zf = _z(rankdata(d.p)), _z(rankdata(d.f))
            r2 = _rho(zp, zf)
            pp_results.append(dict(contrast=_c, level=PP_LEVELS[1], prop=_p,
                                   stat=r2, n=len(d),
                                   p=_perm_p(r2, _rho_null(zp, zf, PP_N_PERM, rng))))

        # ---------- L3 / L4: complexes ----------
        rows = []
        for name, mem in members.items():
            g = [x for x in mem if x in pv_all.index and np.isfinite(pv_all.get(x, np.nan))]
            if len(g) < PP_MIN_MEMBERS:
                continue
            rows.append((float(pv_all.loc[g].mean()),
                         float(np.median(lfc_all.loc[g].to_numpy()))))
        if len(rows) >= PP_MIN_COMPLEXES:
            P, F = map(np.asarray, zip(*rows))
            zp, zf = _z(rankdata(P)), _z(rankdata(F))
            r3 = _rho(zp, zf)
            pp_results.append(dict(contrast=_c, level=PP_LEVELS[2], prop=_p,
                                   stat=r3, n=len(P),
                                   p=_perm_p(r3, _rho_null(zp, zf, PP_N_PERM, rng))))
            lo, hi = np.quantile(F, [0.2, 0.8])
            m = (F <= lo) | (F >= hi)
            if m.sum() >= PP_MIN_COMPLEXES:
                zp4, zf4 = _z(rankdata(P[m])), _z(rankdata(F[m]))
                r4 = _rho(zp4, zf4)
                pp_results.append(dict(contrast=_c, level=PP_LEVELS[3], prop=_p,
                                       stat=r4, n=int(m.sum()),
                                       p=_perm_p(r4, _rho_null(zp4, zf4, PP_N_PERM, rng))))
pp = pd.DataFrame(pp_results)
print(f"{len(pp)} (contrast, level, property) statistics in {time.time() - _t0:.0f}s")

# --- BH within each level and contrast separately -----------------------
pp["q"] = np.nan
for (_c, _l), g in pp.groupby(["contrast", "level"]):
    pp.loc[g.index, "q"] = false_discovery_control(g.p.to_numpy(), method="bh")

# --- keep only properties computable at every level, in every contrast ---
# A property missing at one level is dropped from all panels.
_cov = (pp.groupby(["prop", "contrast"])["level"].nunique()
        .unstack(fill_value=0).min(axis=1))
PP_KEEP = [p for p in PROPERTIES if _cov.get(p, 0) == len(PP_LEVELS)]
_dropped = [p for p in PROPERTIES if p not in PP_KEEP]
print(f"properties computable at all {len(PP_LEVELS)} levels in every contrast: "
      f"{len(PP_KEEP)} of {len(PROPERTIES)}")
if _dropped:
    for p in _dropped:
        _lv = pp[pp.prop == p].groupby("contrast")["level"].nunique()
        print(f"    DROPPED {p}: levels present per contrast {_lv.to_dict()}")
pp = pp[pp.prop.isin(PP_KEEP)].reset_index(drop=True)

# ----------------------------------------------------------------- CHECKS
assert pp.stat.between(-1, 1).all(), \
    "a statistic outside [-1, 1]; every level must be a rank correlation"
assert (pp.q >= pp.p - 1e-12).all(), "a q below its own p"
_shape = pp.groupby(["contrast", "level"])["prop"].apply(lambda s: tuple(sorted(s)))
assert _shape.nunique() == 1, "the property set differs between panels"
print(f"CHECK rho within [-1, 1] at every level, q >= p everywhere, and the same "
      f"{len(PP_KEEP)} properties in every (contrast, level) cell")
print(f"CHECK all four levels are rank correlations in [-1, 1]. Per-level range:")
for _l in PP_LEVELS:
    _g = pp[pp.level == _l].stat
    print(f"    {_l:<14} {_g.min():+.3f} to {_g.max():+.3f}")
pp.to_csv(f"{output_dir}/{today}_property_four_levels.csv", index=False)
print(f"Saved -> {output_dir}/{today}_property_four_levels.csv")
for _c in _uni_store:
    print(f"\n--- {_c} ---")
    print(pp[pp.contrast == _c].pivot(index="prop", columns="level",
                                      values="stat").round(3).to_string())

In [ ]:
# ============================================================
# Figure S8 n: property against fold change at the four levels of aggregation,
# one subplot per level, the two contrasts of Figure 5 e.
#
#   Same properties, row order, x limits and ticks in all four subplots. All four
#   statistics are rank correlations bounded at +-1. The row order is set once,
#   by |L3 rho| in the first contrast of CX_TRUSTED, and reused.
#
#   Marker fill is the only encoding of significance (q < 0.05).
#
#   Orientation: positive means a higher property value goes with enrichment in
#   the aggregate. CamSol is a solubility score, so a negative value there means
#   the poorly soluble proteins are the enriched ones.
# Drawn by property_levels (src/crosssystem_plots.py).
# ============================================================
PP_PRIMARY = CX_TRUSTED[0]
CC = {c: SYSTEM_COLORS[CX_CONTRAST_SYSTEM[c]] for c in CX_TRUSTED}

_d = pp[pp.contrast.isin(CX_TRUSTED)].copy()
_ord = (_d[(_d.contrast == PP_PRIMARY) & (_d.level == PP_LEVELS[2])]
        .assign(a=lambda x: x.stat.abs()).sort_values("a", ascending=False))
PP_ORDER = list(_ord["prop"])
assert set(PP_ORDER) == set(PP_KEEP), "row order does not cover every property"
assert _d.stat.between(-1, 1).all() and (_d.q >= _d.p - 1e-12).all()
n = len(PP_ORDER)

property_levels(pp, PP_LEVELS, list(CX_TRUSTED), PP_ORDER, PROPERTY_LABEL, CC,
                CX_LABELS, "n", "property_four_levels", SI8_DIR)

_not_shown = [PROPERTY_LABEL[p] for p in PROPERTIES if p not in PP_KEEP]
print(f"\n=== CAPTION ===\nProperty against aggregate-vs-host fold change at four "
      f"levels of aggregation, in {' and '.join(CX_LABELS.get(c, c) for c in CX_TRUSTED)}. "
      f"Rows are the {n} properties computable at every level, ordered once by the "
      f"absolute complex-level correlation in {CX_LABELS.get(PP_PRIMARY, PP_PRIMARY)} "
      f"and reused in all four panels. All four are rank correlations bounded at "
      f"+-1: L1 is the within-complex Spearman correlation averaged over "
      f"complexes, so each complex is its own control; L2 to L4 are Spearman "
      f"correlations across proteins, complexes and extreme complexes. "
      f"Positive means a HIGHER "
      f"property value goes with enrichment in the aggregate. CamSol is a "
      f"solubility score, so a negative correlation there means the poorly soluble "
      f"proteins are the enriched ones. Filled marker: BH q < 0.05 within that "
      f"level and contrast, corrected per level and contrast separately, "
      f"{PP_N_PERM:,} permutations; the legend states the same rule. "
      f"Not shown, because not computable at every level: "
      f"{', '.join(_not_shown) if _not_shown else 'none'}.")
plt.show()

## 12.4 Physicochemical difference between the split parts (Figure 5f)

In [ ]:
# ============================================================
# Figure 5 f: physicochemical difference between the two parts of each pair.
#
#   The rows are the rows of Figure 5 e, in the same order; every row is a
#   tier-1 pair.
#
#   One panel for both contrasts. The difference between two parts is computed
#   from the sequences of their members, which are the same in both systems; two
#   panels would differ only by the few proteins quantified in one system and
#   not the other, and would show the same numbers twice. Significance and which
#   part is higher are specific to the contrast and shown in Figure 5 e.
#
#   Members are the union of the proteins quantified in either contrast of
#   CX_TRUSTED, so each part is represented by every measured member.
#
#   Oriented delta = (mean property of the part drawn higher in Figure 5 e,
#   minus that of the other part) times the property's expected sign, divided by
#   the property's SD across all quantified proteins. The SD is not taken down
#   the column of the figure, so the colour does not depend on which rows are
#   shown; colour and glyph come from the same number.
#
#   The panel is 2 x 1 units; the row label is one line, because two lines need
#   about twice the row height. Drawn by split_property_heatmap
#   (src/crosssystem_plots.py).
# ============================================================
HT_MIN_MEMBERS = 4
# Six properties, fixed in advance: two hydrophobicity-and-order terms, one
# solubility term, three charge terms. With more columns a cell is too narrow
# for its glyph and header.
HT_PROPS = ["gravy", "order_propensity", "camsol",
            "frac_charged", "abs_net_charge", "charge_blockiness"]
HT_EXPECTED = {"gravy": +1, "order_propensity": +1, "camsol": -1,
               "frac_charged": -1, "abs_net_charge": -1, "charge_blockiness": -1}
HT_HEADER = {"gravy": "hydro-\nphobicity", "order_propensity": "order\npropensity",
             "camsol": "CamSol\nsolubility", "frac_charged": "fraction\ncharged",
             "abs_net_charge": "abs. net\ncharge",
             "charge_blockiness": "charge\nblockiness"}
HT_CBAR_LABEL = ("+ = the property predicts the retained part\n"
                 "difference relative to its spread across all proteins")

# |net charge| in place of the signed net charge: the proteome-wide expectation
# is "less charged", a statement about magnitude; the signed value has no
# expected sign.
_pt = prop_table.copy()
_pt["abs_net_charge"] = _pt["net_charge_per_res"].abs()
assert set(HT_PROPS) == set(HT_EXPECTED) == set(HT_HEADER)
assert set(HT_PROPS) <= set(_pt.columns)

_quant = set().union(*[set(_uni_store[c].index) for c in CX_TRUSTED])
_sd = _pt.loc[_pt.index.intersection(_quant), HT_PROPS].std(ddof=1)
print("SD across all quantified proteins, the unit for every column:")
print(_sd.to_frame("sd").to_string())


def _find_pair(contrast, a, b):
    """The tested row for this part pair in this contrast, or None."""
    d = cx_split[contrast]
    m = ((d.part_a == a) & (d.part_b == b)) | ((d.part_a == b) & (d.part_b == a))
    return d[m].iloc[0] if m.any() else None


# --- one row per cx_split_rows row, members pooled over the contrasts ---
_cells, _sizes, _n_blank = {}, {}, 0
for i, r in cx_split_rows.iterrows():
    hi, lo = set(), set()
    for c in CX_TRUSTED:
        src = _find_pair(c, r.spared, r.lost)
        if src is None:
            continue
        # orientation as in cx_split_rows, not recomputed
        hi |= set(src.set_a if src.part_a == r.spared else src.set_b)
        lo |= set(src.set_b if src.part_a == r.spared else src.set_a)
    _sizes[i] = (len(hi), len(lo))
    for p in HT_PROPS:
        vh = _pt[p].reindex(sorted(hi)).dropna()
        vl = _pt[p].reindex(sorted(lo)).dropna()
        if len(vh) < HT_MIN_MEMBERS or len(vl) < HT_MIN_MEMBERS:
            _cells[(i, p)] = np.nan
            _n_blank += 1
        else:
            _cells[(i, p)] = (vh.mean() - vl.mean()) * HT_EXPECTED[p] / _sd[p]
n = len(cx_split_rows)
print(f"\n{n} rows x {len(HT_PROPS)} properties, {_n_blank} blank "
      f"(a part with < {HT_MIN_MEMBERS} members carrying that property)")
print(pd.DataFrame([dict(row=f"{r.spared} vs {r.lost}",
                         n_higher=_sizes[i][0], n_lower=_sizes[i][1])
                    for i, r in cx_split_rows.iterrows()]).to_string(index=False))

_labels = [f"{_short(r.spared)} vs {_short(r.lost)}"   # one line, see the header
           for _, r in cx_split_rows.iterrows()]
values = pd.DataFrame([[_cells[(i, p)] for p in HT_PROPS] for i in range(n)],
                      columns=HT_PROPS)

# ----------------------------------------------------------------- CHECKS
assert sum(1 for v in _cells.values() if not np.isfinite(v)) == _n_blank
print(f"CHECK {_n_blank} blank cells of {len(_cells)}, each a part with "
      f"< {HT_MIN_MEMBERS} members carrying that property")
for i, r in cx_split_rows.iterrows():
    assert _sizes[i][0] >= 1 and _sizes[i][1] >= 1, f"row {i} has an empty part"
print(f"CHECK member counts per part: higher "
      f"{min(v[0] for v in _sizes.values())}-{max(v[0] for v in _sizes.values())}, "
      f"lower {min(v[1] for v in _sizes.values())}-{max(v[1] for v in _sizes.values())}")
# pairs per property with a positive oriented delta (the prediction holds)
_agree = pd.DataFrame({"k": (values > 0).sum(), "n": values.notna().sum()}).reindex(HT_PROPS)
_agree["pct"] = (100 * _agree.k / _agree.n).round(0)
print("CHECK agreement per property, from the drawn cells:")
print(_agree.to_string())

split_property_heatmap(values, _labels, HT_HEADER, HT_CBAR_LABEL,
                       "f", "split_property_heatmap", FIG5_DIR)

_exp = ", ".join(f"{PROPERTY_LABEL.get(p, p)} {'+' if HT_EXPECTED[p] > 0 else '-'}"
                 for p in HT_PROPS)
print(f"\n=== CAPTION ===\nPhysicochemical difference between the two parts of "
      f"each pair, for the same {n} pairs and in the same order as the dumbbell "
      f"figure. Every pair is documented as simultaneously present in one "
      f"particle. Six properties, fixed in advance as two "
      f"hydrophobicity-and-order terms, one solubility term and three charge "
      f"terms. Each difference is the mean over the members of the part that "
      f"sits higher on the aggregate axis minus the mean over the other part, "
      f"multiplied by that property's expected sign ({_exp}) so that a positive "
      f"value always means the pair points the way the proteome-wide axis "
      f"predicts, and divided by the property's SD across all quantified "
      f"proteins rather than across the rows shown. The glyph carries the claim "
      f"and the colour only its magnitude, both from the same number. "
      f"A single panel: the difference is computed from the member sequences and "
      f"is therefore common to both systems, and members are pooled over the "
      f"proteins quantified in either contrast. Which comparisons are "
      f"significant and which part sits higher is system specific and is shown "
      f"in the dumbbell figure. Blank cells are pairs where one part has fewer "
      f"than {HT_MIN_MEMBERS} members carrying that property "
      f"({_n_blank} of {len(_cells)}).")
plt.show()

# 13. Organelle modules (Figure 5d, Figure S8 a–m)

The modules are built in section 4.

In [ ]:
# ============================================================
# Organelle module scores per sample, module pairs and the eBayes test
# (Figure 5 d, Figure S8 a-m; the count of called pairs is quoted in the Results).
#
#   The unit of replication is the sample in the cell models and the donor in
#   brain. A protein-level test of one module against another would be
#   pseudoreplicated: the members of one module are in one structure and, in
#   brain, come from the same donors.
#
#   All pairs of all organelles form one matrix per contrast, so eBayes moderates
#   the variance over all pairs; with nine iPSC samples per contrast the variance
#   of one pair alone is poorly estimated.
#
#   The bar and the test come from this matrix and the same members; no result
#   table of the contrast tests is read.
#
#   Each protein is centred on its own mean over the samples of this contrast
#   before a module score is taken. A module score is the mean over the members
#   that one sample quantified, and members differ in absolute level, so without
#   the centring a sample that quantified more low-abundance members would score
#   lower although no protein changed.
#
#   delta_module is the group difference of the per-sample score, the quantity
#   the test fits: the pair feature is score_a - score_b and the limma
#   coefficient is the group-mean difference, so bar_a - bar_b equals the tested
#   delta (assertion below, to 1e-8). delta_module_permember is the robustness
#   column: per member the aggregate mean minus the host mean, averaged over the
#   members; each protein is its own control, so it does not depend on which
#   members a sample detected. Modules where the two differ by more than
#   ORG_SHIFT_WARN are printed.
#
#   Brain: many samples from five donors. Every quantity is first reduced to one
#   value per donor, the mean over that donor's aggregate samples minus the mean
#   over its host samples. The bar is the mean of the donor values and the test
#   an intercept-only limma over the donors, as in the region-paired fit of the
#   3_BrainTissue notebook.
# ============================================================
ORG_MIN_FRAC = 0.5     # a member must be quantified in this fraction of both groups
ORG_MIN_DELTA = 0.30   # a pair is called only if it also differs by this much in
                       # log2: many pairs reach FDR < 0.05 with differences of a
                       # few hundredths of a log2 unit. 0.3 is the threshold of
                       # the cross-system gene classification, so one effect size
                       # applies throughout.
                       # This is a reporting rule: the p value tests a difference
                       # of zero, not a difference above 0.3. The test of the
                       # threshold, TREAT, is not in inmoose, which provides
                       # eBayes and topTable only.

# Aggregate group first, host group second. `extra` restricts the samples
# further (brain: disease only). `donor` reduces the samples to one value per
# donor.
ORG_CONTRASTS = [
    dict(name="HEK", system="HEK", label="HEK EA vs RC",
         adata=adata_log_autoph, class_col=biological_autoph,
         agg=["-BafA1 PFF Pure Aggregates"], host=["-BafA1 PFF Rest Cells"],
         extra={}, donor=None),
    dict(name="iPSC_somatic", system="iPSC", label="iPSC somatic",
         adata=adata_strict_no_contaminants_ipsc, class_col=biological_ipsc,
         agg=["iDA D56 PFF N2 Aggregates Somatic"],
         host=["iDA D56 PFF N2 Soma Without Aggregates"],
         extra={}, donor=None),
    dict(name="iPSC_neuritic", system="iPSC", label="iPSC neuritic",
         adata=adata_strict_no_contaminants_ipsc, class_col=biological_ipsc,
         agg=["iDA D56 PFF N2 Aggregates Neuritic"],
         host=["iDA D56 PFF N2 Neurites Without Aggregates"],
         extra={}, donor=None),
    dict(name="Brain", system="Brain", label="brain EA vs DAB+",
         adata=adata_log_brain, class_col="fraction",
         agg=["Round Aggregate"], host=["Neuron | DAB Positive"],
         extra={"disease_binary": "Disease"}, donor="Brain"),
]
ORG_ORDER = [c["name"] for c in ORG_CONTRASTS]


def _org_mask(adata, col, values, extra):
    if col not in adata.obs.columns:
        raise KeyError(f"{col!r} is not in adata.obs. Columns: "
                       f"{list(adata.obs.columns)[:25]}")
    m = adata.obs[col].astype(str).isin([str(v) for v in values]).to_numpy()
    for k, v in extra.items():
        m &= (adata.obs[k].astype(str) == str(v)).to_numpy()
    if m.sum() == 0:
        raise ValueError(f"no samples for {values} in {col!r}. Values present: "
                         f"{sorted(adata.obs[col].astype(str).unique())[:25]}")
    return m


org_bars, org_pair_rows, org_scores = [], [], {}
for _c in ORG_CONTRASTS:
    _ad = _c["adata"]
    _ma = _org_mask(_ad, _c["class_col"], _c["agg"], _c["extra"])
    _mb = _org_mask(_ad, _c["class_col"], _c["host"], _c["extra"])
    _X = np.asarray(_ad.X, dtype=float)
    assert np.nanmax(_X) < 100, (f"{_c['name']}: max value {np.nanmax(_X):.1f}; "
                                 f"this matrix does not look log2-transformed")
    _sym = _ad.var["single_gene_name"].astype(str).str.upper().str.strip()
    _na, _nb = int(_ma.sum()), int(_mb.sum())
    _min_a, _min_b = int(np.ceil(ORG_MIN_FRAC * _na)), int(np.ceil(ORG_MIN_FRAC * _nb))
    _keep = _ma | _mb
    _dv = (_ad.obs[_c["donor"]].astype(str).to_numpy() if _c["donor"]
           else np.zeros(_ad.n_obs, dtype=object))

    # every protein centred on its mean over this contrast's samples (header)
    with np.errstate(invalid="ignore"):
        _lvl = np.nanmean(_X[_keep], axis=0)
    _Z = _X - _lvl[None, :]

    # brain: aggregate mean minus host mean within each donor, one row per donor;
    # the masked means skip missing values
    _donors = sorted(set(_dv[_ma]) & set(_dv[_mb]))

    def _donor_diff(v):
        v = np.ma.masked_invalid(np.asarray(v, dtype=float))
        return np.ma.vstack([v[_ma & (_dv == u)].mean(axis=0)
                             - v[_mb & (_dv == u)].mean(axis=0) for u in _donors])

    # the matrix the per-member estimate reads: NaN-aware column means
    _Xc = np.ma.masked_invalid(_X)

    _scores, _kept = {}, []
    for _, m in org_modules.iterrows():
        # one var row per symbol: the one with the most values in these samples
        _cand = _sym[_sym.isin(m.genes)]
        _best = {}
        for _vi, _g in _cand.items():
            _j = _ad.var_names.get_loc(_vi)
            _n_ok = int(np.isfinite(_X[_keep, _j]).sum())
            if _g not in _best or _n_ok > _best[_g][1]:
                _best[_g] = (_j, _n_ok)
        # member filter: quantified in >= ORG_MIN_FRAC of the samples of both groups
        _mem = [(g, j) for g, (j, _) in _best.items()
                if np.isfinite(_X[_ma, j]).sum() >= _min_a
                and np.isfinite(_X[_mb, j]).sum() >= _min_b]
        _cols = [j for _, j in _mem]
        # the robustness column: per member, its mean over aggregate samples
        # minus its mean over host samples (in brain within each donor, then over
        # the donors), then averaged over members. Each
        # protein is its own control, so the value does not depend on which
        # members a sample detected. With no missing value it equals delta_module.
        if not _cols:
            _dm = np.nan
        elif _c["donor"]:
            _dm = float(_donor_diff(_X[:, _cols]).mean(axis=0).mean())
        else:
            _dm = float(np.mean(_Xc[_ma][:, _cols].mean(axis=0)
                                - _Xc[_mb][:, _cols].mean(axis=0)))
        _row = dict(contrast=_c["name"], system=_c["system"], organelle=m.organelle,
                    go_id=m.go_id, module=m.module, n_annotated=m.n_annotated,
                    n_matched=len(_best), n_members=len(_mem),
                    members=sorted(g for g, _ in _mem),
                    delta_module=np.nan, delta_module_permember=_dm)
        if len(_mem) >= ORG_MIN_MEMBERS:
            with np.errstate(invalid="ignore"):
                _s = np.nanmean(_Z[:, _cols], axis=1)
            if np.isfinite(_s[_keep]).all():
                # the bar is the tested quantity: the group difference of the
                # per-sample score, whose pair differences go into eBayes below
                if _c["donor"]:
                    _row["delta_module"] = float(_donor_diff(_s).mean())
                else:
                    _row["delta_module"] = float(_s[_ma].mean() - _s[_mb].mean())
                _scores[m.go_id] = _s
                _kept.append(m.go_id)
            else:
                _row["n_members"] = 0      # a sample with no member measured at all
                _row["delta_module_permember"] = np.nan
                print(f"  {_c['name']} {m.module}: dropped, a sample has no "
                      f"member quantified")
        org_bars.append(_row)
    org_scores[_c["name"]] = _scores

    # --- pairs within one organelle ------------------------------------
    _pairs, _vals = [], []
    for _org, _grp in org_modules.groupby("organelle", sort=False):
        _ids = [g for g in _grp.go_id if g in _scores]
        for _a, _b in itertools.combinations(_ids, 2):
            _pairs.append((_org, _a, _b))
            _vals.append(_scores[_a] - _scores[_b])
    if not _pairs:
        raise RuntimeError(f"{_c['name']}: no module pair survived scoring")
    _M = np.vstack(_vals).T                      # samples x pairs
    _names = [f"{a}|{b}" for _, a, b in _pairs]
    if _c["donor"]:
        # one value per donor and pair, then an intercept-only limma over donors
        _E = pd.DataFrame(_donor_diff(_M).filled(np.nan).T, index=_names,
                          columns=_donors)
        _fit = eBayes(lmFit(_E, pd.DataFrame({"delta": np.ones(len(_donors))},
                                             index=_donors)))
        # sort_by="none" is rejected by inmoose, so the table is sorted by p and
        # reindexed to the pair order
        _tt = topTable(_fit, coef=_fit.coefficients.columns[0], number=_E.shape[0],
                       sort_by="P").reindex(_E.index)
        _out = pd.DataFrame({"log2fc": _tt["log2FoldChange"], "p_value": _tt["pvalue"],
                             "fdr": _tt["adj_pvalue"]})
        print(f"  {_c['name']}: one value per donor, {len(_donors)} donors over "
              f"{_keep.sum()} samples ({_na} aggregate, {_nb} host), "
              f"df.prior {np.ravel(_fit.df_prior)[0]:.1f}")
    else:
        _pa = ad.AnnData(
            X=_M[_keep],
            obs=pd.DataFrame({"__contrast": np.where(_ma[_keep], "A", "B")},
                             index=_ad.obs_names[_keep]),
            var=pd.DataFrame(index=_names),
        )
        _key, _out = at.tl.diff_exp_ebayes(adata=_pa, between_column="__contrast",
                                           comparison=("A", "B"))
    _name2 = org_modules.set_index("go_id").module.to_dict()
    _nmem = {r["go_id"]: r["n_members"] for r in org_bars if r["contrast"] == _c["name"]}
    for (_org, _a, _b) in _pairs:
        _r = _out.loc[f"{_a}|{_b}"]
        org_pair_rows.append(dict(
            contrast=_c["name"], system=_c["system"], organelle=_org,
            go_a=_a, go_b=_b, module_a=_name2[_a], module_b=_name2[_b],
            n_a=_nmem[_a], n_b=_nmem[_b], n_samples=int(_keep.sum()),
            delta=float(_r.log2fc), p_value=float(_r.p_value), fdr=float(_r.fdr)))

    # member counts stage by stage: annotated, matched to a matrix row,
    # passing the member filter
    _st = pd.DataFrame([r for r in org_bars if r["contrast"] == _c["name"]])
    print(f"{_c['name']}: {_na} aggregate + {_nb} host samples "
          f"(>= {_min_a} and >= {_min_b} values required) | "
          f"{len(_kept)} modules scored | {len(_pairs)} pairs | "
          f"{int((_out.fdr < 0.05).sum())} at FDR < 0.05")
    print(f"    annotated {_st.n_annotated.sum()} -> matched to a row in the "
          f"matrix {_st.n_matched.sum()} -> quantified in half of both groups "
          f"{_st.n_members.sum()}")

org_bars = pd.DataFrame(org_bars)
org_pairs = pd.DataFrame(org_pair_rows)
# one column, read by the figure, the stars and the captions
org_pairs["called"] = ((org_pairs.fdr < 0.05)
                       & (org_pairs.delta.abs() >= ORG_MIN_DELTA))

# --- checks -------------------------------------------------------------
# bar_a - bar_b is the group-mean difference of the pair feature the test fits,
# so it equals the tested delta.
_bar = org_bars.set_index(["contrast", "go_id"]).delta_module
_recon = np.array([_bar[(r.contrast, r.go_a)] - _bar[(r.contrast, r.go_b)]
                   for r in org_pairs.itertuples()])
_id_gap = np.abs(org_pairs.delta.to_numpy() - _recon)
assert np.nanmax(_id_gap) < 1e-8, (
    f"the drawn bars no longer reproduce the tested difference, max "
    f"{np.nanmax(_id_gap):.2e}: a caption quoting the test would contradict "
    f"the figure")
print(f"CHECK every drawn bar difference equals the tested difference "
      f"(max |delta - (bar_a - bar_b)| = {np.nanmax(_id_gap):.1e} over "
      f"{len(org_pairs)} pairs)")

# modules whose score still differs from the per-member estimate after the
# centring
ORG_SHIFT_WARN = 0.05
_rb = org_bars.dropna(subset=["delta_module", "delta_module_permember"]).copy()
_rb["gap"] = (_rb.delta_module - _rb.delta_module_permember).abs()
_worst = _rb[_rb.gap > ORG_SHIFT_WARN].sort_values("gap", ascending=False)
print(f"CHECK drawn against per-member estimate: median "
      f"{_rb.gap.median():.3f}, max {_rb.gap.max():.3f}, "
      f"{len(_worst)} of {len(_rb)} modules above {ORG_SHIFT_WARN}")
if len(_worst):
    print(_worst[["contrast", "organelle", "module", "n_members",
                  "delta_module", "delta_module_permember", "gap"]]
          .to_string(index=False, float_format=lambda v: f"{v:.3f}"))

assert (org_pairs.fdr >= org_pairs.p_value - 1e-12).all(), \
    "an FDR fell below its own raw p"
assert (org_pairs[["n_a", "n_b"]].to_numpy() >= ORG_MIN_MEMBERS).all(), \
    "a tested pair used a module with fewer than ORG_MIN_MEMBERS proteins"
print(f"\nCHECK every FDR is at or above its own raw p "
      f"({len(org_pairs)} pairs over {org_pairs.contrast.nunique()} contrasts)")
print(f"CHECK BH ran once per contrast, never pooled: "
      f"{org_pairs.groupby('contrast').size().to_dict()}")

_call = pd.DataFrame({
    "pairs": org_pairs.groupby("contrast").size(),
    "FDR < 0.05": org_pairs[org_pairs.fdr < 0.05].groupby("contrast").size(),
    f"and |delta| >= {ORG_MIN_DELTA}":
        org_pairs[org_pairs.called].groupby("contrast").size(),
}).fillna(0).astype(int)
print(f"\nCalled pairs, FDR alone against FDR with the {ORG_MIN_DELTA} log2 floor:")
print(_call.to_string())

org_bars.assign(members=org_bars.members.map(lambda s: ";".join(s))).to_csv(
    f"{output_dir}/{today}_organelle_module_bars.csv", index=False)
org_pairs.to_csv(f"{output_dir}/{today}_organelle_module_pairs.csv", index=False)

_sig = org_pairs[org_pairs.called].sort_values("fdr")
print(f"\n{len(_sig)} called module pairs (FDR < 0.05 and at least "
      f"{ORG_MIN_DELTA} log2):")
print(_sig[["contrast", "organelle", "module_a", "module_b", "delta", "fdr"]]
      .to_string(index=False, float_format=lambda v: f"{v:.3g}") if len(_sig)
      else "  none")

## 13.1 Figure 5d

In [ ]:
# ============================================================
# Figure 5 d: organelle panels, and the settings shared with Figure S8 a-m.
#
#   One panel per organelle; the x positions are the modules, one bar per
#   contrast in a fixed order and colour. Bar height is the module's mean log2
#   intensity difference, aggregate minus host, from the same matrix and the same
#   members as the eBayes test. Above each bar the number of proteins it rests
#   on, and a star when the module is in a called pair in that contrast (FDR <
#   0.05 and a difference of at least ORG_MIN_DELTA log2). The GO identifier and
#   the annotated size of each module are in the supplementary table.
#
#   The star marks the module, not one pair: with three or more modules a module
#   can be in several pairs. Brackets joining each called pair would take a
#   quarter to a half of the panel height; the pairs are printed as caption text
#   instead. In an organelle with two modules the star identifies the pair.
#
#   No hatching for the coverage of a GO set: the set contains proteins the
#   tissue does not express, so coverage does not measure robustness.
# Drawn by organelle_bars (src/crosssystem_plots.py).
# ============================================================
ORG_WIDE = {"mitochondrion"}    # organelles drawn on a two-unit canvas: eight
                                # modules do not fit at TICK_FS on one unit

# One colour per contrast, the same in every panel. The hues of SYSTEM_COLORS;
# lightness separates the two iPSC contrasts.
ORG_COLORS = {
    "HEK":           SYSTEM_COLORS["HEK"],
    "iPSC_somatic":  SYSTEM_COLORS["iPSC"],
    "iPSC_neuritic": shade(SYSTEM_COLORS["iPSC"], 0.45),
    "Brain":         SYSTEM_COLORS["Brain"],
}
ORG_LABELS = {c["name"]: c["label"] for c in ORG_CONTRASTS}

# short x labels; the full term name and the identifier are in the table
ORG_SHORT = {
    "GO:0005765": "membrane", "GO:0043202": "lumen",
    "GO:0010008": "membrane", "GO:0031904": "lumen", "GO:0036452": "ESCRT",
    "GO:0000139": "membrane", "GO:0005796": "lumen", "GO:0030126": "COPI coat",
    "GO:0005789": "membrane", "GO:0005788": "lumen", "GO:0005784": "Sec61",
    "GO:0005741": "outer membrane", "GO:0005743": "inner membrane",
    "GO:0005759": "matrix", "GO:0005758": "IMS",
    "GO:0005762": "mito LSU", "GO:0005763": "mito SSU",
    "GO:0140596": "TOM", "GO:0005744": "TIM23",
    "GO:0005778": "membrane", "GO:0005782": "matrix",
    "GO:0008021": "synaptic vesicle", "GO:0048786": "active zone",
    "GO:0005839": "20S core", "GO:0005838": "19S RP", "GO:0008537": "activator",
    "GO:0033179": "V0", "GO:0033180": "V1",
    "GO:0022625": "60S", "GO:0022627": "40S",
    "GO:0071439": "clathrin", "GO:0030122": "AP-2", "GO:0030131": "adaptors",
    "GO:0005652": "lamina", "GO:0005643": "pore",
    "GO:0000421": "membrane", "GO:0034423": "lumen",
    "GO:0005832": "TRiC", "GO:0016272": "prefoldin",
    "GO:0045271": "complex I", "GO:0045275": "complex III",
    "GO:0045277": "complex IV", "GO:0045259": "ATP synthase",
    "GO:0005686": "U2 snRNP", "GO:0005682": "U5 snRNP", "GO:0120114": "LSm",
}
assert set(ORG_SHORT) >= set(org_modules.go_id), \
    f"no short label for {sorted(set(org_modules.go_id) - set(ORG_SHORT))}"

assert {"go_a", "go_b", "n_a", "n_b", "delta", "fdr", "called"} <= \
    set(org_pairs.columns), \
    f"org_pairs is missing columns; it has {list(org_pairs.columns)}"

_BARS = org_bars.set_index(["contrast", "go_id"])
# modules in at least one called pair, per contrast -> the star; the same column
# "called" as the caption and the table
_SIG_MOD = {(r.contrast, g) for r in org_pairs[org_pairs.called].itertuples()
            for g in (r.go_a, r.go_b)}


def _org_caption(organelle):
    """The called pairs of one organelle, as caption text.

    Returns the items alone, without a leading sentence, so the main figure can
    print one line per organelle and the supplement one paragraph for all
    organelles. An empty string means nothing was called.

    The direction is the tested quantity. org_pairs.delta is the aggregate minus
    host difference of module A minus the same difference of module B, so its
    sign says which module of the pair changed more, and the FDR refers to that
    quantity. The bars reproduce it (assertion in the module test cell); the
    NOTE below prints if a drawn bar pair contradicts the test.
    """
    _r = (org_pairs[org_pairs.called & (org_pairs.organelle == organelle)]
          .sort_values("fdr"))
    items = []
    for r in _r.itertuples():
        _hi, _lo = (r.go_a, r.go_b) if r.delta > 0 else (r.go_b, r.go_a)
        _nhi, _nlo = (r.n_a, r.n_b) if r.delta > 0 else (r.n_b, r.n_a)
        items.append(f"{ORG_SHORT[_hi]} exceeds {ORG_SHORT[_lo]} by "
                     f"{abs(r.delta):.2f} log2 in {ORG_LABELS[r.contrast]} "
                     f"(FDR {r.fdr:.2g}, {_nhi} and {_nlo} proteins)")
        _bd = (_BARS.loc[(r.contrast, _hi), "delta_module"]
               - _BARS.loc[(r.contrast, _lo), "delta_module"])
        if _bd < 0:
            print(f"  NOTE {organelle}, {ORG_LABELS[r.contrast]}: the test puts "
                  f"{ORG_SHORT[_hi]} {abs(r.delta):.2f} above {ORG_SHORT[_lo]}, "
                  f"the drawn bars put it {abs(_bd):.2f} below. Say the "
                  f"direction as the test gives it, or drop this pair.")
    return "; ".join(items)


ORG_CAPTION_RULE = ("module pairs differing at FDR < 0.05 by at least "
                    f"{ORG_MIN_DELTA:g} log2, the module named first being the "
                    "one with the larger aggregate minus host difference")


# CHECK every drawn bar rests on at least ORG_MIN_MEMBERS proteins, and the
# number printed above it is the number the test used.
_drawn = org_bars[org_bars.n_members >= ORG_MIN_MEMBERS]
assert (_drawn.n_members >= ORG_MIN_MEMBERS).all()
assert (_drawn.n_members == _drawn.members.map(len)).all(), \
    "the printed count is not the length of the member list the bar was built from"
assert _drawn.delta_module.notna().all(), "a drawn bar has no value"
print(f"CHECK {len(_drawn)} bars will be drawn, all on >= {ORG_MIN_MEMBERS} "
      f"proteins, counts equal to the member sets used in the test")

# Figure 5 d: two organelles of two modules each, stacked in one column (a 1 x 2
# panel), the cytosolic ribosome on top, each on its own y axis. The lysosome
# axis gets 0.2 log2 more at the top: the counts over its small membrane bars
# have three digits, and the stars above them would reach the title.
ORG_MAIN = ["cytosolic ribosome", "lysosome"]
_lims = {o: organelle_ylim(_BARS, org_modules, [o], ORG_ORDER) for o in ORG_MAIN}
_lims["lysosome"] = (_lims["lysosome"][0], _lims["lysosome"][1] + 0.2)
organelle_bars(_BARS, org_modules, ORG_MAIN, ORG_ORDER, ORG_COLORS, ORG_LABELS,
               ORG_SHORT, _SIG_MOD, "d", "organelle_modules_main", FIG5_DIR,
               ylim=_lims, wide=ORG_WIDE)
for _o in ORG_MAIN:
    _items = _org_caption(_o)
    print(f"CAPTION {_o[0].upper() + _o[1:]}: "
          + (f"{ORG_CAPTION_RULE}: {_items}." if _items
             else f"no module pair differed at FDR < 0.05 by at least "
                  f"{ORG_MIN_DELTA:g} log2."))
plt.show()

## 13.2 Figure S8 a–m

In [ ]:
# ============================================================
# Figure S8 a-m: one figure per organelle, the encoding of Figure 5 d.
#
#   Each figure scales to its own bars, as in Figure 5 d, and each side of zero
#   is sized by its own bars. The limits are printed with every figure: the
#   panels do not share an axis and are compared by reading their axes.
#
#   Organelles in ORG_WIDE get a two-unit canvas, because module labels stay at
#   TICK_FS. ORG_PAD_SI gives these panels more room above the stars.
#
#   One figure carries the colour key: the key is the same on all of them and
#   would take the bottom of every panel. The figures without it use their full
#   height, so the mitochondrion, the figure with the key, is Figure S8 a and its
#   axes are 5 mm shorter than the others'.
#
#   The respiratory chain is not drawn (ORG_SUPP_DROPPED); its modules are in the
#   member table and in the tests.
#
#   save_panel writes the canvas, not a crop to the ink, so every figure is
#   figsize(1, 1), and the mitochondrion figsize(2, 1).
#
#   One caption paragraph for all figures, printed at the end.
# ============================================================
ORG_PAD_SI = 0.42       # these panels are half as wide as Figure 5 d, so the count
                        # labels sit closer to the bar ends and the stars need more
                        # room above

# the one figure with the key: the two-unit mitochondrion figure, where the four
# contrasts fit in one row
ORG_SUPP_LEGEND = "mitochondrion"

# Panel order, written out: the mitochondrion first (S8 a) because it carries
# the key. The assertion below stops the cell if an organelle of the module table
# is neither here, in ORG_MAIN nor in ORG_SUPP_DROPPED.
ORG_SUPP = ["mitochondrion", "endosome", "Golgi apparatus", "peroxisome",
            "presynapse", "proteasome", "V-ATPase", "endoplasmic reticulum",
            "clathrin coat", "nuclear envelope", "autophagosome", "chaperonin",
            "spliceosome"]
ORG_SUPP_DROPPED = {"respiratory chain"}     # not drawn; see the header
assert ORG_SUPP_DROPPED <= set(org_modules.organelle), \
    f"ORG_SUPP_DROPPED names an organelle the module table does not define: {sorted(ORG_SUPP_DROPPED)}"
_expect = set(org_modules.organelle) - set(ORG_MAIN) - ORG_SUPP_DROPPED
assert set(ORG_SUPP) == _expect and len(ORG_SUPP) == len(_expect), (
    f"ORG_SUPP does not cover the supplement: missing "
    f"{sorted(_expect - set(ORG_SUPP))}, unknown {sorted(set(ORG_SUPP) - _expect)}")
assert ORG_SUPP_LEGEND in ORG_SUPP, \
    f"{ORG_SUPP_LEGEND!r} is not a supplement organelle; choose one of {ORG_SUPP}"
print(f"supplement: {len(ORG_SUPP)} separate figures, each on its own axis, "
      f"the colour key on {ORG_SUPP_LEGEND} only")
# Figure S8 letters in this order, from a
ORG_SUPP_LETTERS = dict(zip(ORG_SUPP, "abcdefghijklmnopqrstuvwxyz"))

_sizes = {}
for _o in ORG_SUPP:
    _slug = re.sub(r"[^a-z0-9]+", "_", _o.lower()).strip("_")
    _fig = organelle_bars(_BARS, org_modules, [_o], ORG_ORDER, ORG_COLORS, ORG_LABELS,
                          ORG_SHORT, _SIG_MOD, ORG_SUPP_LETTERS[_o],
                          f"organelle_modules_{_slug}", SI8_DIR, pad=ORG_PAD_SI,
                          wide=ORG_WIDE, legend=(_o == ORG_SUPP_LEGEND))
    _sizes[_o] = tuple(round(float(v), 2) for v in _fig.get_size_inches())
    plt.show()

# CHECK the saved files are one size, the wide organelles excepted
_one = {o: s for o, s in _sizes.items() if o not in ORG_WIDE}
assert len(set(_one.values())) == 1, \
    f"supplement figures differ in size: {sorted(set(_one.values()))}"
print(f"CHECK {len(_one)} supplement figures at {next(iter(_one.values()))} in, "
      f"{len(_sizes) - len(_one)} on the wide canvas: "
      f"{ {o: s for o, s in _sizes.items() if o in ORG_WIDE} }")

# --- one caption for the whole supplement ------------------------------
_parts = {o: _org_caption(o) for o in ORG_SUPP}
_with = [o for o in ORG_SUPP if _parts[o]]
_without = [o for o in ORG_SUPP if not _parts[o]]
print(f"\nCHECK called pairs in {len(_with)} of {len(ORG_SUPP)} supplement "
      f"organelles, none in {len(_without)}")
print("\nCAPTION supplement, one paragraph:\n")
print(ORG_CAPTION_RULE[0].upper() + ORG_CAPTION_RULE[1:] + ". "
      + " ".join(f"{o[0].upper() + o[1:]}: {_parts[o]}." for o in _with)
      + (f" No module pair reached that threshold in "
         f"{', '.join(_without)}." if _without else ""))

# 14. Composed figures

## 14.1 Figure 5

In [ ]:
# Figure 5, panels a-f on the 4-column page grid: one letter per grid unit,
# "." an empty unit. compose_figure takes, per letter, the newest SVG in the
# folder whose name starts with "<letter>_".
figure_name = f"{today}_Figure5_Comparison"
layout = "a a a b\n c c d b\n c c d b\n e e f f"

compose_figure(
    folder=FIG5_DIR,
    layout=layout,
    output=f"{FIG5_DIR}/{figure_name}.svg",
    show_box=True,
)

## 14.2 Figures S7 and S8

In [ ]:
# Figure S7, panels a-f
figure_name = f"{today}_SI7_CrossSystem"
layout = "a b b c\n a b b c\n d e f .\n . e f ."

compose_figure(
    folder=SI7_DIR,
    layout=layout,
    output=f"{SI7_DIR}/{figure_name}.svg",
    show_box=True,
)

In [ ]:
# Figure S8, panels a-n (a: the two-unit mitochondrion figure, n: the four-level
# property panel)
figure_name = f"{today}_SI8_CrossSystem"
layout = "a a b c\n d e f g\n h i j k\n l m n n"

compose_figure(
    folder=SI8_DIR,
    layout=layout,
    output=f"{SI8_DIR}/{figure_name}.svg",
    show_box=True,
)